# 06_rebuild — 재구축 v2

04·05에서 발견한 결함을 **구조적으로** 없앤 파이프라인을 다시 세운다.

## 왜 다시 짓는가

04~05의 잘못된 결정 세 개가 전부 **모델이 아니라 측정**에서 나왔다.

| 무엇이 틀렸나 | 어떻게 고치나 |
|---|---|
| 노이즈(시드 σ 0.0024)보다 작은 0.0003 차이로 순위를 매겼다 | `ms.compare()` — fold별 **짝지은 차이** + 문턱 미달은 **동률** |
| 오염 fold(2024-12 group_3 정지율 0.815)를 동등 가중으로 평균 | `ms.Bench` — 점수를 **그룹×fold로 분해 보관**, 정제 B평균 항상 병기 |
| 피처를 **만든 뒤에** 효과를 쟀다 (speed-up 반나절 낭비) | `ms.headroom()` — 만들기 **전에** 상한 계산 (학습 0회) |
| 라벨 참고 컬럼이 parquet에 있어 제외 목록이 두 번 빠뜨렸다 | 파생 컬럼을 **parquet에서 읽지 않는다**. 전부 런타임 fold-safe 생성 |

## 구조

```
0층  측정·판정        src/measure.py   ← 지금 여기
1층  데이터·fold      원본 예보만 로드 + 가동률
2층  풍속             fold-safe GBDT + 오라클 순환 점검
3층  피처             상한 게이트 통과분만
4층  발전량           라벨·가중 수정
5층  배치             행별 기대 FICR 최대화
```

**규칙**: 한 번에 **하나만** 바꾼다. 04의 후버 `alpha` 사건(두 개를 같이 바꿔서 원인을 못 찾음)이 그래서 났다.

## 0-1. 측정 계층이 공식 산식과 정확히 같은가

**이 셀이 하는 일**: 학습을 한 번도 하지 않고, 04 보고서에 기록된 베이스라인 사다리
0단계(**시간대×월 평균** — 기상 정보를 전혀 안 쓰고 "작년 이맘때 이 시각의 평균"으로
찍는 것)를 다시 계산해서, 새 측정 계층이 뱉는 숫자가 **04 원장과 같은지** 본다.

**왜 이걸로 검증이 되나**: 이 한 셀이 세 가지를 동시에 확인한다.

1. `assert_matches_official` — 그룹별로 쪼갠 뒤 다시 합친 값이 공식 `metric()`과 **소수점 12자리까지** 같은가
2. fold 정의(학습/검증 경계)가 04와 같은가
3. `Bench`가 fold를 제대로 모으고 있는가

**맞아야 하는 값** (04 보고서 "베이스라인 사다리" 0단계):

| A안 | B fold1 | B fold2 | B fold3 |
|---|---|---|---|
| 0.4334 | 0.4188 | 0.4486 | 0.4183 |

**비용**: 학습 0회, 10초 내외. 메모리도 거의 안 쓴다(`with_test=False`).

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

import src.pipeline as pl          # 1층(데이터·fold)은 검증된 것을 그대로 재사용
import src.measure as ms           # ★ 0층 — 이번에 새로 만든 측정·판정 계층
from src.metric import TARGET_COLS

# test와 가동률은 이 셀에 필요 없다 → 메모리 절약 (HANDOFF 주의사항 7번)
ctx = pl.load_context(with_test=False, with_avail=False)
t = ctx.train["kst_dtm"]
print(f"train {ctx.train.shape},  fold {list(ctx.fold_info)}")

# --- 베이스라인 0단계: 시간대×월 평균 (학습 없음, fold-safe) -----------------
bench = ms.Bench()
for fold, info in ctx.fold_info.items():
    tm, vi = info["train_mask"], info["valid_idx"]     # tm=학습구간, vi=검증구간
    preds = {}
    for g in TARGET_COLS:
        y = ctx.train[g]
        # ⚠️ 평균표는 **학습 구간에서만** 만든다 — 검증 구간 라벨을 보면 누수다
        tab = y[tm].groupby([t[tm].dt.month, t[tm].dt.hour]).mean()
        key = pd.MultiIndex.from_arrays([t[vi].dt.month, t[vi].dt.hour])
        preds[g] = pd.Series(tab.reindex(key).to_numpy(), index=vi).fillna(y[tm].mean())
    pred = pd.DataFrame(preds, index=vi)

    ms.assert_matches_official(info["actual_df"], pred)        # ★ 핵심 검증
    bench.add("hour×month", fold, ms.per_group_scores(info["actual_df"], pred))

print("\n[OK] 그룹별 분해 → 재합성이 공식 metric()과 소수점 12자리까지 일치\n")
display(bench.table())
print("04 보고서 기록값 :  A안 0.4334 / fold1 0.4188 / fold2 0.4486 / fold3 0.4183")

train (26304, 888),  fold ['A안(2024)', 'B안 fold1', 'B안 fold2', 'B안 fold3']

[OK] 그룹별 분해 → 재합성이 공식 metric()과 소수점 12자리까지 일치



,A안(2024),B안 fold1,B안 fold2,B안 fold3,B평균,B σ,B평균(정제),시드 σ
hour×month,0.4334,0.4188,0.4486,0.4183,0.4285,0.0174,0.4272,NaN


04 보고서 기록값 :  A안 0.4334 / fold1 0.4188 / fold2 0.4486 / fold3 0.4183


## 0-2. 그룹별로 쪼개서 본다 — 04에서 한 번도 안 본 것

**이 셀이 하는 일**: 위에서 만든 점수를 그룹별로 펼친다.

**왜 중요한가**: 공식 산식은 그룹별 NMAE와 FICR을 각각 **단순평균**한다.

```
총점 = 0.5 × (1 − 평균(nmae_1, nmae_2, nmae_3)) + 0.5 × 평균(ficr_1, ficr_2, ficr_3)
```

즉 **그룹 하나가 총점의 정확히 1/3**을 쥐고 있다. 그런데 04에서 딱 한 번 찍힌 그룹별
FICR은 **0.3795 / 0.4432 / 0.3044** 였다. group_3이 group_2보다 **0.14** 낮다.

group_3만 group_2 수준으로 올리면 총점이 `0.5 × (1/3) × 0.14 = +0.023` 오른다.
**04·05의 모든 튜닝을 합친 것(+0.010, +0.010)보다 크다.** 그런데 group_3은
학습 데이터가 1/3 적고(2022년 없음) 제조사도 다른데(UNISON) 가장 적은 관심을 받았다.

여기서 확인할 것: **어느 그룹이, 어느 fold에서, NMAE 때문에 지는지 FICR 때문에 지는지.**

In [2]:
d = bench.df
for col, title in [("ficr", "FICR (밴드 적중)"), ("nmae", "NMAE (오차 크기)"),
                   ("sigma", "σ  잔차 표준편차 ÷ 용량  ← FICR의 진짜 병목"),
                   ("bias", "편향  (+면 과대예측)"), ("n", "채점 대상 행 수")]:
    print(f"\n■ {title}")
    display(d.pivot_table(index="group", columns="fold", values=col)
             .reindex(columns=[c for c in [ms.A_FOLD, *ms.B_FOLDS] if c in set(d["fold"])])
             .round(4))

print("\n※ 지금은 기상을 전혀 안 쓴 베이스라인이라 절대값은 의미 없다.")
print("   보려는 것은 '그룹×fold로 쪼개서 볼 수 있게 됐다'는 사실 자체다.")


■ FICR (밴드 적중)


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3
group,,,,
kpx_group_1,0.1190,0.1011,0.1367,0.1015
kpx_group_2,0.1142,0.1052,0.1433,0.0868
kpx_group_3,0.1213,0.1126,0.1309,0.1116



■ NMAE (오차 크기)


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3
group,,,,
kpx_group_1,0.2474,0.2712,0.2357,0.2596
kpx_group_2,0.2498,0.2752,0.2297,0.2712
kpx_group_3,0.2567,0.2599,0.2542,0.2595



■ σ  잔차 표준편차 ÷ 용량  ← FICR의 진짜 병목


...


■ 편향  (+면 과대예측)


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3
group,,,,
kpx_group_1,-0.1600,-0.1993,-0.1313,-0.1898
kpx_group_2,-0.1470,-0.1706,-0.0989,-0.1981
kpx_group_3,-0.1869,-0.1896,-0.1745,-0.2004



■ 채점 대상 행 수


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3
group,,,,
kpx_group_1,4978.0,2652.0,2535.0,2443.0
kpx_group_2,4944.0,2665.0,2547.0,2397.0
kpx_group_3,4510.0,2311.0,2351.0,2159.0



※ 지금은 기상을 전혀 안 쓴 베이스라인이라 절대값은 의미 없다.
   보려는 것은 '그룹×fold로 쪼개서 볼 수 있게 됐다'는 사실 자체다.


---

## 1-1. 기준선 등록 — 확정 구성(LightGBM 단독)을 새 측정 계층에 올린다

**이 셀이 하는 일**: 지금까지 확정된 발전량 모델(`B_norm` 라벨 + τ=0.50 + 피처 상위 200)을
4개 fold × 3그룹에 돌려서, **앞으로 모든 비교의 기준점**으로 등록한다.

**왜 지금 하나**: 0층 검증은 "산식을 재는 법"이 맞다는 것까지만 보였다. 이제 **실제 모델의
그룹별 성적**이 필요하다. 특히 0-2에서 나온 가설 하나를 검증해야 한다.

> 기상을 안 쓴 베이스라인에서는 group_3이 뒤지지 않았다(FICR 0.1213 vs 0.1142/0.1190).
> 그런데 04의 실제 모델에서는 group_3만 0.14 낮았다(0.3044 vs 0.4432).
> **⇒ group_3의 열세는 데이터가 아니라 모델 쪽 문제일 가능성이 있다.**

이게 맞으면 group_3 하나만 group_2 수준으로 끌어올려도 **총점 +0.023**이다
(산식이 그룹별 단순평균이라 그룹 하나가 총점의 1/3을 쥔다).
**04·05의 모든 튜닝을 합친 것(+0.010, +0.010)보다 크다.**

**왜 MLP를 빼고 LightGBM만 쓰나**: LightGBM 기본값은 **완전 결정적**이다
(`subsample=1.0`, `subsample_freq=0` — HANDOFF 주의사항 11번). 시드 노이즈가 0이라
구조 실험의 기준선으로 가장 깨끗하다. MLP 블렌드는 5층에서 마지막에 얹는다.

**비용**: 학습 약 30회(풍속 GBDT 9 + 피처선택 9 + 발전량 12), **5~8분**.
**메모리 주의**: 피처 프레임 9개가 캐시된다(약 1.5 GB). 다른 앱을 닫아두는 게 안전하다.

In [3]:
import importlib
importlib.reload(ms)          # measure.py를 고쳤으므로 다시 읽는다

# `B_norm` 라벨은 가동률이 필요하다 → with_avail=True 로 다시 로드
ctx = pl.load_context(with_test=False, with_avail=True)
print("가동률 결측 비율: "
      + ", ".join(f"{g.split('_')[-1]}={ctx.avail[g].isna().mean():.3f}" for g in TARGET_COLS))

PRED = {}          # (fold, variant, group) -> pd.Series   재학습 없는 블렌드 실험용
_ = pl.run_variant(ctx, PRED, "L0_lgb",
                   pl.lgbm_fold_fit_fn(mode="B_norm", tau=0.50, top_n=200))

bench.add_from_cache(ctx, PRED, "L0_lgb")
print()
display(bench.table())

가동률 결측 비율: 1=0.000, 2=0.000, 3=0.360


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

  [L0_lgb] A안(2024): score=0.6418  (1-NMAE=0.8730, FICR=0.4105)


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

  [L0_lgb] B안 fold1: score=0.6177  (1-NMAE=0.8556, FICR=0.3798)


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  [L0_lgb] B안 fold2: score=0.6428  (1-NMAE=0.8708, FICR=0.4148)


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

  [L0_lgb] B안 fold3: score=0.6462  (1-NMAE=0.8762, FICR=0.4161)



,A안(2024),B안 fold1,B안 fold2,B안 fold3,B평균,B σ,B평균(정제),시드 σ
hour×month,0.4334,0.4188,0.4486,0.4183,0.4285,0.0174,0.4272,NaN
L0_lgb,0.6418,0.6177,0.6428,0.6462,0.6356,0.0156,0.6429,NaN


## 1-2. group_3 가설 검증 — 어느 그룹이 어디서 지는가

**이 셀이 하는 일**: 방금 돌린 실제 모델의 성적을 그룹별로 펼쳐 세 가지를 가른다.

1. **group_3이 정말 뒤지는가** (04는 단 한 번 봤다 → 4개 fold로 재확인)
2. 뒤진다면 **NMAE 때문인가 FICR 때문인가**
3. FICR이라면 **σ(오차 폭) 때문인가 편향(위치) 때문인가**

**왜 이 갈래가 중요한가**: 처방이 완전히 다르다.

| 진단 | 처방 |
|---|---|
| σ가 크다 | 풍속·피처 — **오차 자체**를 줄여야 한다. 위치 조정은 소용없다 |
| 편향이 크다 | τ·라벨·가동률 복원 — **위치만** 옮기면 된다. 값싸다 |
| NMAE만 나쁘다 | 큰 오차(꼬리)의 문제. 손실함수·이상치 |

04는 이 갈래를 **세 그룹 평균으로만** 보고 "σ가 문제"라고 결론 냈다.
그룹마다 병목이 다르면 **그룹마다 다른 처방**이 필요하다.

**`밴드÷σ`**: FICR의 4원 밴드(±6% 용량)가 오차 폭의 몇 배인가.
04 전체 평균은 **0.35**였다(정규분포라면 통과율 27%).

**비용**: 학습 0회, 즉시.

In [4]:
summ = ms.group_report(bench, "L0_lgb")

■ [L0_lgb] 그룹별 (4 fold 평균)


,1-NMAE,ficr,sigma,bias,band6,band8,n,밴드÷σ
group,,,,,,,,
kpx_group_1,0.8755,0.4225,0.1644,0.0096,0.3461,0.4480,3152.00,0.3649
kpx_group_2,0.8745,0.4557,0.1662,0.0357,0.3813,0.4806,3138.25,0.3611
kpx_group_3,0.8568,0.3377,0.1845,0.0276,0.2745,0.3587,2832.75,0.3252



■ FICR — fold별


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3
group,,,,
kpx_group_1,0.4293,0.3811,0.4258,0.4538
kpx_group_2,0.4620,0.4323,0.4773,0.4513
kpx_group_3,0.3402,0.3260,0.3413,0.3433



■ σ (오차 폭 ÷ 용량) — fold별


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3
group,,,,
kpx_group_1,0.1580,0.1885,0.1585,0.1528
kpx_group_2,0.1606,0.1870,0.1666,0.1504
kpx_group_3,0.1775,0.2039,0.1793,0.1773



■ 편향 (+면 과대예측) — fold별


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3
group,,,,
kpx_group_1,0.0088,0.0019,0.0100,0.0176
kpx_group_2,0.0364,0.0295,0.0495,0.0273
kpx_group_3,0.0277,0.0218,0.0249,0.0361



■ FICR을 최고 그룹(2, 0.4557) 수준으로 올렸을 때 총점 이득
   kpx_group_3    FICR 0.3377  →  +0.0197
   kpx_group_1    FICR 0.4225  →  +0.0055
   kpx_group_2    FICR 0.4557  →  +0.0000
   (참고) 04 라벨정제 +0.010 / 05 MLP블렌드 +0.010 / 1등과의 격차 0.0321


---

# 2층 — group_3의 σ는 회수 가능한가

## 1-2에서 확정된 것

**group_3이 4개 fold 전부에서 FICR 최저였다** (0.3402 / 0.3260 / 0.3413 / 0.3433).
사전 등록한 판정 규칙대로 **가설 확정**이다. 회수하면 **총점 +0.0197** — 1등과의 격차 0.0321의 61%다.

그리고 **원인이 σ(오차 폭)이지 편향(위치)이 아니라는 것**까지 갈렸다.

| | group_1 | group_2 | group_3 |
|---|---|---|---|
| FICR | 0.4225 | **0.4557** | **0.3377** |
| σ | 0.1644 | 0.1662 | **0.1845** ← 11% 큼 |
| 편향 | 0.0096 | **0.0357** | 0.0276 |

**편향이 가장 큰 group_2가 FICR도 가장 높다.** 04의 결론 7번("FICR은 편향이 아니라
밴드 안 확률질량에 반응한다")이 그룹 단위로 독립 재확인됐다.
**⇒ group_3에 τ·편향 카드를 쓰지 않는다.** σ를 줄여야 한다.

## 그런데 σ가 큰 이유가 두 갈래다 — 처방이 정반대

**(a) 회수 불가 — 물리적으로 원래 그렇다.**
group_3은 UNISON **5기**(1기당 4.2 MW), group_1/2는 VESTAS **6기**(1기당 3.6 MW)다.
발전기가 적을수록 국지적 바람 변동이 서로 상쇄되지 않아 합계 출력이 더 출렁인다.
독립이라 가정하면 `√(6/5) = 1.095` → **σ가 9.5% 커야 한다.**
**실측은 11.6% (0.1845 / 0.1653).** 숫자가 거의 맞는다.

**(b) 회수 가능 — 우리 모델이 group_3에서만 못 하고 있다.**
05 진단의 **풍속** 오차는 group_1/2/3 = 1.403 / 1.536 / 1.523이었다.
**group_3의 풍속은 group_2보다 오히려 정확하다.** 그런데 발전량 σ는 group_3이 최악이다.
**⇒ 바람은 비슷하게 맞히는데 바람→발전량 변환에서 잃고 있다**는 뜻이다.

두 갈래는 **서로 배타적이지 않다.** 얼마씩인지를 재야 한다.

## 어떻게 가르나 — 그룹별 오라클

**바람을 완벽히 알았을 때의 σ**를 그룹별로 잰다. SCADA 실측 풍속을 풍속 피처 자리에
그대로 넣고 같은 모델을 돌린다.

- 오라클 σ도 group_3만 11% 크다 → **(a) 회수 불가.** 카드를 닫고 다른 데 쓴다
- 오라클 σ는 세 그룹이 비슷하다 → **(b) 회수 가능.** 바람→발전량 단(파워커브·라벨)을 판다

**통제**: 피처 집합(`keep`)과 컬럼 이름(`WIND_TAG`)을 기준선과 **똑같이** 쓴다.
바뀌는 것은 **풍속 값 하나뿐**이다(05 speed-up 실험과 같은 통제 방식).

⚠️ SCADA는 test(2025년)에 없다. **진단 전용이며 제출에는 절대 쓸 수 없다.**

**비용**: 학습 12회, 2~3분.

In [5]:
from src.metric import CAPACITY_KWH


def oracle_fit_fn(ctx, g, cv_suffix, train_mask, valid_idx):
    """풍속 자리에 SCADA 실측을 넣는다. **바뀌는 것은 풍속 값 하나뿐.**

    · 컬럼 이름을 기준선과 같은 `WIND_TAG`로 두어 `keep`(피처 200개)을 그대로 재사용한다
    · SCADA 결측(group_3은 2022년 전체가 없다)은 **우리 추정 풍속**으로 메운다
      → 오라클이 '풍속을 안다'는 것 말고 다른 이점을 갖지 않게 한다
    · 파워커브는 `build_frame_with_wind`가 **학습 구간에서만** 적합한다(fold-safe 유지)
    """
    ws_true = ctx.train[f"scada_ws_{g}"]
    ws_est = pl.wind_estimate(ctx, g, cv_suffix, train_mask)      # 1-1에서 이미 학습·캐시됨
    ws = ws_true.fillna(ws_est).clip(lower=0.0)

    X, _ = pl.build_frame_with_wind(ctx, ctx.train, g, train_mask, ws, pl.WIND_TAG)
    keep = pl.keep_for_fold(ctx, g, cv_suffix, train_mask, pl.BEST_TOPN)
    m = pl.lgbm_train_label(ctx, X[keep], g, train_mask,
                            pl.DEFAULT_TAU, pl.SEED, pl.DEFAULT_LABEL_MODE)
    p = pd.Series(m.predict(X.loc[valid_idx, keep]), index=valid_idx)
    return p.clip(lower=0, upper=CAPACITY_KWH[g])


_ = pl.run_variant(ctx, PRED, "L2_oracle", oracle_fit_fn)
bench.add_from_cache(ctx, PRED, "L2_oracle")
print()
display(bench.table())

d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  [L2_oracle] A안(2024): score=0.8759  (1-NMAE=0.9561, FICR=0.7957)


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  [L2_oracle] B안 fold1: score=0.8972  (1-NMAE=0.9528, FICR=0.8415)


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  [L2_oracle] B안 fold2: score=0.8976  (1-NMAE=0.9579, FICR=0.8373)


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  [L2_oracle] B안 fold3: score=0.8541  (1-NMAE=0.9532, FICR=0.7550)



,A안(2024),B안 fold1,B안 fold2,B안 fold3,B평균,B σ,B평균(정제),시드 σ
hour×month,0.4334,0.4188,0.4486,0.4183,0.4285,0.0174,0.4272,NaN
L0_lgb,0.6418,0.6177,0.6428,0.6462,0.6356,0.0156,0.6429,NaN
L2_oracle,0.8759,0.8972,0.8976,0.8541,0.8830,0.0250,0.8993,NaN


In [6]:
summ_o = ms.group_report(bench, "L2_oracle", show_folds=False)

# --- 핵심 판정: group_3의 σ 초과분이 오라클에서도 남는가 --------------------
G1, G2, G3 = TARGET_COLS
comp = pd.DataFrame({"기준선 σ": summ["sigma"], "오라클 σ": summ_o["sigma"]})
comp["오라클로 줄어든 σ %"] = (comp["오라클 σ"] / comp["기준선 σ"] - 1) * 100
print("\n■ 풍속을 완벽히 알았을 때 σ가 얼마나 줄어드나")
display(comp.round(4))

r_base = summ["sigma"][G3] / summ["sigma"][[G1, G2]].mean()
r_orac = summ_o["sigma"][G3] / summ_o["sigma"][[G1, G2]].mean()
print(f"\n■ group_3 σ 초과 배율  (= σ_g3 ÷ σ_g1,g2 평균)")
print(f"   기준선 : {r_base:.3f}")
print(f"   오라클 : {r_orac:.3f}")
print(f"   발전기 대수만으로 설명되는 값 √(6/5) = 1.095")
print()
if r_orac >= 1.07:
    print("   ⇒ (a) 회수 불가 쪽. 바람을 완벽히 알아도 group_3만 σ가 크다.")
    print("      발전기 5기(1기 4.2MW)라 국지 변동이 덜 상쇄되는 물리적 한계.")
    print("      **group_3 특별대우 카드를 닫고** 세 그룹 공통 개선으로 간다.")
elif r_orac <= 1.03:
    print("   ⇒ (b) 회수 가능. 바람만 알면 group_3의 열세가 사라진다.")
    print("      풍속 자체는 group_2보다 정확했으므로(1.523 vs 1.536),")
    print("      **바람→발전량 변환단**(파워커브·라벨·가동률)에서 잃고 있다는 뜻.")
else:
    print("   ⇒ 중간. 물리적 하한과 회수 가능분이 섞여 있다. 3층에서 변환단을 먼저 판다.")

■ [L2_oracle] 그룹별 (4 fold 평균)


,1-NMAE,ficr,sigma,bias,band6,band8,n,밴드÷σ
group,,,,,,,,
kpx_group_1,0.9628,0.8691,0.0641,0.0170,0.8227,0.8846,3152.00,0.9358
kpx_group_2,0.9589,0.8484,0.0717,0.0311,0.7963,0.8658,3138.25,0.8364
kpx_group_3,0.9433,0.7046,0.0897,0.0182,0.6509,0.7226,2832.75,0.6691



■ FICR을 최고 그룹(1, 0.8691) 수준으로 올렸을 때 총점 이득
   kpx_group_3    FICR 0.7046  →  +0.0274
   kpx_group_2    FICR 0.8484  →  +0.0034
   kpx_group_1    FICR 0.8691  →  +0.0000
   (참고) 04 라벨정제 +0.010 / 05 MLP블렌드 +0.010 / 1등과의 격차 0.0321

■ 풍속을 완벽히 알았을 때 σ가 얼마나 줄어드나


,기준선 σ,오라클 σ,오라클로 줄어든 σ %
group,,,
kpx_group_1,0.1644,0.0641,-61.0094
kpx_group_2,0.1662,0.0717,-56.8274
kpx_group_3,0.1845,0.0897,-51.3951



■ group_3 σ 초과 배율  (= σ_g3 ÷ σ_g1,g2 평균)
   기준선 : 1.116
   오라클 : 1.320
   발전기 대수만으로 설명되는 값 √(6/5) = 1.095

   ⇒ (a) 회수 불가 쪽. 바람을 완벽히 알아도 group_3만 σ가 크다.
      발전기 5기(1기 4.2MW)라 국지 변동이 덜 상쇄되는 물리적 한계.
      **group_3 특별대우 카드를 닫고** 세 그룹 공통 개선으로 간다.


## 2-2. 오라클이 '바람'만 아는 게 맞나 — 순환 점검

**이 셀이 하는 일**: 학습 0회. SCADA 나셀 풍속이 **터빈이 도는지 멈췄는지**에 따라
다르게 읽히는지 확인한다.

**왜 이걸 의심하나 (도메인)**: 나셀 풍속계는 **로터 뒤에** 달려 있다. 날개가 지나간
교란된 흐름을 재는 것이라 자유류 풍속이 아니다(IEC 61400-12-2가 다루는 주제).
그런데 **로터가 돌면 운동량을 뽑아가므로 뒤쪽 바람이 느려지고, 멈추거나 페더링하면
덜 뽑아가므로 더 빠르게 읽힌다.**

**⇒ 나셀 풍속에 "터빈이 돌고 있었는가"라는 정보가 섞여 있을 수 있다.**

**왜 이게 심각한가 (DS)**: 그렇다면 오라클은 *"바람을 완벽히 알았을 때"* 가 아니라
*"바람 + 터빈 가동 상태를 알았을 때"* 를 재고 있는 것이다. 그러면

- 04 14절의 σ 분해(**풍속오차 0.146 ⊕ 예측불가 0.079**)에서 0.079가 실제보다 **작게** 나온다
- 거기서 유도한 **"풍속 RMSE를 1.2~1.3으로 내리면 1등"** 이 과대 낙관이 된다
- 05의 **"격자 CNN만 남았다"** 는 로드맵도 근거가 약해진다

이 세 결론이 전부 오라클 하나에 매달려 있는데, **오라클 자체를 검증한 적이 없다.**

**방법**: 예보 풍속(`ws10_nearest`)으로 구간을 나눠 **같은 바람 조건**을 만든 뒤,
그 안에서 `가동률 = 1`인 시간과 `< 1`인 시간의 나셀 풍속 평균을 비교한다.

- 차이가 거의 없다(< 0.2 m/s) → 오라클 깨끗함. 기존 결론 유지
- **멈췄을 때 더 빠르게 읽힌다** → 순환 확인. 상한과 로드맵을 다시 계산해야 한다

In [7]:
MIN_PER_CELL = 30          # 구간 안에서 양쪽 다 이만큼은 있어야 비교한다

print("나셀 풍속(SCADA) 차이  =  [정지 있음] − [전대수 가동]      단위 m/s")
print("예보 풍속으로 구간을 고정했으므로 '같은 바람일 때'의 비교다.\n")

for g in TARGET_COLS:
    d = pd.DataFrame({"fc": ctx.train[f"{g}_ws10_nearest"],
                      "ws": ctx.train[f"scada_ws_{g}"],
                      "av": ctx.avail[g]}).dropna()
    d["bin"] = pd.cut(d["fc"], np.arange(0, 15, 1.0))

    out = []
    for thr, name in [(0.999, "1대 이상 정지"), (0.5, "절반 이상 정지")]:
        d["down"] = d["av"] < thr
        t = d.groupby(["bin", "down"], observed=True)["ws"].agg(["mean", "count"]).unstack("down")
        if True not in t["mean"].columns or False not in t["mean"].columns:
            out.append((name, np.nan, 0)); continue
        ok = (t["count"][True] >= MIN_PER_CELL) & (t["count"][False] >= MIN_PER_CELL)
        if not ok.any():
            out.append((name, np.nan, 0)); continue
        diff = (t["mean"][True] - t["mean"][False])[ok]
        w = t["count"][True][ok]                       # 정지 표본 수로 가중
        out.append((name, float((diff * w).sum() / w.sum()), int(w.sum())))

    line = "  ".join(f"{n}: {v:+.3f} (n={c})" if np.isfinite(v) else f"{n}: 표본부족"
                     for n, v, c in out)
    print(f"{g:14s}  {line}")

print("\n판정: 양수 = 멈췄을 때 더 빠르게 읽힘 = 로터 후류 효과가 실재 = 오라클에 가동상태가 샌다")
print("      |차이| < 0.2 → 무시 가능 / 0.2~0.5 → 주의 / > 0.5 → 오라클 상한을 다시 계산해야 함")

나셀 풍속(SCADA) 차이  =  [정지 있음] − [전대수 가동]      단위 m/s
예보 풍속으로 구간을 고정했으므로 '같은 바람일 때'의 비교다.

kpx_group_1     1대 이상 정지: +0.652 (n=4296)  절반 이상 정지: -0.482 (n=521)
kpx_group_2     1대 이상 정지: +0.369 (n=4106)  절반 이상 정지: +0.041 (n=512)
kpx_group_3     1대 이상 정지: +0.557 (n=3922)  절반 이상 정지: +1.112 (n=148)

판정: 양수 = 멈췄을 때 더 빠르게 읽힘 = 로터 후류 효과가 실재 = 오라클에 가동상태가 샌다
      |차이| < 0.2 → 무시 가능 / 0.2~0.5 → 주의 / > 0.5 → 오라클 상한을 다시 계산해야 함


---

## 2-3. 부정행위를 걷어내고 다시 재기

### 앞 셀에서 밝혀진 문제

2-2에서 **터빈이 멈춰 있을 때 풍속계가 0.37~0.65 m/s 더 빠르게 읽힌다**는 걸 확인했다.
날개가 돌면 바람에서 힘을 뽑아가니 날개 뒤가 느려지고, 멈추면 그대로 지나가니 빨라진다.

그래서 "정답 바람을 알려준 모델"(앞으로 **정답바람 모델**이라 부른다)은
바람만 아는 게 아니라 **그날 터빈이 고장났는지까지 눈치챌 수 있었다.**
고장은 우리가 2025년에 절대 알 수 없는 정보다. 그러니 지금 나온
"바람만 좋아지면 σ가 51~61% 준다"는 값은 **부풀려져 있다.**

### 걷어내는 방법 — 아주 싸다

**모든 터빈이 정상 가동 중이던 시간만 골라서 채점한다.**
그 시간대에는 고장 여부가 전부 똑같으니(전부 정상) 눈치챌 정보 자체가 없다.
**학습을 다시 할 필요가 없다** — 이미 만들어둔 예측을 다른 행만 골라 채점하면 끝이다.

### 읽는 법 — 표가 2×2다

`σ`(시그마)는 **예측이 실제에서 평균적으로 얼마나 빗나가는지**를 설비용량 대비 비율로 쓴 것이다.
0.16이면 "평균적으로 설비용량의 16%만큼 빗나간다". **작을수록 좋다.**

|  | 전체 채점 시간 | 전 터빈 정상 가동 시간만 |
|---|---|---|
| **지금 모델** | ① 현재 실력 | ② 고장 시간을 뺀 현재 실력 |
| **정답바람 모델** | ③ 부풀려진 하한 | ④ **진짜 하한** |

여기서 세 가지를 읽는다.

- **① → ②** : 줄어든 만큼이 **고장·정비 때문에 틀린 몫**이다
- **② → ④** : 줄어든 만큼이 **바람을 몰라서 틀린 진짜 몫** ← 이게 우리가 노릴 수 있는 전부
- **④ 자체** : 바람도 알고 고장도 없는데 남는 오차. **아무리 해도 못 줄이는 바닥**

④가 여전히 크다면, 04가 말한 "이제 바람(격자 CNN)만 남았다"는 로드맵을 다시 짜야 한다.

### group_3도 여기서 갈린다

2-1에서 **바람을 완벽히 알려줬는데 group_3만 오차가 오히려 더 벌어졌다**(배율 1.116 → 1.320).
발전기 대수(5대 vs 6대)로 설명되는 몫은 1.095뿐이라 **3분의 1밖에 설명이 안 된다.**

group_3이 ②(고장 시간 제외)에서 정상으로 돌아오면 → 원인은 **고장·가동률**이다.
④에서도 여전히 튀면 → **UNISON 터빈 자체가 예측하기 어렵다**는 뜻이고, 그건 못 고친다.

**비용**: 학습 0회, 즉시.

In [8]:
importlib.reload(ms)          # per_group_scores에 '특정 행만 채점' 기능을 넣었다

SUBSETS = {"전체 채점시간": False, "전터빈 정상가동만": True}
VARIANTS = {"지금 모델": "L0_lgb", "정답바람 모델": "L2_oracle"}

rows = []
for vlabel, variant in VARIANTS.items():
    for slabel, only_full in SUBSETS.items():
        acc = []
        for fold, info in ctx.fold_info.items():
            vi = info["valid_idx"]
            pred = pd.DataFrame({g: PRED[(fold, variant, g)] for g in TARGET_COLS}, index=vi)
            mask = None
            if only_full:
                # 가동률 1.0 = 그 시각 모든 터빈이 돌고 있었다 (결측은 정상으로 간주)
                av = ctx.avail.reindex(vi).fillna(1.0)
                mask = {g: (av[g] >= 0.999).to_numpy() for g in TARGET_COLS}
            acc.append(ms.per_group_scores(info["actual_df"], pred, row_mask=mask))
        m = pd.concat(acc).groupby(level=0).mean()      # 4개 fold 평균
        for g in TARGET_COLS:
            rows.append(dict(모델=vlabel, 시간대=slabel, group=g.split("_")[-1],
                             σ=m.loc[g, "sigma"], FICR=m.loc[g, "ficr"], 채점행=m.loc[g, "n"]))
tab = pd.DataFrame(rows)

for col in ["σ", "FICR", "채점행"]:
    print(f"\n■ {col}")
    p = tab.pivot_table(index="group", columns=["모델", "시간대"], values=col)
    p = p.reindex(columns=pd.MultiIndex.from_product([list(VARIANTS), list(SUBSETS)]))
    display(p.round(4) if col != "채점행" else p.round(0))

# --- 오차를 세 몫으로 쪼갠다 ------------------------------------------------
s = tab.set_index(["모델", "시간대", "group"])["σ"]
print("\n" + "=" * 68)
print("오차 폭(σ)을 셋으로 쪼개기      ※ 4개 fold 평균")
print("=" * 68)
print(f"{'group':>6} | {'①지금':>7} {'②고장뺌':>8} {'④진짜하한':>9} | "
      f"{'고장 몫':>8} {'바람 몫':>8} {'남는 몫':>8}")
print("-" * 68)
for g in ["1", "2", "3"]:
    a = s[("지금 모델", "전체 채점시간", g)]
    b = s[("지금 모델", "전터빈 정상가동만", g)]
    d = s[("정답바람 모델", "전터빈 정상가동만", g)]
    print(f"{g:>6} | {a:7.4f} {b:8.4f} {d:9.4f} | "
          f"{(a-b)/a*100:7.1f}% {(b-d)/a*100:7.1f}% {d/a*100:7.1f}%")
print("-" * 68)
print("고장 몫 = 정비·고장 시간대라서 틀린 것 (우리가 예측 못 하는 정보)")
print("바람 몫 = 바람을 몰라서 틀린 것  ← 격자 CNN 등이 노릴 수 있는 전부")
print("남는 몫 = 바람도 알고 고장도 없는데 남는 것 (아무리 해도 못 줄임)")


■ σ


지금 모델           정답바람 모델          
      전체 채점시간 전터빈 정상가동만 전체 채점시간 전터빈 정상가동만
group                                    
1      0.1644    0.1598  0.0641    0.0374
2      0.1662    0.1559  0.0717    0.0353
3      0.1845    0.1805  0.0897    0.0677


■ FICR


지금 모델           정답바람 모델          
      전체 채점시간 전터빈 정상가동만 전체 채점시간 전터빈 정상가동만
group                                    
1      0.4225    0.4338  0.8691    0.9353
2      0.4557    0.4798  0.8484    0.9227
3      0.3377    0.3384  0.7046    0.8021


■ 채점행


지금 모델           정답바람 모델          
      전체 채점시간 전터빈 정상가동만 전체 채점시간 전터빈 정상가동만
group                                    
1      3152.0    2763.0  3152.0    2763.0
2      3138.0    2726.0  3138.0    2726.0
3      2833.0    1896.0  2833.0    1896.0


오차 폭(σ)을 셋으로 쪼개기      ※ 4개 fold 평균
 group |     ①지금     ②고장뺌     ④진짜하한 |     고장 몫     바람 몫     남는 몫
--------------------------------------------------------------------
     1 |  0.1644   0.1598    0.0374 |     2.8%    74.5%    22.7%
     2 |  0.1662   0.1559    0.0353 |     6.2%    72.6%    21.2%
     3 |  0.1845   0.1805    0.0677 |     2.1%    61.2%    36.7%
--------------------------------------------------------------------
고장 몫 = 정비·고장 시간대라서 틀린 것 (우리가 예측 못 하는 정보)
바람 몫 = 바람을 몰라서 틀린 것  ← 격자 CNN 등이 노릴 수 있는 전부
남는 몫 = 바람도 알고 고장도 없는데 남는 것 (아무리 해도 못 줄임)


---

# 3층 — group_3의 바닥이 왜 두 배인가

## 2-3에서 나온 것

| group | 지금 오차 | 고장 몫 | 바람 몫 | **못 줄이는 바닥** |
|---|---|---|---|---|
| 1 | 0.1644 | 2.8% | 74.5% | 0.0374 (22.7%) |
| 2 | 0.1662 | 6.2% | 72.6% | 0.0353 (21.2%) |
| **3** | 0.1845 | 2.1% | 61.2% | **0.0677 (36.7%)** |

**세 가지가 정해졌다.**

**① 고장·정비 카드는 닫는다.** 고장 시간을 통째로 빼도 오차가 2~6%밖에 안 줄었다.
특히 group_3은 채점 시간의 **3분의 1**(2,833 → 1,896행)이 "1대 이상 정지" 상태인데도
빼나 마나였다. **`B_norm`(가동률로 라벨 보정)이 이미 제 몫을 다 하고 있다**는 뜻이다.
FICR로 봐도 최대 이득이 0.024(group_2)라 총점으로는 0.004 미만 — **채택 문턱(0.002) 근처**다.

**② 바람은 여전히 가장 큰 덩어리다** (61~75%). 04의 방향 자체는 맞았다.
다만 88%가 아니라 61~75%다.

**③ group_3의 바닥이 다른 둘의 1.87배다** (0.0677 vs 0.0374 / 0.0353).
**바람도 완벽히 알고, 터빈도 전부 정상 가동 중인데** 그렇다.
발전기 대수(5대 vs 6대)로 설명되는 건 1.095배뿐이라 **거의 설명이 안 된다.**

## 그래서 무엇이 남았나 — 세 가지 후보

**(a) 학습 데이터가 절반이다.** group_3은 2022년이 통째로 없다.
바람이 부정확할 때는 바람 오차가 워낙 커서 데이터 부족이 가려졌지만,
**바람을 완벽히 알려주면 그때부터는 "얼마나 잘 배웠나"가 병목**이 된다.
→ 고칠 수 있다 (다른 그룹 데이터를 같이 쓰는 등)

**(b) 파워커브를 그릴 재료가 부족하다.** 바람 세기별 발전량 곡선을 SCADA로 그리는데,
group_3은 그 재료가 절반이다. 특히 표본이 드문 강풍 구간이 엉성할 수 있다.
→ 고칠 수 있다 (구간을 넓히거나 매끄럽게 다듬기)

**(c) UNISON 터빈이 원래 들쭉날쭉하다.** 제조사가 다르면 날개 각도 제어 방식도 다르다.
같은 바람에도 출력이 더 흔들릴 수 있다.
→ **못 고친다.** 이거면 카드를 닫는다

## 이 셀이 가르는 법 — 전부 학습 0회

**(a) 판정**: 정답바람 모델의 오차를 fold별로 본다.
fold1은 group_3 학습 데이터가 6개월, fold3은 30개월이다.
데이터가 5배 늘 때 바닥이 뚜렷이 낮아지면 → **(a)가 범인**이다. 그대로면 아니다.

**(b) 판정**: 파워커브를 그릴 때 쓰는 0.5 m/s 칸마다 SCADA 표본이 몇 개인지 센다.
발전이 실제로 일어나는 3~15 m/s 구간에서 표본이 빈약한 칸이 group_3에만 많으면 → **(b)**.

둘 다 아니면 **(c)** 로 보고 group_3 카드를 닫는다.

In [9]:
FOLD_ORDER = [ms.A_FOLD, *ms.B_FOLDS]

# ── (a) 학습 데이터가 늘면 '못 줄이는 바닥'이 낮아지나 ──────────────────────
print("■ 정답바람 모델의 오차 폭 — fold별")
o = bench.df.query("variant == 'L2_oracle'")
sig = o.pivot_table(index="group", columns="fold", values="sigma").reindex(columns=FOLD_ORDER)
display(sig.round(4))

print("\n■ 그 fold에서 실제로 쓴 학습 행 수 (라벨이 있는 시간)")
nrow = pd.DataFrame(
    {fold: {g: int((info["train_mask"] & ctx.train[g].notna()).sum()) for g in TARGET_COLS}
     for fold, info in ctx.fold_info.items()}).reindex(columns=FOLD_ORDER)
display(nrow)

print("\n데이터가 늘 때 바닥이 낮아졌나  (fold1 → fold3, 학습행 배수와 함께)")
for g in TARGET_COLS:
    s1, s3 = sig.loc[g, "B안 fold1"], sig.loc[g, "B안 fold3"]
    n1, n3 = nrow.loc[g, "B안 fold1"], nrow.loc[g, "B안 fold3"]
    print(f"  {g:14s} 학습행 {n1:5d} → {n3:5d} ({n3/n1:.1f}배)   "
          f"오차 {s1:.4f} → {s3:.4f} ({(s3/s1-1)*100:+.1f}%)")

# ── (b) 파워커브를 그릴 재료가 충분한가 ────────────────────────────────────
print("\n\n■ 파워커브 재료 — 0.5 m/s 칸마다 SCADA 표본이 몇 개인가")
EDGES = np.arange(0, 25.5, 0.5)
MIN_OK = 30                      # 이 미만이면 그 칸의 평균은 못 믿는다
for g in TARGET_COLS:
    ws, y = ctx.train[f"scada_ws_{g}"], ctx.train[g]
    ok = ws.notna() & y.notna()
    cnt = pd.cut(ws[ok], EDGES).value_counts().sort_index()
    mid = np.array([iv.mid for iv in cnt.index])
    core = (mid >= 3) & (mid <= 15)                    # 발전이 실제로 일어나는 구간
    thin = int((cnt[core] < MIN_OK).sum())
    print(f"  {g:14s} 표본 {int(ok.sum()):6d}개 | 3~15 m/s 칸 {core.sum()}개 중 "
          f"표본 {MIN_OK}개 미만인 칸 = {thin}개"
          + (f"   (가장 얇은 칸 {int(cnt[core].min())}개)" if core.sum() else ""))

print("\n  ※ 얇은 칸이 group_3에만 많으면 (b) 파워커브 재료 부족이 범인이다.")
print("     세 그룹이 비슷하면 (b)는 아니다.")

■ 정답바람 모델의 오차 폭 — fold별


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3
group,,,,
kpx_group_1,0.0573,0.0861,0.0601,0.0529
kpx_group_2,0.0688,0.0840,0.0818,0.0522
kpx_group_3,0.0841,0.1099,0.0708,0.0938



■ 그 fold에서 실제로 쓴 학습 행 수 (라벨이 있는 시간)


,A안(2024),B안 fold1,B안 fold2,B안 fold3
kpx_group_1,17341,12925,17341,21371
kpx_group_2,17341,12925,17341,21371
kpx_group_3,8662,4246,8662,12692



데이터가 늘 때 바닥이 낮아졌나  (fold1 → fold3, 학습행 배수와 함께)
  kpx_group_1    학습행 12925 → 21371 (1.7배)   오차 0.0861 → 0.0529 (-38.5%)
  kpx_group_2    학습행 12925 → 21371 (1.7배)   오차 0.0840 → 0.0522 (-37.9%)
  kpx_group_3    학습행  4246 → 12692 (3.0배)   오차 0.1099 → 0.0938 (-14.6%)


■ 파워커브 재료 — 0.5 m/s 칸마다 SCADA 표본이 몇 개인가
  kpx_group_1    표본  25788개 | 3~15 m/s 칸 24개 중 표본 30개 미만인 칸 = 0개   (가장 얇은 칸 170개)
  kpx_group_2    표본  25788개 | 3~15 m/s 칸 24개 중 표본 30개 미만인 칸 = 0개   (가장 얇은 칸 294개)
  kpx_group_3    표본  17099개 | 3~15 m/s 칸 24개 중 표본 30개 미만인 칸 = 0개   (가장 얇은 칸 106개)

  ※ 얇은 칸이 group_3에만 많으면 (b) 파워커브 재료 부족이 범인이다.
     세 그룹이 비슷하면 (b)는 아니다.


## 3-2. 조건을 똑같이 맞춰 다시 비교 — group_3이 불리한 게 데이터 때문인가

### 앞 셀 결과

**(b) 파워커브 재료 부족 → 기각.** 세 그룹 모두 3~15 m/s 구간에서 표본이 모자란 칸이
**0개**였다. group_3의 가장 얇은 칸도 106개다. 곡선을 그릴 재료는 충분하다.

**(a) 데이터 양 → 아직 못 가름.**

| group | 학습행 (fold1 → fold3) | 오차 변화 |
|---|---|---|
| 1 | 12,925 → 21,371 (1.7배) | -38.5% |
| 2 | 12,925 → 21,371 (1.7배) | -37.9% |
| 3 | 4,246 → 12,692 (**3.0배**) | **-14.6%** |

group_3은 데이터가 세 배 늘었는데 절반도 못 따라왔다. 데이터 탓이 아닌 것처럼 보인다.

**그런데 이 비교는 공정하지 않다.** fold1과 fold3은 학습량만 다른 게 아니라
**검증하는 기간 자체가 다르다**(2023년 하반기 vs 2024년 하반기). 실제로 group_3의
fold별 오차는 0.1099 → 0.0708 → 0.0938로 **오르락내리락**한다.
데이터가 늘수록 좋아지는 모양이 아니다.

### 이 셀의 방법 — 조건을 완전히 똑같이 만든다

**group_1/2의 학습 데이터를 group_3과 똑같이 2023년부터로 잘라낸다.**
group_3은 원래 2022년이 없으니 아무것도 안 바뀌고, 다른 둘만 절반으로 줄어든다.
그러면 세 그룹이 **같은 기간, 같은 양, 같은 검증 시간대**가 된다.

- 조건을 맞춰도 group_1/2가 훨씬 정확하다 → **(c) UNISON 터빈이 원래 어렵다.** 카드를 닫는다
- group_1/2도 group_3만큼 나빠진다 → **(a) 데이터 부족이 범인.** 고칠 수 있다

### 공정성을 위해 챙긴 것

- **두 조건을 이 셀에서 나란히 새로 돌린다.** 앞의 정답바람 모델은 SCADA 결측을 우리 추정
  풍속으로 메웠는데 여기서는 중앙값으로 메운다. 채우는 방식이 다르면 비교가 흐려지므로
  **"전체 기간"판도 같은 방식으로 다시 돌린다.** 바뀌는 것은 **학습 기간 하나뿐**이다
- 쓰는 피처 목록은 양쪽 다 기준선과 똑같이 고정한다
- 채점은 **전 터빈 정상 가동 시간만** (2-3과 같은 기준 — 고장 정보가 새는 걸 막는다)

**덤으로 답이 나오는 것**: 04에서 "2022년 group_1 정지율이 0.538~0.616으로 이상하다"를
발견해놓고 아무 조치도 안 했다. 이 실험이 **2022년을 빼면 좋아지는가**에도 답한다.

**비용**: 학습 24회, 5~8분.

In [10]:
MATCH_START = pd.Timestamp("2023-01-01")      # group_3의 라벨이 시작되는 시점


def make_oracle_fn(match_period: bool):
    """정답 바람(SCADA 실측)을 넣은 모델. `match_period=True`면 학습을 2023년부터로 자른다.

    ⚠️ 두 조건에서 **딱 하나만** 다르게 한다 — 학습 기간.
       결측 채우는 방식·피처 목록·라벨·τ는 전부 같다.
    """
    def fit_fn(ctx, g, cv_suffix, train_mask, valid_idx):
        tm = train_mask & (ctx.train["kst_dtm"] >= MATCH_START) if match_period else train_mask
        ws = ctx.train[f"scada_ws_{g}"]
        ws = ws.fillna(ws.median()).clip(lower=0.0)          # 양쪽 동일하게 중앙값으로
        X, _ = pl.build_frame_with_wind(ctx, ctx.train, g, tm, ws, pl.WIND_TAG)
        keep = pl.keep_for_fold(ctx, g, cv_suffix, train_mask, pl.BEST_TOPN)   # 기준선과 동일
        m = pl.lgbm_train_label(ctx, X[keep], g, tm,
                                pl.DEFAULT_TAU, pl.SEED, pl.DEFAULT_LABEL_MODE)
        p = pd.Series(m.predict(X.loc[valid_idx, keep]), index=valid_idx)
        return p.clip(lower=0, upper=CAPACITY_KWH[g])
    return fit_fn


for name, matched in [("L3_full", False), ("L3_match23", True)]:
    print(f"\n--- {name} ({'2023년부터만 학습' if matched else '전체 기간 학습'}) ---")
    _ = pl.run_variant(ctx, PRED, name, make_oracle_fn(matched), verbose=False)
    bench.add_from_cache(ctx, PRED, name)
    print("   완료")


# --- 전 터빈 정상가동 시간만 골라 오차 폭을 비교한다 ------------------------
def clean_sigma(variant):
    acc = []
    for fold, info in ctx.fold_info.items():
        vi = info["valid_idx"]
        pred = pd.DataFrame({g: PRED[(fold, variant, g)] for g in TARGET_COLS}, index=vi)
        av = ctx.avail.reindex(vi).fillna(1.0)
        mask = {g: (av[g] >= 0.999).to_numpy() for g in TARGET_COLS}
        acc.append(ms.per_group_scores(info["actual_df"], pred, row_mask=mask))
    return pd.concat(acc).groupby(level=0).mean()["sigma"]


full, match = clean_sigma("L3_full"), clean_sigma("L3_match23")
cmp = pd.DataFrame({"전체 기간 학습": full, "2023년부터만 학습": match})
cmp["2022년 빼면"] = (cmp["2023년부터만 학습"] / cmp["전체 기간 학습"] - 1) * 100
print("\n■ 못 줄이는 바닥 (정답바람 + 전터빈 정상가동)")
display(cmp.round(4))

G1, G2, G3 = TARGET_COLS
r_full = full[G3] / full[[G1, G2]].mean()
r_match = match[G3] / match[[G1, G2]].mean()
print(f"\n■ group_3이 다른 둘보다 몇 배 나쁜가")
print(f"   전체 기간 학습     : {r_full:.3f}")
print(f"   조건을 맞췄을 때   : {r_match:.3f}   ← 이게 판정 대상")
print(f"   발전기 대수로 설명되는 몫 : 1.095")
print()
if r_match <= 1.15:
    print("   ⇒ (a) 데이터 부족이 범인. 조건을 맞추니 격차가 거의 사라졌다.")
    print("      group_1/2의 데이터를 group_3 학습에 끌어다 쓰는 카드가 유효하다.")
elif r_match >= 1.5:
    print("   ⇒ (c) UNISON 터빈이 원래 어렵다. 데이터를 똑같이 줘도 못 따라온다.")
    print("      **group_3 카드를 닫는다.** 남은 시간을 바람의 공간정보와 밴드 배치에 쓴다.")
else:
    print("   ⇒ 절반씩. 데이터로 일부는 회수 가능하나 나머지는 터빈 특성이다.")

print("\n■ 덤 — 2022년을 빼는 게 group_1/2에 도움이 되나 (04의 미해결 질문 2번)")
for g in [G1, G2]:
    d = cmp.loc[g, "2022년 빼면"]
    print(f"   {g:14s} {d:+.1f}%  " + ("(빼는 게 낫다)" if d < -2 else
          "(빼면 나빠진다 — 2022년도 쓸모가 있다)" if d > 2 else "(차이 없음)"))


--- L3_full (전체 기간 학습) ---


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

   완료

--- L3_match23 (2023년부터만 학습) ---


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

   완료

■ 못 줄이는 바닥 (정답바람 + 전터빈 정상가동)


,전체 기간 학습,2023년부터만 학습,2022년 빼면
group,,,
kpx_group_1,0.0374,0.0350,-6.5185
kpx_group_2,0.0353,0.0367,4.2051
kpx_group_3,0.0670,0.0670,0.0000



■ group_3이 다른 둘보다 몇 배 나쁜가
   전체 기간 학습     : 1.843
   조건을 맞췄을 때   : 1.868   ← 이게 판정 대상
   발전기 대수로 설명되는 몫 : 1.095

   ⇒ (c) UNISON 터빈이 원래 어렵다. 데이터를 똑같이 줘도 못 따라온다.
      **group_3 카드를 닫는다.** 남은 시간을 바람의 공간정보와 밴드 배치에 쓴다.

■ 덤 — 2022년을 빼는 게 group_1/2에 도움이 되나 (04의 미해결 질문 2번)
   kpx_group_1    -6.5%  (빼는 게 낫다)
   kpx_group_2    +4.2%  (빼면 나빠진다 — 2022년도 쓸모가 있다)


---

# 4층 — 남은 카드는 둘뿐이다

## 3-2에서 닫힌 것들

**① group_3 특별대우 → 닫는다.**
학습 기간을 group_3과 똑같이 2023년부터로 맞췄는데도 격차가 그대로였다(1.843 → **1.868**).
데이터를 똑같이 줘도 못 따라온다 = **UNISON 터빈 자체가 예측하기 어렵다.**

01 보고서에 이미 단서가 있었다 — **라벨과 SCADA의 상관이 group_1/2는 0.9998인데
group_3만 0.9966**이다. 자기 터빈이 실제로 낸 전력과 KPX가 기록한 발전량이 유독 덜 맞는다.
설명 못 하는 변동이 **17배** 크다는 뜻이고, 이건 어떤 모델로도 못 맞힌다.

**② 데이터를 더 모으는 방향 → 닫는다.**
group_1의 학습 데이터를 **절반으로 줄였는데 바닥이 오히려 6.5% 좋아졌다.**
이미 데이터가 남아돈다는 뜻이다. 다른 그룹 데이터를 끌어다 쓰는 카드도 같이 닫힌다.

**③ 2022년 제외 → 닫는다** (04의 미해결 질문 2번, 드디어 답).
group_1은 -6.5%(빼는 게 낫다), group_2는 +4.2%(빼면 나빠진다). **부호가 반대**다.
우리 판정 규칙("모든 fold·그룹에서 같은 방향이어야 채택")에 걸려 **효과 불확실**이다.

## 그래서 남은 것

| 몫 | 크기 | 상태 |
|---|---|---|
| 고장·정비 | 2~6% | 닫힘 (2-3) |
| **바람** | **61~75%** | 조건별 보정 9개 축은 05에서 전부 닫힘. **공간 정보만 미시도** |
| 못 줄이는 바닥 | 21~37% | 줄일 수 없음 → **밴드 안 배치**로 짜내는 수밖에 |

**카드가 정확히 둘 남았다.**
1. **바람의 공간 정보** ← 이 셀이 값어치를 먼저 잰다
2. **밴드 안 배치** (행마다 예측을 어디 놓을지 다르게)

## 이 셀 — 만들기 전에 값어치부터 잰다 (학습 0회)

05에서 지형 speed-up 피처를 **만들고 나서** 재봤다가 반나절을 버렸다.
그 뒤로 정한 규칙이 **"피처를 만들기 전에 상한을 먼저 계산한다"** 이다. 그대로 따른다.

### 무엇을 재나

GFS 격자는 3×3, 0.25도 간격이다. 실제 크기로는 **가로 44 km × 세로 55 km**.
풍속 10 m/s면 바람이 가장자리에서 중심까지 오는 데 **1시간 넘게** 걸린다.
1시간 단위 예측에서 **의미 있는 시간차**다. (LDAPS 16격자는 6 km 상자라 10분이면
통과해버려서 이 용도로는 못 쓴다.)

그래서 세 가지를 만들어 각각의 상한을 잰다.

- **상류 풍속 − 현재 풍속**: 바람이 불어오는 쪽 격자가 지금보다 센가 약한가
  → 앞으로 바람이 세질지 약해질지의 힌트
- **바람 방향 풍속 기울기**: 9개 격자에 직선을 맞춰 구한 기울기
- **9격자 산포**: 격자마다 값이 얼마나 다른가 → 예보 자체의 불확실성

### 검산 하나를 같이 돌린다

**풍향(30도) 축**도 같이 잰다. 05에서 이 축의 상한을 **-0.096%** 로 측정했었다.
비슷한 값이 나오면 이 계산이 05와 같은 자를 쓰고 있다는 뜻이다.
크게 다르면 **거기서 멈추고** 계산부터 고쳐야 한다.

### 판정

`σ감소상한%`가 **-1%보다 작으면(더 음수면) 만들 값어치 있음**, 아니면 기각.
음수일수록 좋다.

**비용**: 학습 0회, 1분.

In [11]:
importlib.reload(ms)

# ── GFS 격자 기하: 3x3, 0.25도 간격. 중심 g5에 터빈이 있다 ──────────────────
gm = pd.read_parquet(pl.PROCESSED_DIR / "gfs_grid_meta.parquet").set_index("grid_id")
gids = gm.index.to_numpy()
i5 = int(np.where(gids == 5)[0][0])
lat0, lon0 = gm.loc[5, "latitude"], gm.loc[5, "longitude"]
dx = ((gm["longitude"] - lon0) * 111.32 * np.cos(np.radians(lat0))).to_numpy()   # 동쪽 + (km)
dy = ((gm["latitude"] - lat0) * 110.57).to_numpy()                               # 북쪽 + (km)
print(f"격자 범위: 가로 {dx.max()-dx.min():.0f} km × 세로 {dy.max()-dy.min():.0f} km")

# 허브고도에 가장 가까운 100 m 바람을 쓴다
U = np.column_stack([ctx.train[f"gfs_g{i}_heightAboveGround_100_100u"] for i in gids])
V = np.column_stack([ctx.train[f"gfs_g{i}_heightAboveGround_100_100v"] for i in gids])
WS = np.hypot(U, V)                                    # (시간, 격자9)

# 중심 격자의 바람으로 '불어오는 쪽' 단위벡터를 만든다
u0, v0 = U[:, i5], V[:, i5]
spd0 = np.maximum(np.hypot(u0, v0), 1e-6)
ux, uy = -u0 / spd0, -v0 / spd0                        # 상류 방향
proj = ux[:, None] * dx[None, :] + uy[:, None] * dy[None, :]   # 상류일수록 +

# ① 바람 방향으로의 풍속 기울기 (격자 9점에 직선 맞추기)
pm, wm = proj.mean(1, keepdims=True), WS.mean(1, keepdims=True)
grad_along = ((proj - pm) * (WS - wm)).sum(1) / np.maximum(((proj - pm) ** 2).sum(1), 1e-6)
# ② 상류 격자 가중평균 풍속 − 현재 격자 풍속
w = np.clip(proj, 0, None)
w = w / np.maximum(w.sum(1, keepdims=True), 1e-6)
adv_minus_here = (w * WS).sum(1) - WS[:, i5]
# ③ 9격자 산포
grid_std = WS.std(1)
# 검산용 풍향 (불어오는 쪽, 0~360)
wd_from = np.degrees(np.arctan2(-u0, -v0)) % 360

# ── 검증 구간의 잔차를 모은다 (fold 4개 × 그룹 3개) ─────────────────────────
parts = []
for fold, info in ctx.fold_info.items():
    vi = info["valid_idx"]
    pos = ctx.train.index.get_indexer(vi)
    for g in TARGET_COLS:
        est = pl.wind_estimate(ctx, g, info["cv_suffix"], info["train_mask"]).loc[vi].to_numpy()
        tru = ctx.train[f"scada_ws_{g}"].loc[vi].to_numpy()
        act = info["actual_df"][g].to_numpy(dtype=float)
        prd = PRED[(fold, "L0_lgb", g)].to_numpy()
        parts.append(pd.DataFrame({
            "g": g,
            "res_ws": est - tru,                                   # 풍속 오차
            "res_pw": (prd - act) / CAPACITY_KWH[g],               # 발전량 오차(용량 대비)
            "scored": act >= CAPACITY_KWH[g] * 0.10,
            "풍향(30도)": (wd_from[pos] // 30).astype(int),
            "상류-현재 풍속차": adv_minus_here[pos],
            "바람방향 풍속기울기": grad_along[pos],
            "9격자 산포": grid_std[pos],
        }))
d = pd.concat(parts, ignore_index=True)
AXES = ["풍향(30도)", "상류-현재 풍속차", "바람방향 풍속기울기", "9격자 산포"]

print(f"\n잔차 표본: 풍속 {d['res_ws'].notna().sum():,}행 / 발전량(채점행) {d['scored'].sum():,}행")

for label, col, sub in [("풍속 오차", "res_ws", d[d["res_ws"].notna()]),
                        ("발전량 오차", "res_pw", d[d["scored"]])]:
    print(f"\n{'='*62}\n■ {label}를 줄일 수 있는 상한  (음수일수록 좋다, -1% 미만이면 채택)\n{'='*62}")
    for g in TARGET_COLS:
        s = sub[sub["g"] == g]
        t = ms.headroom_table(s[col], {a: s[a] for a in AXES}, min_count=50)
        print(f"\n  [{g}]")
        print(t[["σ감소상한%", "구간수", "표본", "판정"]].round(3).to_string())

print("\n" + "="*62)
print("검산: '풍향(30도)' 상한이 05 보고서의 -0.096%와 비슷해야 한다.")
print("      크게 다르면 계산이 05와 다른 자를 쓰고 있는 것이므로 여기서 멈춘다.")

격자 범위: 가로 44 km × 세로 55 km

잔차 표본: 풍속 65,940행 / 발전량(채점행) 36,492행

■ 풍속 오차를 줄일 수 있는 상한  (음수일수록 좋다, -1% 미만이면 채택)

  [kpx_group_1]
            σ감소상한%  구간수     표본             판정
label                                        
9격자 산포      -0.229   12  21984  기각(문턱 -1% 미달)
풍향(30도)     -0.182   12  21984  기각(문턱 -1% 미달)
상류-현재 풍속차   -0.170   12  21984  기각(문턱 -1% 미달)
바람방향 풍속기울기  -0.089   12  21984  기각(문턱 -1% 미달)

  [kpx_group_2]
            σ감소상한%  구간수     표본             판정
label                                        
상류-현재 풍속차   -0.208   12  21984  기각(문턱 -1% 미달)
풍향(30도)     -0.150   12  21984  기각(문턱 -1% 미달)
9격자 산포      -0.140   12  21984  기각(문턱 -1% 미달)
바람방향 풍속기울기  -0.129   12  21984  기각(문턱 -1% 미달)

  [kpx_group_3]
            σ감소상한%  구간수     표본             판정
label                                        
풍향(30도)     -0.444   12  21972  기각(문턱 -1% 미달)
9격자 산포      -0.244   12  21972  기각(문턱 -1% 미달)
바람방향 풍속기울기  -0.180   12  21972  기각(문턱 -1% 미달)
상류-현재 풍속차   -0.089   12  21972  기각(문턱 -1% 미달)

■ 발전량 오차를

---

# 5층 — 마지막 남은 카드: 밴드 안에 잘 앉히기

## 4층 결과 — 바람의 공간 정보도 닫혔다

세 가지 공간 피처의 상한을 다 쟀는데 **전부 문턱(-1%) 미달**이었다.

| 축 | 발전량 오차 상한 (그룹별) |
|---|---|
| 상류 − 현재 풍속차 | -0.83 / **-0.91** / -0.59 % |
| 풍향(30도) | -0.77 / -0.64 / -0.38 % |
| 9격자 산포 | -0.44 / -0.20 / -0.51 % |
| 바람방향 풍속기울기 | -0.16 / -0.15 / -0.11 % |

가장 좋은 게 **-0.91%**, 문턱에 못 미친다. 게다가 05에서 확인했듯 **이 상한은 실제 이득보다
후하게 나온다** — 풍향 축은 상한 -0.1%를 예측했는데 실제로 만들어 넣어보니 **+0.11%**
(개선 없음)였다. 즉 -0.9%짜리 상한은 실제로는 거의 0에 가깝다.

점수로 환산해도 넘지 못한다: σ를 0.9% 줄이면 FICR이 약 0.8% 오르고, 총점으로는
**+0.0017** — 채택 문턱 0.002 미만이다.

**⇒ 공간 피처를 만들지 않는다. 격자 CNN도 같은 정보를 노리므로 기대치를 같이 내린다.**

*(검산 참고: 풍향 축 상한이 -0.18/-0.15/-0.44%로 나왔다. 05의 -0.096%와 정확히 같지는
않지만 같은 자릿수·같은 판정이다. 그리고 05는 이 축을 실제로 만들어 "개선 없음"을 확인했으므로
자의 눈금은 검증된 셈이다.)*

## 그래서 남은 것은 하나뿐이다

| 오차의 몫 | 크기 | 상태 |
|---|---|---|
| 고장·정비 | 2~6% | 닫힘 |
| 바람 | 61~75% | **손잡이가 없음** — 조건별 보정 9축(05) + 공간 3축(4층) 전부 닫힘 |
| 못 줄이는 바닥 | 21~37% | 원래 못 줄임 |

**오차를 줄이는 길은 다 막혔다.** 남은 것은 **같은 오차로 점수를 더 받는 것**이다.

## 왜 이게 실제로 가능한가

점수의 절반인 FICR은 **계단**이다. 오차가 설비용량의 6% 안이면 4원, 8% 안이면 3원, 넘으면 0원.
**5.9%나 0.1%나 똑같이 4원**이다. 이미 들어온 예측을 더 정밀하게 만들어봐야 한 푼도 안 는다.

그러면 "예측을 어디에 놓아야 4원 구간에 들어갈 확률이 가장 높은가"라는 문제가 된다.
그리고 그 답은 **행마다 다르다.**

- 발전량이 정격 근처일 때는 실제값이 설비용량 쪽에 몰려 있다(위로 잘려 있음)
- 중간 풍속일 때는 좌우로 넓게 퍼져 있다

04는 이 문제를 **τ(분위수) 하나를 전체에 똑같이 적용**하는 방식으로 근사했고,
"위치 최적화는 끝났다"고 닫았다. **그건 모든 행을 같은 방향으로 미는 방식이 끝났다는 뜻**이지,
**행마다 다르게 놓는 방식**을 해봤다는 뜻이 아니다. 그건 한 번도 안 했다.

## 이 셀 — 먼저 '실제값이 어디쯤 나올지'의 폭을 알아야 한다

행마다 최적 위치를 찾으려면, 그 행에서 **실제 발전량이 어느 범위에 나올지**를 알아야 한다.
그래서 τ를 여러 개(0.10 / 0.25 / 0.50 / 0.75 / 0.90) 학습해 **분포의 윤곽**을 그린다.
τ=0.50은 이미 기준선에 있으므로 **네 개만 새로 학습**한다.

### 그리고 반드시 검산한다 — 이 분포를 믿어도 되는가

τ=0.10 예측보다 실제값이 작게 나오는 비율이 정말 10%인지 세어본다.

**특히 의심스러운 지점**: 우리 라벨은 `B_norm`(가동률로 나눠 보정한 값)이다.
고장으로 인한 변동을 미리 지워버렸으므로 **분포가 실제보다 좁게 나올 수 있다.**
좁은 분포를 믿고 배치하면 "확실하다"고 착각해서 오히려 손해다.

- 비율이 τ와 비슷하다 → 분포를 믿고 다음 셀로
- 비율이 τ보다 **한쪽으로 쏠린다** → 폭을 보정한 뒤에 쓴다

**비용**: 학습 48회, 6~10분.

In [12]:
TAUS_NEW = [0.10, 0.25, 0.75, 0.90]          # 0.50은 기준선(L0_lgb)을 그대로 쓴다

for tau in TAUS_NEW:
    name = f"Q{int(round(tau*100)):02d}"
    if (list(ctx.fold_info)[0], name, TARGET_COLS[0]) in PRED:
        print(f"{name} 이미 있음 — 건너뜀"); continue
    print(f"--- {name} (τ={tau:.2f}) 학습 중 ---")
    _ = pl.run_variant(ctx, PRED, name,
                       pl.lgbm_fold_fit_fn(mode="B_norm", tau=tau, top_n=200), verbose=False)

QVAR = {0.10: "Q10", 0.25: "Q25", 0.50: "L0_lgb", 0.75: "Q75", 0.90: "Q90"}
print("\n학습 완료:", list(QVAR.values()))

# ── 검산: τ 예측보다 실제값이 작게 나오는 비율이 정말 τ인가 ──────────────────
rows = []
for fold, info in ctx.fold_info.items():
    vi = info["valid_idx"]
    for g in TARGET_COLS:
        a = info["actual_df"][g].to_numpy(dtype=float)
        lab = np.isfinite(a)                                   # 라벨이 있는 행 전부
        scored = lab & (a >= CAPACITY_KWH[g] * 0.10)           # 실제 채점되는 행
        for tau, var in QVAR.items():
            q = PRED[(fold, var, g)].to_numpy()
            rows.append(dict(fold=fold, g=g, tau=tau,
                             전체행=float((a[lab] <= q[lab]).mean()),
                             채점행=float((a[scored] <= q[scored]).mean())))
cal = pd.DataFrame(rows)

for col in ["전체행", "채점행"]:
    print(f"\n■ 실제값이 τ 예측보다 작게 나온 비율 — {col}  (τ와 같아야 정상)")
    p = cal.pivot_table(index="tau", columns="g", values=col)
    p.columns = [c.split("_")[-1] for c in p.columns]
    p.insert(0, "목표(τ)", p.index)
    display(p.round(3))

print("""
읽는 법
  · '전체행'에서 비율이 τ와 비슷하면 분포 추정이 정상이다
  · 비율이 τ보다 **크면** 예측 분포가 실제보다 **위쪽에 치우쳐** 있다는 뜻
  · 비율이 τ보다 **작으면** 아래쪽에 치우쳐 있다는 뜻
  · 0.10과 0.90 두 끝이 안쪽으로 몰려 있으면(예: 0.18 / 0.82)
    → 분포가 **실제보다 좁다**. B_norm이 고장 변동을 지워서 그럴 수 있다. 폭 보정이 필요하다

  · '채점행'은 비율이 낮게 나오는 게 **정상**이다.
    발전량이 큰 시간만 골라 채점하므로 실제값이 예측보다 큰 쪽으로 쏠린다(선택 효과).
    다음 셀의 배치 계산은 이 선택 효과를 식 안에서 직접 다룬다.""")

--- Q10 (τ=0.10) 학습 중 ---


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

--- Q25 (τ=0.25) 학습 중 ---


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

--- Q75 (τ=0.75) 학습 중 ---


...


학습 완료: ['Q10', 'Q25', 'L0_lgb', 'Q75', 'Q90']

■ 실제값이 τ 예측보다 작게 나온 비율 — 전체행  (τ와 같아야 정상)


,목표(τ),1,2,3
tau,,,,
0.10,0.10,0.310,0.380,0.386
0.25,0.25,0.443,0.555,0.504
0.50,0.50,0.611,0.708,0.647
0.75,0.75,0.818,0.849,0.834
0.90,0.90,0.916,0.938,0.923



■ 실제값이 τ 예측보다 작게 나온 비율 — 채점행  (τ와 같아야 정상)


,목표(τ),1,2,3
tau,,,,
0.10,0.10,0.162,0.197,0.210
0.25,0.25,0.327,0.394,0.398
0.50,0.50,0.503,0.573,0.532
0.75,0.75,0.712,0.768,0.714
0.90,0.90,0.859,0.899,0.858



읽는 법
  · '전체행'에서 비율이 τ와 비슷하면 분포 추정이 정상이다
  · 비율이 τ보다 **크면** 예측 분포가 실제보다 **위쪽에 치우쳐** 있다는 뜻
  · 비율이 τ보다 **작으면** 아래쪽에 치우쳐 있다는 뜻
  · 0.10과 0.90 두 끝이 안쪽으로 몰려 있으면(예: 0.18 / 0.82)
    → 분포가 **실제보다 좁다**. B_norm이 고장 변동을 지워서 그럴 수 있다. 폭 보정이 필요하다

  · '채점행'은 비율이 낮게 나오는 게 **정상**이다.
    발전량이 큰 시간만 골라 채점하므로 실제값이 예측보다 큰 쪽으로 쏠린다(선택 효과).
    다음 셀의 배치 계산은 이 선택 효과를 식 안에서 직접 다룬다.


## 5-2. 행마다 최적 위치 찾기

### 5-1 검산 결과 — 분포를 그대로 믿으면 안 된다

**전체행**에서 비율이 τ보다 전부 위였다(τ=0.10인데 실제로는 0.31~0.39가 아래).
원인은 명확하다 — 우리 모델은 **표본가중**(실제발전량/용량, 하한 0.1)을 걸고 학습한다.
발전량 0인 시간은 가중치가 0.1, 정격인 시간은 1.0이다. **10배 차이**다.
그래서 모델이 추정하는 건 보통 분위수가 아니라 **가중 분위수**이고,
가중 없이 세면 어긋나는 게 당연하다.

**채점행**만 보면 훨씬 낫다.

| τ | group_1 | group_2 | group_3 |
|---|---|---|---|
| 0.10 | 0.162 | 0.197 | 0.210 |
| 0.50 | **0.503** | 0.573 | 0.532 |
| 0.90 | 0.859 | 0.899 | 0.858 |

group_1은 중앙값이 0.503으로 거의 정확하다. 다만 **양 끝이 안쪽으로 몰려 있다**
(아래 0.16 > 0.10, 위 0.86 < 0.90) — 즉 **추정한 분포가 실제보다 좁다.**
`B_norm`이 고장으로 인한 변동을 미리 지운 탓으로 보인다.
group_2는 중앙값이 0.573이라 **위로 치우쳐** 있다(이미 알던 +0.036 편향과 일치).

**⇒ 폭(spread)과 위치(shift) 손잡이 두 개로 보정한 뒤에 쓴다.**

### 손잡이를 어디서 맞추나 — 누수를 막는 방법

보정값을 검증 구간에서 맞추면 그 구간 점수는 당연히 좋아진다(**부정행위**).
그래서 **시간 순서**를 쓴다.

```
fold1 검증구간 (2023년 하반기)  ← 여기서 손잡이를 맞춘다
        ↓ 그 뒤 기간에만 적용
fold2 (2024 상반기) · fold3 (2024 하반기) · A안 (2024 전체)  ← 여기서 채점한다
```

실제 운영에서도 "작년 데이터로 보정값을 정하고 올해에 적용"하는 것과 같다.
**fold1 점수는 낙관적이므로 판정에서 뺀다.**

### 계산은 어떻게 하나

각 행에서 실제 발전량이 어느 범위에 나올지를 분위수 5개로 그리고,
그 분포 위에서 **점수 기여가 가장 큰 위치**를 격자탐색으로 찾는다.

```
기여 = −(0.5/N)·E[채점여부 · |f−a|/용량]  +  (0.125/A)·E[채점여부 · a · 단가]
```

`N`(채점 행 수)과 `A`(채점 행 발전량 합)는 **그 fold의 학습 구간 라벨**로만 잰다(누수 방지).

### 모듈은 이미 자가 시험을 통과했다 (`src/placement.py`)

| 시험 | 기대 | 결과 |
|---|---|---|
| 좌우 대칭 분포 + NMAE만 | 중앙값 | **차이 0.00%** |
| 위로 몰린 분포 + FICR만 | 중앙값보다 위 | **+4.13%용량** |
| 분포가 아주 좁을 때 | 중앙값에 붙음 | **차이 0.000%** |

두 번째가 핵심이다 — **전역 τ로는 만들 수 없는 행동**이다.

**비용**: 학습 0회. 격자탐색 1~3분.

In [13]:
import src.placement as pp
importlib.reload(pp)

QTAUS = np.array([0.10, 0.25, 0.50, 0.75, 0.90])
QNAME = ["Q10", "Q25", "L0_lgb", "Q75", "Q90"]
FIT_FOLD = "B안 fold1"                 # 손잡이를 맞추는 fold (시간상 가장 앞)


def place(fold, g, spread, shift):
    """그 fold·그룹의 검증 구간에 대해, 행마다 최적 위치를 계산한다."""
    info = ctx.fold_info[fold]
    cap = CAPACITY_KWH[g]
    q = np.column_stack([PRED[(fold, v, g)].to_numpy() for v in QNAME])
    s = pp.cdf_samples(q, QTAUS, spread=spread, shift=shift, cap=cap)
    y_tr = ctx.train.loc[info["train_mask"], g].to_numpy(dtype=float)   # 학습구간만
    w_nmae, w_ficr = pp.objective_weights(y_tr, cap)
    return pd.Series(pp.best_placement(s, cap, w_nmae, w_ficr), index=info["valid_idx"])


def fold_score(fold, preds):
    info = ctx.fold_info[fold]
    pg = ms.per_group_scores(info["actual_df"],
                             pd.DataFrame(preds, index=info["valid_idx"]))
    return ms.combine(pg)[0]


# ── 1) fold1에서 폭·위치 손잡이를 맞춘다 ────────────────────────────────────
SPREADS = [0.8, 1.0, 1.3, 1.6, 2.0]
SHIFTS = [-0.02, -0.01, 0.0, 0.01, 0.02]
base1 = fold_score(FIT_FOLD, {g: PRED[(FIT_FOLD, "L0_lgb", g)] for g in TARGET_COLS})
print(f"[{FIT_FOLD}] 기준선 {base1:.4f}   — 이제 손잡이 {len(SPREADS)}×{len(SHIFTS)}칸 탐색")

grid = []
for sp in SPREADS:
    for sh in SHIFTS:
        sc = fold_score(FIT_FOLD, {g: place(FIT_FOLD, g, sp, sh) for g in TARGET_COLS})
        grid.append(dict(spread=sp, shift=sh, score=sc))
gdf = pd.DataFrame(grid)
print("\n■ fold1 점수 (행=폭 배율, 열=위치 이동)")
display(gdf.pivot(index="spread", columns="shift", values="score").round(4))

best = gdf.loc[gdf["score"].idxmax()]
SP, SH = float(best["spread"]), float(best["shift"])
print(f"\n선택: 폭 배율 {SP}, 위치 이동 {SH:+.2f}  (fold1 {base1:.4f} → {best['score']:.4f})")

# ── 2) 그 손잡이를 나머지 fold에 그대로 적용 ────────────────────────────────
for fold in ctx.fold_info:
    for g in TARGET_COLS:
        PRED[(fold, "L5_place", g)] = place(fold, g, SP, SH)
bench.add_from_cache(ctx, PRED, "L5_place")

# ── 3) 판정 — fold1은 손잡이를 맞춘 곳이라 빼고 본다 ────────────────────────
b = bench.fold_scores("L0_lgb")
c = bench.fold_scores("L5_place")
diff = c - b
print("\n" + "=" * 64)
print("fold별 변화       (fold1은 손잡이를 맞춘 곳이라 낙관적 → 판정에서 제외)")
print("=" * 64)
for f in [ms.A_FOLD, *ms.B_FOLDS]:
    tag = "  ← 튜닝에 사용(참고용)" if f == FIT_FOLD else ""
    print(f"  {f:12s} {b[f]:.4f} → {c[f]:.4f}   ({diff[f]:+.4f}){tag}")

honest = [f for f in [ms.A_FOLD, *ms.B_FOLDS] if f != FIT_FOLD]
d_h = diff[honest]
print("-" * 64)
print(f"  손잡이를 안 맞춘 {len(honest)}개 평균 : {d_h.mean():+.4f}")
print(f"  전부 같은 방향인가        : {'예' if (np.sign(d_h) == np.sign(d_h.iloc[0])).all() else '아니오'}")
print(f"  채택 문턱                 : ±{ms.MIN_EFFECT:.4f}")
print()
if (np.sign(d_h) == np.sign(d_h.iloc[0])).all() and d_h.mean() >= ms.MIN_EFFECT:
    print("  ⇒ 채택. 오차를 하나도 안 줄이고 배치만 바꿔서 점수를 올렸다.")
elif (np.sign(d_h) == np.sign(d_h.iloc[0])).all() and d_h.mean() <= -ms.MIN_EFFECT:
    print("  ⇒ 악화. 배치 규칙이 해롭다. 원인(분포 추정 품질)을 봐야 한다.")
else:
    print("  ⇒ 동률. 구별할 수 없으므로 채택하지 않는다(더 단순한 τ=0.50을 유지).")

print()
display(bench.table())

[B안 fold1] 기준선 0.6177   — 이제 손잡이 5×5칸 탐색

■ fold1 점수 (행=폭 배율, 열=위치 이동)


shift,-0.02,-0.01,0.00,0.01,0.02
spread,,,,,
0.8,0.6226,0.6269,0.6280,0.6293,0.6286
1.0,0.6223,0.6235,0.6252,0.6238,0.6236
1.3,0.6179,0.6177,0.6182,0.6163,0.6138
1.6,0.6114,0.6113,0.6097,0.6063,0.6035
2.0,0.6015,0.5977,0.5936,0.5908,0.5858



선택: 폭 배율 0.8, 위치 이동 +0.01  (fold1 0.6177 → 0.6293)

fold별 변화       (fold1은 손잡이를 맞춘 곳이라 낙관적 → 판정에서 제외)
  A안(2024)     0.6418 → 0.6381   (-0.0037)
  B안 fold1     0.6177 → 0.6293   (+0.0116)  ← 튜닝에 사용(참고용)
  B안 fold2     0.6428 → 0.6388   (-0.0040)
  B안 fold3     0.6462 → 0.6492   (+0.0031)
----------------------------------------------------------------
  손잡이를 안 맞춘 3개 평균 : -0.0015
  전부 같은 방향인가        : 아니오
  채택 문턱                 : ±0.0020

  ⇒ 동률. 구별할 수 없으므로 채택하지 않는다(더 단순한 τ=0.50을 유지).



,A안(2024),B안 fold1,B안 fold2,B안 fold3,B평균,B σ,B평균(정제),시드 σ
hour×month,0.4334,0.4188,0.4486,0.4183,0.4285,0.0174,0.4272,NaN
L0_lgb,0.6418,0.6177,0.6428,0.6462,0.6356,0.0156,0.6429,NaN
L2_oracle,0.8759,0.8972,0.8976,0.8541,0.8830,0.0250,0.8993,NaN
L3_full,0.8770,0.8981,0.8993,0.8550,0.8841,0.0252,0.9001,NaN
L3_match23,0.8836,0.8993,0.9094,0.8542,0.8876,0.0294,0.9035,NaN
L5_place,0.6381,0.6293,0.6388,0.6492,0.6391,0.0100,0.6458,NaN


---

# 6층 — 오차를 줄이는 길이 다 막혔다. 남은 건 '기본 튜닝'이다

## 5-2 결과 — 배치 카드도 닫힌다

| fold | 기준선 → 배치최적화 | 변화 |
|---|---|---|
| A안(2024) | 0.6418 → 0.6381 | **-0.0037** |
| B fold2 | 0.6428 → 0.6388 | **-0.0040** |
| B fold3 | 0.6462 → 0.6492 | +0.0031 |
| *B fold1 (손잡이를 맞춘 곳)* | *0.6177 → 0.6293* | *+0.0116* |

손잡이를 안 맞춘 3개 평균 **-0.0016**, 부호도 엇갈린다. **채택하지 않는다.**

### 이 실험 설계가 값을 했다

fold1만 보면 **+0.0116**이다. 손잡이를 맞춘 fold에서 채점했으면
"배치 최적화로 +0.012 얻었다"고 발표했을 것이다. **실제로는 -0.0016이다.**
과거 fold에서 맞추고 미래 fold에서 채점하는 설계가 아니었다면 그대로 속았다.

### 왜 안 됐나 — 격자가 답을 말해준다

```
shift    -0.02   -0.01    0.00    0.01    0.02
0.8     0.6226  0.6269  0.6280  0.6293  0.6286   ← 최고
1.0     0.6223  0.6235  0.6252  0.6238  0.6236
1.3     0.6179  0.6177  0.6182  0.6163  0.6138
1.6     0.6114  0.6113  0.6097  0.6063  0.6035
2.0     0.6015  0.5977  0.5936  0.5908  0.5858
```

**폭을 좁힐수록 좋아진다.** 그리고 최고값 0.8은 격자의 **맨 끝**이다.
폭이 0에 가까워지면 분포가 한 점으로 붕괴하고, 그러면 배치 계산은
그냥 **중앙값을 고르는 것**이 된다 — 즉 기준선과 같아진다.

**⇒ "분포를 쓸수록 나빠진다. 안 쓰는 게 최선이다"** 는 뜻이다.

### 5-1 검산과 반대인데, 그게 핵심이다

검산에서는 분포가 **실제보다 좁다**고 나왔다(폭을 넓혀야 한다).
그런데 실제로 넓히면 점수가 떨어진다. 모순이 아니라 이런 뜻이다.

> **폭은 맞출 수 있어도 모양은 못 맞춘다.**

분포가 한쪽으로 치우쳤다고 판단해서 예측을 옮기는 순간, 그 '치우침'이 진짜가 아니라
분위수 5개를 이어붙인 데서 생긴 잡음이면 손해만 본다. 폭을 넓힐수록 더 크게 옮기니까
더 크게 손해를 본다. 격자가 정확히 그 모양이다.

원래 이 카드의 이론적 이득은 **분포가 심하게 치우친 행**에서만 나온다.
밴드(±6% 용량)가 오차 폭의 0.36배밖에 안 되는 상황에서, 좌우로 비슷하게 퍼진 분포라면
최적 위치는 결국 중앙값이다. **치우침을 믿을 만큼 정확히 재지 못한다**는 게 결론이다.

## 지금까지 닫힌 카드 전부

| 카드 | 왜 닫혔나 |
|---|---|
| 고장·정비 보정 | 오차의 2~6%뿐 (2-3) |
| group_3 따라잡기 | 조건 맞춰도 1.87배 — UNISON 특성 (3-2) |
| 데이터 더 모으기 | 절반으로 줄여도 오히려 좋아짐 (3-2) |
| 2022년 빼기 | 그룹별 부호 반대 (3-2) |
| 바람 조건별 보정 9축 | 전부 -1% 미달 (05) |
| 바람 공간정보 3축 | 전부 -1% 미달 (4층) |
| **행별 밴드 배치** | **미래 fold에서 -0.0016 (5-2)** |
| (이전) 그룹별 파라미터·단조제약·전역 τ·지형 speedup·안정도·후류·사후보정 | 04·05에서 기각 |

**오차를 줄이는 길도, 배치를 바꾸는 길도 다 막혔다.**

## 그런데 한 가지가 통째로 안 되어 있다 — 로드맵 8단계

CLAUDE.md 로드맵의 **8단계(하이퍼파라미터 튜닝)** 를 발전량 모델에 대해 **한 번도 안 했다.**
`learning_rate` · `num_leaves` · `min_child_samples` · 정규화가 전부 **기본값**이다.
그리고 **산식손실 MLP도 `T_SOFT` 하나만 건드렸고** 나머지는 이전 프로젝트가
179개 피처로 정한 값을 그대로 쓰고 있다(우리는 200개다).

지금까지 닫은 카드들은 전부 **"데이터에 신호가 있는가"** 를 묻는 것이었고 답은 "없다"였다.
튜닝은 **"있는 신호를 제대로 뽑아내고 있는가"** 를 묻는 다른 질문이다. 아직 안 물어봤다.

## 이 셀 — 먼저 챔피언 구성을 새 틀에서 복원한다

지금 기준선 `L0_lgb`(B평균 0.6356)는 **LightGBM 단독**이다.
리더보드 0.6416을 낸 v6는 여기에 **산식손실 MLP를 0.5 비중으로 섞은 것**이고,
HANDOFF에 기록된 그 구성의 B평균은 **0.6377**이다.

MLP를 04와 **똑같은 시드 다섯 개**(42, 7, 123, 2024, 31)로 다시 붙여
**0.6377이 재현되는지** 확인한다. 재현되면 그때부터 튜닝을 시작한다.

블렌드 비중 곡선도 같이 그린다 — 재학습 없이 섞기만 하면 되므로 공짜다.

**비용**: MLP 학습 60회(5시드 × 3그룹 × 4fold), 8~15분.

In [14]:
import src.nn as mnn          # ⚠️ torch.nn과 충돌하므로 반드시 mnn (HANDOFF 주의사항 6번)

MLP_SEEDS = (42, 7, 123, 2024, 31)      # 04 21절에서 v5/v6를 만든 그 시드
print(f"산식손실 MLP — 시드 {MLP_SEEDS}, 그룹 3 × fold 4 = {5*3*4}회 학습")

_ = pl.run_variant(ctx, PRED, "L6_mlp", pl.metric_mlp_fit_fn(seeds=MLP_SEEDS), verbose=True)
bench.add_from_cache(ctx, PRED, "L6_mlp")

# ── 블렌드 비중 곡선 (재학습 없음) ─────────────────────────────────────────
rows = []
for w in [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:
    name = f"blend{int(round(w*100)):03d}"
    for fold, info in ctx.fold_info.items():
        for g in TARGET_COLS:
            PRED[(fold, name, g)] = ((1 - w) * PRED[(fold, "L0_lgb", g)]
                                     + w * PRED[(fold, "L6_mlp", g)])
    bench.add_from_cache(ctx, PRED, name)
    s = bench.fold_scores(name)
    rows.append(dict(MLP비중=w, **{k.replace("안", ""): v for k, v in s.items()},
                     B평균=s[ms.B_FOLDS].mean()))
curve = pd.DataFrame(rows).set_index("MLP비중")
print("\n■ MLP 블렌드 비중 곡선")
display(curve.round(4))

best_w = curve["B평균"].idxmax()
print(f"\nB평균 최고: 비중 {best_w}  →  {curve.loc[best_w, 'B평균']:.4f}")
print(f"HANDOFF 기록: 비중 0.3 = 0.6377 / 0.5 = 0.6377  (리더보드 0.6413 / 0.6416)")
print(f"비중 0.0 재현 확인: {curve.loc[0.0, 'B평균']:.4f}  (L0_lgb = 0.6356)")

d = curve.loc[0.5, "B평균"] - 0.6377
print(f"\n비중 0.5에서 기록값과의 차이: {d:+.4f}  "
      + ("→ 재현 OK. 튜닝을 시작한다." if abs(d) < 0.002 else
         "→ ⚠️ 어긋난다. 원인을 찾기 전에는 다음으로 넘어가지 말 것"))

산식손실 MLP — 시드 (42, 7, 123, 2024, 31), 그룹 3 × fold 4 = 60회 학습
  [L6_mlp] A안(2024): score=0.6420  (1-NMAE=0.8783, FICR=0.4057)
  [L6_mlp] B안 fold1: score=0.6053  (1-NMAE=0.8598, FICR=0.3507)
  [L6_mlp] B안 fold2: score=0.6363  (1-NMAE=0.8768, FICR=0.3957)
  [L6_mlp] B안 fold3: score=0.6502  (1-NMAE=0.8837, FICR=0.4166)

■ MLP 블렌드 비중 곡선


,A(2024),B fold1,B fold2,B fold3,B평균
MLP비중,,,,,
0.0,0.6418,0.6177,0.6428,0.6462,0.6356
0.2,0.6423,0.6206,0.6431,0.6487,0.6374
0.3,0.6424,0.6202,0.6429,0.6499,0.6377
0.4,0.6428,0.6185,0.6429,0.6505,0.6373
0.5,0.6429,0.6180,0.6417,0.6533,0.6377
0.6,0.6432,0.6161,0.6410,0.6546,0.6372
0.7,0.6431,0.6130,0.6401,0.6533,0.6355
0.8,0.6438,0.6098,0.6397,0.6538,0.6344
1.0,0.6420,0.6053,0.6363,0.6502,0.6306



B평균 최고: 비중 0.3  →  0.6377
HANDOFF 기록: 비중 0.3 = 0.6377 / 0.5 = 0.6377  (리더보드 0.6413 / 0.6416)
비중 0.0 재현 확인: 0.6356  (L0_lgb = 0.6356)

비중 0.5에서 기록값과의 차이: -0.0000  → 재현 OK. 튜닝을 시작한다.


## 6-2. 산식손실 MLP에 'FICR 쪽으로 더 가라'고 지시하기 (λ)

### 6-1에서 재현이 완벽했다

블렌드 곡선이 HANDOFF 기록과 **소수점 넷째 자리까지 전부** 일치했다
(비중 0.0 → 0.6356, 0.3 → 0.6377, 0.5 → 0.6377, 0.8 → 0.6344).
리더보드 0.6416을 낸 구성이 새 틀에서 그대로 복원됐다. **여기서부터 튜닝을 시작한다.**

### 그리고 이번 실행이 가설 하나를 직접 확인해줬다

| 4 fold 평균 | 1-NMAE | FICR |
|---|---|---|
| LightGBM 단독 | 0.8689 | **0.4053** |
| 산식손실 MLP 단독 | **0.8747** | 0.3922 |

**대회 산식을 그대로 손실로 쓴 모델인데 FICR이 LightGBM보다 0.013 나쁘다.**
NMAE로만 기여하고 있다. v5→v7 리더보드에서 관찰됐던 그대로다
(비중을 올리면 1-NMAE는 오르다 포화, FICR은 단조 하락).

### 왜 이런 일이 생기나 — 두 항의 남은 여지가 다르다

점수는 `0.5×(1−NMAE) + 0.5×FICR`로 두 항이 대등해 보인다. 그런데

- **1−NMAE = 0.869** → 1까지 **0.131** 남았다
- **FICR = 0.414** → 1까지 **0.586** 남았다

**FICR 쪽에 남은 여지가 4.5배 크다.** 그런데 손실은 둘을 대등하게 취급하니,
학습이 쉬운 쪽(매끄러운 NMAE)으로 쏠린다. FICR 항은 계단이라 기울기가 거의 없다.

### 손잡이 λ

```
손실 = 0.5 × NMAE  −  0.5 × λ × FICR
```

λ를 키우면 **"NMAE를 조금 내주더라도 밴드 안으로 들어가라"** 는 지시가 된다.
λ=1이 대회 산식 그대로이고, 우리는 1.5 / 2 / 3을 시험한다.

⚠️ **λ≠1은 대회 산식이 아닌 다른 목적함수를 최적화하는 것**이다.
그러므로 반드시 **진짜 점수(계단 그대로)** 로 검증해서 이득이 있을 때만 쓴다.

### 이 셀의 판정 순서 — 두 단계로 본다

**1단계, 손잡이가 작동하나 (덜 흔들리는 신호)**
λ를 키웠을 때 **MLP 단독 FICR이 실제로 오르는가**. 안 오르면 손잡이가 헛도는 것이고
점수를 볼 것도 없다.

**2단계, 점수가 오르나**
LightGBM과 섞은 뒤의 B평균. 다만 **이번 스윕은 시드 1개**라 흔들림이 크다
(5시드 앙상블도 σ가 0.0024인데 1시드는 그보다 크다).
**방향만 보고, 이긴 λ는 나중에 5시드로 다시 확인한다.**
0.005 미만 차이는 여기서 읽지 않는다.

**비용**: MLP 학습 48회(4개 λ × 3그룹 × 4fold), 6~10분.

In [15]:
LAMBDAS = [1.0, 1.5, 2.0, 3.0]
SWEEP_SEEDS = (42,)                # 스윕은 1시드 — 방향만 본다
BLEND_WS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

rows = []
for lam in LAMBDAS:
    name = f"mlpL{lam:g}"
    print(f"--- λ={lam} 학습 중 (12회) ---")
    _ = pl.run_variant(ctx, PRED, name,
                       pl.metric_mlp_fit_fn(cfg=dict(ficr_weight=lam), seeds=SWEEP_SEEDS),
                       verbose=False)
    bench.add_from_cache(ctx, PRED, name)

    # 1단계 — 손잡이가 작동하나: MLP 단독 성적
    d = bench.df.query("variant == @name")
    solo_nmae = 1 - d["nmae"].mean()
    solo_ficr = d["ficr"].mean()

    # 2단계 — 섞은 뒤 점수 (섞기는 공짜)
    best_w, best_s = None, -1.0
    for w in BLEND_WS:
        bn = f"{name}_w{int(round(w*100))}"
        for fold in ctx.fold_info:
            for g in TARGET_COLS:
                PRED[(fold, bn, g)] = ((1 - w) * PRED[(fold, "L0_lgb", g)]
                                       + w * PRED[(fold, name, g)])
        bench.add_from_cache(ctx, PRED, bn)
        s = bench.fold_scores(bn)[ms.B_FOLDS].mean()
        if s > best_s:
            best_w, best_s = w, s
    rows.append(dict(λ=lam, MLP단독_1NMAE=solo_nmae, MLP단독_FICR=solo_ficr,
                     최적비중=best_w, 섞은뒤_B평균=best_s))

res = pd.DataFrame(rows).set_index("λ")
print("\n■ λ 스윕 결과   (참고: LightGBM 단독 1-NMAE 0.8689 / FICR 0.4053 / B평균 0.6356)")
display(res.round(4))

print("\n[1단계] 손잡이가 작동하나 — λ를 키우면 MLP 단독 FICR이 오르는가")
f = res["MLP단독_FICR"]
if f.iloc[-1] - f.iloc[0] > 0.005:
    print(f"   예. {f.iloc[0]:.4f} → {f.iloc[-1]:.4f} ({f.iloc[-1]-f.iloc[0]:+.4f}). 손잡이가 돈다.")
elif abs(f.iloc[-1] - f.iloc[0]) <= 0.005:
    print(f"   아니오. {f.iloc[0]:.4f} → {f.iloc[-1]:.4f} — 거의 안 움직인다. 손잡이가 헛돈다.")
else:
    print(f"   반대로 간다. {f.iloc[0]:.4f} → {f.iloc[-1]:.4f}. 가설이 틀렸다.")

print("\n[2단계] 점수가 오르나 — 1시드라 0.005 미만은 노이즈로 본다")
b = res["섞은뒤_B평균"]
print(f"   λ=1.0 기준 {b.loc[1.0]:.4f}  →  최고 λ={b.idxmax()} 에서 {b.max():.4f} "
      f"({b.max()-b.loc[1.0]:+.4f})")
if b.max() - b.loc[1.0] >= 0.005:
    print(f"   ⇒ 방향 있음. λ={b.idxmax()}를 **5시드로 다시 확인**한다.")
else:
    print("   ⇒ 차이가 노이즈 안. λ는 1.0(대회 산식 그대로)을 유지하고 다음 손잡이로 간다.")

--- λ=1.0 학습 중 (12회) ---
--- λ=1.5 학습 중 (12회) ---
--- λ=2.0 학습 중 (12회) ---
--- λ=3.0 학습 중 (12회) ---

■ λ 스윕 결과   (참고: LightGBM 단독 1-NMAE 0.8689 / FICR 0.4053 / B평균 0.6356)


,MLP단독_1NMAE,MLP단독_FICR,최적비중,섞은뒤_B평균
λ,,,,
1.0,0.8738,0.3938,0.2,0.6381
1.5,0.8738,0.3972,0.4,0.6387
2.0,0.8729,0.3908,0.3,0.6378
3.0,0.8728,0.3986,0.4,0.6385



[1단계] 손잡이가 작동하나 — λ를 키우면 MLP 단독 FICR이 오르는가
   아니오. 0.3938 → 0.3986 — 거의 안 움직인다. 손잡이가 헛돈다.

[2단계] 점수가 오르나 — 1시드라 0.005 미만은 노이즈로 본다
   λ=1.0 기준 0.6381  →  최고 λ=1.5 에서 0.6387 (+0.0007)
   ⇒ 차이가 노이즈 안. λ는 1.0(대회 산식 그대로)을 유지하고 다음 손잡이로 간다.


## 6-3. MLP가 애초에 제대로 학습된 적이 있나

### 6-2 결과 — λ는 헛돌았다

| λ | MLP 단독 FICR | 섞은 뒤 B평균 |
|---|---|---|
| 1.0 | 0.3938 | 0.6381 |
| 1.5 | 0.3972 | 0.6387 |
| 2.0 | **0.3908** | 0.6378 |
| 3.0 | 0.3986 | 0.6385 |

FICR이 오르내리기만 하고 **방향이 없다**(폭 0.008). λ를 3배로 키웠는데도 그렇다.
점수 차이도 최대 +0.0007로 노이즈 안이다. **λ는 1.0으로 두고 이 카드를 닫는다.**

### 그런데 왜 헛돌았나 — 이게 다음 실험을 지목한다

손실의 FICR 항은 계단을 부드럽게 편 것인데(`t_soft=0.006`), 기울기가
**밴드 경계 근처에서만** 생긴다. 그 항에 3을 곱해도 **기울기가 원래 거의 없는 곳에서는
3배 해봐야 여전히 거의 없다.**

더 근본적인 문제가 있다. 이 MLP는 **전체 배치 학습**이다 —
데이터를 쪼개지 않고 한 번에 넣으므로 **1에폭 = 기울기 1걸음**이다.
그런데 시험 학습에서 **61에폭**에서 최적을 찍었다.

> **파라미터 12만 개짜리 모델을 61걸음 걸려서 학습을 끝냈다는 뜻이다.**

Adam은 한 걸음에 파라미터를 대략 `lr`만큼 움직인다. `lr=0.001` × 61걸음 = **0.061**.
신경망 초기값의 크기가 보통 0.05~0.1인 걸 감안하면,
**모델이 출발점에서 거의 못 벗어났다.**

그러면 이렇게 설명이 붙는다.

- 거의 안 움직인 모델 = **아주 매끄러운 함수** → NMAE에는 유리 (실제로 LightGBM보다 높다)
- 계단 모양인 FICR을 맞추려면 **울퉁불퉁한 결정면**이 필요한데 거기까지 못 갔다
- 그래서 손실에서 FICR 비중을 아무리 키워도 **갈 수 있는 곳이 없다**

**⇒ λ가 안 듣는 게 아니라, 모델이 λ를 따라갈 만큼 학습되지 않았을 가능성이 크다.**

### 이 셀 — 걸음 수와 보폭을 2×2로 본다

|  | 보폭 그대로 (lr 0.001) | 보폭 키움 (lr 0.005) |
|---|---|---|
| **걸음 그대로** (최대 400, 인내 60) | A = 현행 | C |
| **걸음 늘림** (최대 2000, 인내 400) | B | D |

**핵심으로 볼 숫자는 '몇 에폭에서 멈췄나'** 다.

- 현행(A)이 60~70에폭에서 멈추고, B/D는 훨씬 뒤에서 멈추면서 점수도 오른다
  → **덜 학습된 게 맞다.** 학습을 늘리는 게 답
- B/D도 비슷한 곳에서 멈추고 점수도 그대로면
  → **이미 갈 데까지 간 것이다.** MLP 카드를 닫는다

*(용어: **인내(patience)** = 점수가 더 이상 안 좋아져도 이만큼은 더 기다렸다가 멈춘다.
성적이 안 오른다고 바로 야자를 그만두지 않고 며칠 더 지켜보는 것과 같다.)*

**비용**: MLP 학습 48회. 오래 도는 설정이 있어 **10~25분**.

In [16]:
TRAIN_CFGS = {
    "A 현행":        dict(lr=1e-3, max_epochs=400,  patience=60),
    "B 걸음늘림":     dict(lr=1e-3, max_epochs=2000, patience=400),
    "C 보폭키움":     dict(lr=5e-3, max_epochs=400,  patience=60),
    "D 둘다":        dict(lr=5e-3, max_epochs=2000, patience=400),
}


def mlp_stop_epochs(cfg, seeds):
    """캐시에서 '몇 에폭에서 멈췄나'를 꺼낸다 (이미 학습된 것이라 공짜)."""
    key_cfg = pl._cfg_key({**pl.MLP_CFG, **cfg})
    out = []
    for fold, info in ctx.fold_info.items():
        for g in TARGET_COLS:
            for s in seeds:
                k = ("mlp", g, info["cv_suffix"], key_cfg, s)
                if k in ctx.cache:
                    out.append(ctx.cache[k][1])
    return out


rows = []
for label, cfg in TRAIN_CFGS.items():
    name = "mlp_" + label.split()[0]
    print(f"--- {label}  lr={cfg['lr']}, 최대 {cfg['max_epochs']}에폭, 인내 {cfg['patience']} ---")
    _ = pl.run_variant(ctx, PRED, name,
                       pl.metric_mlp_fit_fn(cfg=cfg, seeds=SWEEP_SEEDS), verbose=False)
    bench.add_from_cache(ctx, PRED, name)

    eps = mlp_stop_epochs(cfg, SWEEP_SEEDS)
    d = bench.df.query("variant == @name")

    best_w, best_s = None, -1.0
    for w in BLEND_WS:
        bn = f"{name}_w{int(round(w*100))}"
        for fold in ctx.fold_info:
            for g in TARGET_COLS:
                PRED[(fold, bn, g)] = ((1 - w) * PRED[(fold, "L0_lgb", g)]
                                       + w * PRED[(fold, name, g)])
        bench.add_from_cache(ctx, PRED, bn)
        s = bench.fold_scores(bn)[ms.B_FOLDS].mean()
        if s > best_s:
            best_w, best_s = w, s

    rows.append(dict(설정=label,
                     멈춘에폭_중앙=int(np.median(eps)) if eps else -1,
                     멈춘에폭_최대=int(np.max(eps)) if eps else -1,
                     상한=cfg["max_epochs"],
                     MLP단독_1NMAE=1 - d["nmae"].mean(),
                     MLP단독_FICR=d["ficr"].mean(),
                     최적비중=best_w, 섞은뒤_B평균=best_s))

res3 = pd.DataFrame(rows).set_index("설정")
print("\n■ 학습량 2×2   (참고: LightGBM 단독 1-NMAE 0.8689 / FICR 0.4053 / B평균 0.6356)")
display(res3.round(4))

print("\n[1단계] 덜 학습된 게 맞나 — 걸음을 늘리면 더 멀리 가는가")
a_ep, b_ep = res3.loc["A 현행", "멈춘에폭_중앙"], res3.loc["B 걸음늘림", "멈춘에폭_중앙"]
print(f"   A 현행   : {a_ep}에폭에서 멈춤 (상한 {res3.loc['A 현행','상한']})")
print(f"   B 걸음늘림: {b_ep}에폭에서 멈춤 (상한 {res3.loc['B 걸음늘림','상한']})")
if b_ep > a_ep * 1.5:
    print("   → 더 멀리 갔다. 현행은 **덜 학습된 상태**였다.")
else:
    print("   → 비슷한 곳에서 멈춘다. 이미 갈 데까지 간 것이다.")

print("\n[2단계] 점수가 오르나 — 1시드라 0.005 미만은 노이즈")
b = res3["섞은뒤_B평균"]
gain = b.max() - b.loc["A 현행"]
print(f"   A 현행 {b.loc['A 현행']:.4f}  →  최고 [{b.idxmax()}] {b.max():.4f}  ({gain:+.4f})")
if gain >= 0.005:
    print(f"   ⇒ 방향 있음. [{b.idxmax()}]를 **5시드로 다시 확인**한다.")
else:
    print("   ⇒ 노이즈 안. MLP 학습량 카드도 닫고, 남은 것은 LightGBM 자체 튜닝뿐이다.")

--- A 현행  lr=0.001, 최대 400에폭, 인내 60 ---
--- B 걸음늘림  lr=0.001, 최대 2000에폭, 인내 400 ---
--- C 보폭키움  lr=0.005, 최대 400에폭, 인내 60 ---
--- D 둘다  lr=0.005, 최대 2000에폭, 인내 400 ---

■ 학습량 2×2   (참고: LightGBM 단독 1-NMAE 0.8689 / FICR 0.4053 / B평균 0.6356)


,멈춘에폭_중앙,멈춘에폭_최대,상한,MLP단독_1NMAE,MLP단독_FICR,최적비중,섞은뒤_B평균
설정,,,,,,,
A 현행,31,106,400,0.8738,0.3938,0.2,0.6381
B 걸음늘림,61,261,2000,0.8725,0.3869,0.2,0.6371
C 보폭키움,46,101,400,0.8695,0.3933,0.4,0.6383
D 둘다,33,141,2000,0.8697,0.3967,0.4,0.6383



[1단계] 덜 학습된 게 맞나 — 걸음을 늘리면 더 멀리 가는가
   A 현행   : 31에폭에서 멈춤 (상한 400)
   B 걸음늘림: 61에폭에서 멈춤 (상한 2000)
   → 더 멀리 갔다. 현행은 **덜 학습된 상태**였다.

[2단계] 점수가 오르나 — 1시드라 0.005 미만은 노이즈
   A 현행 0.6381  →  최고 [D 둘다] 0.6383  (+0.0003)
   ⇒ 노이즈 안. MLP 학습량 카드도 닫고, 남은 것은 LightGBM 자체 튜닝뿐이다.


---

# 7층 — 로드맵 8단계(튜닝), 한 번도 안 한 것

## 6-3 결과 — MLP 학습량도 닫힌다

| 설정 | 멈춘 에폭(중앙/최대) | 상한 | MLP 단독 FICR | 섞은 뒤 B평균 |
|---|---|---|---|---|
| A 현행 (lr .001, 400) | 31 / 106 | 400 | 0.3938 | **0.6381** |
| B 걸음늘림 (lr .001, 2000) | 61 / 261 | 2000 | 0.3869 | 0.6371 |
| C 보폭키움 (lr .005, 400) | 46 / 101 | 400 | 0.3933 | 0.6383 |
| D 둘다 (lr .005, 2000) | 33 / 141 | 2000 | 0.3967 | 0.6383 |

**어느 설정도 상한에 못 닿았다.** A는 400 중 최대 106, B는 2000 중 최대 261에서 멈췄다.
자리가 남았는데도 스스로 멈춘 것이니, **더 오래 돌릴 수 있는데 안 좋아져서 멈춘 것**이다.

B는 실제로 두 배 멀리 갔는데(31→61에폭) **점수는 오히려 떨어졌다**(0.6381 → 0.6371),
FICR도 떨어졌다(0.3938 → 0.3869). **더 학습하면 과적합될 뿐이다.**

점수 폭은 0.6371~0.6383으로 전부 노이즈 안이다. **MLP 카드를 닫는다.**

## 지금까지 아홉 번 연속으로 "효과 없음"이 나왔다

고장 보정 · group_3 따라잡기 · 데이터 늘리기 · 2022년 빼기 · 바람 조건축 9개 ·
바람 공간축 3개 · 밴드 배치 · MLP λ · MLP 학습량.

전부 **제대로 재서** 나온 결과다. 우연이 아니라 **이 파이프라인이 지금 데이터에서
꽤 좋은 자리에 앉아 있다**는 뜻으로 읽어야 한다.

## 그런데 딱 하나, 통째로 안 한 게 남아 있다

CLAUDE.md 로드맵의 **8단계(하이퍼파라미터 튜닝)** 다.
발전량 LightGBM의 설정이 **전부 기본값**이다.

| 항목 | 지금 값 | 무슨 뜻인가 |
|---|---|---|
| `num_leaves` | 31 | 나무 하나가 가질 수 있는 잎의 최대 개수 = **모델 복잡도** |
| `min_child_samples` | 20 | 잎 하나에 최소 몇 개 표본이 있어야 하는가 = **과적합 방지** |
| `learning_rate` | 0.1 | 나무 하나가 결과에 얼마나 크게 반영되는가 = **보폭** |
| `reg_lambda` | 0 | 잎 값이 커지는 것에 주는 벌점 = **정규화 없음** |
| `colsample_bytree` | 1.0 | 나무마다 피처를 몇 %나 보는가 = **전부 봄** |

지금까지 닫은 카드들은 전부 **"데이터에 신호가 있는가"** 를 물었고 답이 "없다"였다.
튜닝은 **"있는 신호를 제대로 뽑고 있는가"** 라는 **다른 질문**이다.

## 왜 격자탐색이 아니라 한 번에 하나씩 바꾸나

다섯 항목을 격자로 다 조합하면 수백 가지가 되고, 그중 하나는 **순전히 운으로**
좋아 보이게 마련이다(많이 시험할수록 우연히 이기는 게 나온다).
그래서 **한 항목만 바꾸고 나머지는 기본값 그대로** 두는 방식으로 본다.
05가 정한 방식이고, 무엇이 차이를 만들었는지도 분리된다.

## 판정

기준선 `L0_lgb`(A안 0.6418 / B평균 0.6356)와 **fold별로 짝지어** 비교한다.
LightGBM 기본값은 완전히 결정적이라 시드 잡음이 없으므로,
문턱은 **실효크기 하한 0.002**가 그대로 적용된다.
B안 세 fold의 방향이 모두 같고 평균 차이가 0.002를 넘어야 채택이다.

**비용**: 학습 132회. `learning_rate`를 낮춘 설정은 나무를 더 많이 쓰므로 느리다. **15~30분**.

In [17]:
SWEEPS = {
    "num_leaves":        [15, 63, 127],
    "min_child_samples": [5, 50, 100],
    "learning_rate":     [0.05, 0.03],
    "reg_lambda":        [1.0, 10.0],
    "colsample_bytree":  [0.7],
}
BASE_A = bench.fold_scores("L0_lgb")[ms.A_FOLD]
BASE_B = bench.fold_scores("L0_lgb")[ms.B_FOLDS].mean()
print(f"기준선 L0_lgb — A안 {BASE_A:.4f} / B평균 {BASE_B:.4f}  (전부 기본값)\n")

rows = [dict(항목="(기본값)", 값="-", A안=BASE_A, B평균=BASE_B, ΔB=0.0)]
for param, values in SWEEPS.items():
    for v in values:
        name = f"lgb_{param}_{v}"
        print(f"--- {param} = {v} ---")
        _ = pl.run_variant(ctx, PRED, name,
                           pl.gbdt_fold_fit_fn(params_override={param: v}), verbose=False)
        bench.add_from_cache(ctx, PRED, name)
        s = bench.fold_scores(name)
        rows.append(dict(항목=param, 값=v, A안=s[ms.A_FOLD],
                         B평균=s[ms.B_FOLDS].mean(),
                         ΔB=s[ms.B_FOLDS].mean() - BASE_B))

tune = pd.DataFrame(rows)
print("\n■ 한 번에 하나씩 바꿔본 결과  (ΔB = 기준선 대비 B평균 변화)")
display(tune.round(4).to_string(index=False))

cand = tune[tune["항목"] != "(기본값)"].sort_values("ΔB", ascending=False)
print(f"\n가장 좋아 보이는 셋 (아직 판정 아님):")
for _, r in cand.head(3).iterrows():
    print(f"   {r['항목']} = {r['값']}   ΔB {r['ΔB']:+.4f}")

top = cand.iloc[0]
print(f"\n■ 1위 후보를 판정 규칙에 넣는다 — {top['항목']} = {top['값']}")
print("=" * 64)
_ = bench.compare("L0_lgb", f"lgb_{top['항목']}_{top['값']}")

print(f"""
※ 여기서 '개선'이 나와도 그것 하나만 채택한다. 여러 개를 한꺼번에 합치면
   {len(cand)}번 시험한 것 중 운 좋은 조합을 고르는 셈이 되어 다시 과적합이다.
   두 개 이상 합치고 싶으면 **합친 상태를 하나의 후보로** 다시 돌려서 판정해야 한다.""")

기준선 L0_lgb — A안 0.6418 / B평균 0.6356  (전부 기본값)

--- num_leaves = 15 ---


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

--- num_leaves = 63 ---


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

--- num_leaves = 127 ---


...

--- colsample_bytree = 0.7 ---


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X


■ 한 번에 하나씩 바꿔본 결과  (ΔB = 기준선 대비 B평균 변화)


'               항목     값     A안    B평균      ΔB\n            (기본값)     - 0.6418 0.6356  0.0000\n       num_leaves    15 0.6403 0.6351 -0.0005\n       num_leaves    63 0.6404 0.6331 -0.0024\n       num_leaves   127 0.6373 0.6311 -0.0044\nmin_child_samples     5 0.6402 0.6350 -0.0006\nmin_child_samples    50 0.6408 0.6346 -0.0009\nmin_child_samples   100 0.6427 0.6351 -0.0004\n    learning_rate  0.05 0.6410 0.6351 -0.0005\n    learning_rate  0.03 0.6421 0.6348 -0.0008\n       reg_lambda   1.0 0.6403 0.6334 -0.0021\n       reg_lambda  10.0 0.6403 0.6347 -0.0008\n colsample_bytree   0.7 0.6395 0.6347 -0.0009'


가장 좋아 보이는 셋 (아직 판정 아님):
   min_child_samples = 100   ΔB -0.0004
   num_leaves = 15   ΔB -0.0005
   learning_rate = 0.05   ΔB -0.0005

■ 1위 후보를 판정 규칙에 넣는다 — min_child_samples = 100
[lgb_min_child_samples_100] vs [L0_lgb]  →  **동률**
  fold별 Δ : A(2024)=+0.0009  B fold1=-0.0016  B fold2=+0.0008  B fold3=-0.0003
  B평균 Δ  : -0.0004   문턱 ±0.0020 (실효크기 하한)   부호일치=X
             노이즈 내역 — 짝지은 SE 0.0007 / 시드 0.0000 / 하한 0.0020
  정제 Δ   : -0.0015  (방향 일치)
  ⇒ 구별 불가. 채택하지 않는다(더 단순한 쪽을 유지).

※ 여기서 '개선'이 나와도 그것 하나만 채택한다. 여러 개를 한꺼번에 합치면
   11번 시험한 것 중 운 좋은 조합을 고르는 셈이 되어 다시 과적합이다.
   두 개 이상 합치고 싶으면 **합친 상태를 하나의 후보로** 다시 돌려서 판정해야 한다.


---

# 8층 — σ를 줄이는 마지막 방법: 서로 다른 알고리즘을 섞는다

## 7층 결과 — 튜닝도 닫힌다

11개 설정 **전부 기본값보다 나빴다**(ΔB -0.0004 ~ -0.0044).
가장 나은 것도 -0.0004이고 fold 부호가 엇갈려 **동률**이다. 로드맵 8단계를 닫는다.

## 지금 문제가 정확히 어디인가

1등과의 격차 **0.0321**을 쪼개면

| | 격차 기여 | 비중 |
|---|---|---|
| NMAE 쪽 | +0.0055 | 17% |
| **FICR 쪽** | **+0.0266** | **83%** |

그룹별로는

| | FICR | NMAE로 잃은 점수 | FICR로 잃은 점수 | 배수 |
|---|---|---|---|---|
| group_1 | 0.4225 | 0.0208 | **0.0962** | 4.6배 |
| group_2 | 0.4557 | 0.0209 | **0.0907** | 4.3배 |
| group_3 | 0.3377 | 0.0239 | **0.1104** | 4.6배 |

**세 그룹 다 FICR로 잃는 게 4배 이상 크다.** 특정 그룹의 문제가 아니다.

그리고 FICR이 낮은 이유는 하나다.

```
4원을 받는 구간 : 실제값 ±6% 용량  (반폭 0.060)
우리 오차 폭 σ  :             0.172
                밴드 ÷ σ = 0.35   → 통과율 33%
```

**밴드가 오차 폭의 3분의 1이다.** 예측을 어디에 놓든 3분의 2는 밖으로 나간다.
1등 수준(FICR 0.4677)이 되려면 **σ를 13% 줄여야** 한다.

## 그런데 안 해본 σ 감소법이 하나 있다

지금 앙상블은 **LightGBM 시드 5개 + MLP**다. 시드만 다를 뿐 **같은 알고리즘**이라
틀리는 방향이 거의 같다. XGBoost와 CatBoost는 04에서 **비교만 하고 섞은 적이 없다** —
"누가 제일 나은가"를 물었지 "같이 쓰면 어떤가"를 안 물었다.

서로 다른 구현은 나무를 기르는 방식이 다르다.

- **LightGBM**: 가장 이득이 큰 잎 하나를 골라 키운다(leaf-wise)
- **XGBoost**: 깊이를 한 층씩 고르게 키운다(level-wise)
- **CatBoost**: 모든 잎이 같은 조건으로 갈라지는 대칭 나무를 쓴다

**같은 데이터를 봐도 다른 곳에서 틀린다.** 평균내면 σ가 줄어든다.
**새 정보가 전혀 필요 없는 유일한 σ 감소법**이다.

## 얼마나 줄어들지 미리 계산할 수 있다

오차끼리의 상관을 ρ라 하면, 세 모델을 평균한 오차 폭은

```
σ_평균  =  σ × √((1 + 2ρ) / 3)
```

| ρ (오차 상관) | σ 감소 | FICR 기대 변화 | 총점 기대 변화 |
|---|---|---|---|
| 0.99 | -0.3% | +0.001 | +0.000 |
| 0.95 | -1.7% | +0.006 | +0.001 |
| 0.90 | -3.4% | +0.012 | +0.002 |
| 0.80 | -6.9% | +0.025 | +0.004 |

**그래서 이 셀은 먼저 오차 상관부터 잰다.** ρ가 0.97보다 크면 얻을 게 거의 없고,
0.9 근처면 채택 문턱(0.002)을 넘길 수 있다.

⚠️ **공정성**: 세 패밀리 모두 **같은 라벨(`B_norm`) · 같은 표본가중 · 같은 τ=0.50 ·
같은 피처 200개 · 같은 조기종료**로 학습한다. 바뀌는 것은 **알고리즘 하나뿐**이다.
(`τ=0.50`이면 분위수 손실이 곧 MAE라 세 패밀리의 목적함수가 수학적으로 같아진다.)

**비용**: 학습 24회. CatBoost가 느려서 **10~25분**.

In [18]:
FAMILIES = {"xgboost": "L8_xgb", "catboost": "L8_cat"}

for fam, name in FAMILIES.items():
    if (list(ctx.fold_info)[0], name, TARGET_COLS[0]) in PRED:
        print(f"{name} 이미 있음 — 건너뜀"); continue
    print(f"--- {fam} 학습 중 (12회) ---")
    _ = pl.run_variant(ctx, PRED, name, pl.gbdt_fold_fit_fn(family=fam), verbose=False)
    bench.add_from_cache(ctx, PRED, name)

print("\n■ 세 패밀리 단독 성적")
display(bench.table().loc[["L0_lgb", "L8_xgb", "L8_cat"]])

# ── 1단계: 오차가 얼마나 다른 방향으로 틀리는가 ─────────────────────────────
MEMBERS = ["L0_lgb", "L8_xgb", "L8_cat"]
errs = {m: [] for m in MEMBERS}
for fold, info in ctx.fold_info.items():
    for g in TARGET_COLS:
        a = info["actual_df"][g].to_numpy(dtype=float)
        ok = np.isfinite(a) & (a >= CAPACITY_KWH[g] * 0.10)      # 채점 대상 행만
        for m in MEMBERS:
            errs[m].append((PRED[(fold, m, g)].to_numpy()[ok] - a[ok]) / CAPACITY_KWH[g])
E = pd.DataFrame({m: np.concatenate(v) for m, v in errs.items()})

print(f"\n■ 오차끼리의 상관  (채점 대상 {len(E):,}행)")
display(E.corr().round(4))
rho = E.corr().to_numpy()[np.triu_indices(3, 1)].mean()
pred_cut = (np.sqrt((1 + 2 * rho) / 3) - 1) * 100
print(f"\n평균 상관 ρ = {rho:.4f}")
print(f"이론상 σ 감소 예측 = √((1+2ρ)/3) − 1 = {pred_cut:+.2f}%")
print(f"실측 σ : " + "  ".join(f"{m}={E[m].std():.4f}" for m in MEMBERS)
      + f"  |  셋 평균 = {E.mean(axis=1).std():.4f} ({(E.mean(axis=1).std()/E[MEMBERS].std().mean()-1)*100:+.2f}%)")

# ── 2단계: 실제로 섞어서 점수를 본다 ────────────────────────────────────────
for fold in ctx.fold_info:
    for g in TARGET_COLS:
        PRED[(fold, "L8_mix3", g)] = sum(PRED[(fold, m, g)] for m in MEMBERS) / 3.0
bench.add_from_cache(ctx, PRED, "L8_mix3")

print("\n" + "=" * 64)
print("■ 판정 ① GBDT 3종 평균  vs  LightGBM 단독")
print("=" * 64)
_ = bench.compare("L0_lgb", "L8_mix3")

# ── 3단계: MLP까지 얹어 최종 구성으로 비교 ─────────────────────────────────
print("\n■ 여기에 산식손실 MLP를 섞은 최종 구성 (비중별)")
rows = []
for w in [0.0, 0.2, 0.3, 0.4, 0.5, 0.6]:
    for base, tag in [("L0_lgb", "지금(LGB만)"), ("L8_mix3", "새것(3종평균)")]:
        bn = f"fin_{tag[:2]}_{int(round(w*100))}"
        for fold in ctx.fold_info:
            for g in TARGET_COLS:
                PRED[(fold, bn, g)] = ((1 - w) * PRED[(fold, base, g)]
                                       + w * PRED[(fold, "L6_mlp", g)])
        bench.add_from_cache(ctx, PRED, bn)
        s = bench.fold_scores(bn)
        rows.append(dict(MLP비중=w, 구성=tag, A안=s[ms.A_FOLD], B평균=s[ms.B_FOLDS].mean()))
fin = pd.DataFrame(rows).pivot(index="MLP비중", columns="구성", values="B평균")
display(fin.round(4))

best_w = fin["새것(3종평균)"].idxmax()
print(f"\n새 구성 최고: MLP 비중 {best_w} → B평균 {fin.loc[best_w,'새것(3종평균)']:.4f}")
print(f"지금 구성 최고: {fin['지금(LGB만)'].max():.4f} (비중 {fin['지금(LGB만)'].idxmax()})")
print("\n" + "=" * 64)
print(f"■ 판정 ② 최종 구성끼리 (MLP 비중 0.5 고정, 리더보드 v6와 같은 조건)")
print("=" * 64)
_ = bench.compare("fin_지금_50", "fin_새것_50")

--- xgboost 학습 중 (12회) ---
--- catboost 학습 중 (12회) ---

■ 세 패밀리 단독 성적


,A안(2024),B안 fold1,B안 fold2,B안 fold3,B평균,B σ,B평균(정제),시드 σ
L0_lgb,0.6418,0.6177,0.6428,0.6462,0.6356,0.0156,0.6429,NaN
L8_xgb,0.6375,0.6124,0.6353,0.6450,0.6309,0.0168,0.6372,NaN
L8_cat,0.6311,0.6099,0.6331,0.6307,0.6246,0.0128,0.6302,NaN



■ 오차끼리의 상관  (채점 대상 36,492행)


,L0_lgb,L8_xgb,L8_cat
L0_lgb,1.0000,0.9819,0.9800
L8_xgb,0.9819,1.0000,0.9675
L8_cat,0.9800,0.9675,1.0000



평균 상관 ρ = 0.9765
이론상 σ 감소 예측 = √((1+2ρ)/3) − 1 = -0.79%
실측 σ : L0_lgb=0.1714  L8_xgb=0.1731  L8_cat=0.1723  |  셋 평균 = 0.1709 (-0.79%)

■ 판정 ① GBDT 3종 평균  vs  LightGBM 단독
[L8_mix3] vs [L0_lgb]  →  **악화**
  fold별 Δ : A(2024)=-0.0037  B fold1=-0.0018  B fold2=-0.0040  B fold3=-0.0012
  B평균 Δ  : -0.0024   문턱 ±0.0020 (실효크기 하한)   부호일치=O
             노이즈 내역 — 짝지은 SE 0.0009 / 시드 0.0000 / 하한 0.0020
  정제 Δ   : -0.0029  (방향 일치)

■ 여기에 산식손실 MLP를 섞은 최종 구성 (비중별)


구성,새것(3종평균),지금(LGB만)
MLP비중,,
0.0,0.6332,0.6356
0.2,0.6344,0.6374
0.3,0.6350,0.6377
0.4,0.6355,0.6373
0.5,0.6356,0.6377
0.6,0.6352,0.6372



새 구성 최고: MLP 비중 0.5 → B평균 0.6356
지금 구성 최고: 0.6377 (비중 0.3)

■ 판정 ② 최종 구성끼리 (MLP 비중 0.5 고정, 리더보드 v6와 같은 조건)
[fin_새것_50] vs [fin_지금_50]  →  **악화**
  fold별 Δ : A(2024)=-0.0006  B fold1=-0.0026  B fold2=-0.0014  B fold3=-0.0022
  B평균 Δ  : -0.0021   문턱 ±0.0020 (실효크기 하한)   부호일치=O
             노이즈 내역 — 짝지은 SE 0.0004 / 시드 0.0000 / 하한 0.0020
  정제 Δ   : -0.0026  (방향 일치)


---

# 9층 — 아직 한 번도 안 돌린 카드 두 개

## 8층 결과 — 모델 종류 섞기도 실패

| | 단독 B평균 |
|---|---|
| LightGBM | **0.6356** |
| XGBoost | 0.6309 |
| CatBoost | 0.6246 |

오차끼리의 상관이 **ρ = 0.9765**로 너무 높았다. 이론 예측 σ 감소 **-0.79%**,
실측도 정확히 **-0.79%**(계산은 맞았다). 그런데 약한 두 멤버가 평균을 끌어내려
**GBDT 3종 평균은 -0.0024로 악화**, MLP까지 얹은 최종 구성도 **-0.0021로 악화**였다.

**세 구현이 사실상 같은 답을 낸다.** 섞을 값어치가 없다.

---

## 그런데 안 돌린 카드가 남아 있었다

### 카드 A — 풍속 모델이 '중요한 구간'을 알게 한다

지금 풍속 모델은 `l2` 손실이라 **모든 풍속 구간을 똑같이 중요하게** 여긴다.
그런데 발전량은 풍속에 그렇게 반응하지 않는다.

| 풍속 | 0.5 m/s 틀렸을 때 발전량 오차 |
|---|---|
| 6 m/s (파워커브 급경사) | **매우 큼** |
| 16 m/s (정격 평탄부) | **거의 0** |

**즉 지금은 아무 상관 없는 구간의 오차를 줄이느라 중요한 구간을 희생하고 있다.**

해결은 풍속 모델 학습에 **파워커브 기울기 `|dP/dv|`를 표본가중으로** 주는 것이다.
"발전량이 민감한 풍속대를 더 정확히 맞혀라"는 지시다.

이건 **바람 오차(σ)를 줄이는 카드가 아니라, 같은 오차를 발전량이 아픈 곳에서
안 아픈 곳으로 옮기는 카드**다. 그래서 05의 진단(모든 조건축이 -1% 미만)에
**반박되지 않는다** — 05 보고서도 이 카드만은 "남은 카드"로 명시해뒀다.

`src/pipeline.py`에 `powercurve_slope_weight()`가 이미 구현돼 있는데
**한 번도 실행된 적이 없다.**

- `slope` : 가중 ∝ |dP/dv| (파워커브 기울기)
- `pcrev` : 가중 ∝ |dP/dv| × 그 풍속의 발전량 (FICR의 발전량 가중까지 반영)

### 카드 B — 표본가중이 산식의 절반만 흉내내고 있었다

산식을 다시 보자.

```
score = 0.5 × (1 − NMAE)  +  0.5 × FICR
                 └ 채점행을 **똑같이** 셈   └ 실제발전량으로 **가중**
```

**NMAE는 채점 대상 행을 전부 동등하게 센다.** 발전량 20,000짜리 행과 3,000짜리 행이
NMAE에는 **똑같이** 기여한다. FICR만 발전량으로 가중한다.

그런데 우리 표본가중은 `max(발전량/용량, 0.1)` — **전부 발전량 가중**이다.
즉 **NMAE가 원하는 균등 가중을 전혀 반영하지 않고 고출력 행에 과하게 쏠려 있다.**

산식에 맞춘 가중은 이렇다.

```
가중 = 0.5 × (균등)  +  0.5 × (발전량/용량)      ← 채점 대상 행만
```

| 이용률 | 지금 (`actual`) | 산식 정합 (`metric`) |
|---|---|---|
| 0.05 (채점 안 됨) | 0.10 | 0.10 |
| 0.10 | 0.10 | **0.55** |
| 0.30 | 0.30 | **0.65** |
| 0.60 | 0.60 | **0.80** |
| 1.00 | 1.00 | 1.00 |

**채점 대상인데 발전량이 중간인 행들이 지금 10배 가까이 눌려 있다.**
`eval_only`(채점행 전부 1.0, 나머지 0.1)도 같이 본다.

## 통제군을 함께 돌린다

방금 `src/pipeline.py`의 표본가중 함수에 손을 댔으므로,
**기본 설정으로 돌렸을 때 기준선과 완전히 같은 값이 나오는지** 먼저 확인한다.
안 맞으면 거기서 멈춘다.

**비용**: 학습 78회. 풍속 모델을 다시 학습하는 카드 A가 무거워 **20~30분**.
(메모리 때문에 카드 A는 변형마다 피처 프레임 캐시를 비운다.)

In [22]:
import importlib, inspect
importlib.reload(pl)          # ⚠️ src/pipeline.py를 고쳤다 — 커널이 새 버전을 읽게 한다
importlib.reload(ms)
assert "wmode" in inspect.signature(pl.gbdt_fold_fit_fn).parameters, \
    "pipeline.py 재로딩 실패 — 커널을 재시작해야 한다"
print("pipeline.py 재로딩 OK (ctx·PRED·bench는 그대로 살아 있다)")

def drop_frame_cache(wweight_tag):
    """피처 프레임 하나가 171 MB다. 변형이 끝나면 즉시 비운다 (HANDOFF 주의사항 7번)."""
    for k in [k for k in list(ctx.cache) if k[0] == "frame" and k[-1] == wweight_tag]:
        del ctx.cache[k]


# ── 통제군: pipeline.py를 고쳤으니 기본 경로가 그대로인지 먼저 확인 ──────────
print("--- 통제군 (기본 설정) — 기준선과 같아야 한다 ---")
_ = pl.run_variant(ctx, PRED, "C_ctrl", pl.gbdt_fold_fit_fn(), verbose=False)
bench.add_from_cache(ctx, PRED, "C_ctrl")
ctrl = bench.fold_scores("C_ctrl"); base = bench.fold_scores("L0_lgb")
same = np.allclose(ctrl.to_numpy(), base.to_numpy(), atol=1e-9)
print(f"   기준선 {base[ms.B_FOLDS].mean():.6f} / 통제군 {ctrl[ms.B_FOLDS].mean():.6f}"
      f"  →  {'동일 OK' if same else '⚠️ 다르다! 여기서 멈출 것'}")
assert same, "pipeline.py 수정이 기본 동작을 바꿨다"

RESULTS = []

# ── 카드 B: 표본가중 (프레임 재사용 — 값싸다) ───────────────────────────────
for wm in ["metric", "eval_only"]:
    name = f"W_{wm}"
    print(f"\n--- 카드 B: 표본가중 = {wm} (12회) ---")
    _ = pl.run_variant(ctx, PRED, name, pl.gbdt_fold_fit_fn(wmode=wm), verbose=False)
    bench.add_from_cache(ctx, PRED, name)
    RESULTS.append(("표본가중 " + wm, name))

# ── 카드 A: 풍속 모델 손실 가중 (풍속 모델을 새로 학습 — 무겁다) ────────────
for ww in ["slope", "pcrev"]:
    name = f"V_{ww}"
    print(f"\n--- 카드 A: 풍속 손실가중 = {ww} (21회, 프레임 새로 만듦) ---")
    _ = pl.run_variant(ctx, PRED, name, pl.lgbm_fold_fit_fn(wweight=ww), verbose=False)
    bench.add_from_cache(ctx, PRED, name)
    RESULTS.append(("풍속 손실가중 " + ww, name))
    drop_frame_cache(ww)

# ── 정리 ────────────────────────────────────────────────────────────────────
rows = []
for label, name in RESULTS:
    s = bench.fold_scores(name)
    rows.append(dict(카드=label, A안=s[ms.A_FOLD], B평균=s[ms.B_FOLDS].mean(),
                     ΔB=s[ms.B_FOLDS].mean() - base[ms.B_FOLDS].mean()))
tab = pd.DataFrame(rows)
print("\n\n■ 결과  (기준선 A안 %.4f / B평균 %.4f)" % (base[ms.A_FOLD], base[ms.B_FOLDS].mean()))
display(tab.round(4).to_string(index=False))

for label, name in sorted(RESULTS, key=lambda x: -bench.fold_scores(x[1])[ms.B_FOLDS].mean()):
    print("\n" + "=" * 64)
    print(f"■ {label}")
    print("=" * 64)
    _ = bench.compare("L0_lgb", name)

pipeline.py 재로딩 OK (ctx·PRED·bench는 그대로 살아 있다)
--- 통제군 (기본 설정) — 기준선과 같아야 한다 ---


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

   기준선 0.635555 / 통제군 0.635555  →  동일 OK

--- 카드 B: 표본가중 = metric (12회) ---


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X


--- 카드 B: 표본가중 = eval_only (12회) ---


...


--- 카드 A: 풍속 손실가중 = pcrev (21회, 프레임 새로 만듦) ---


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X



■ 결과  (기준선 A안 0.6418 / B평균 0.6356)


'            카드     A안    B평균      ΔB\n   표본가중 metric 0.6367 0.6347 -0.0009\n표본가중 eval_only 0.6345 0.6319 -0.0037\n 풍속 손실가중 slope 0.6310 0.6258 -0.0098\n 풍속 손실가중 pcrev 0.6321 0.6260 -0.0095'


■ 표본가중 metric
[W_metric] vs [L0_lgb]  →  **동률**
  fold별 Δ : A(2024)=-0.0051  B fold1=+0.0020  B fold2=-0.0072  B fold3=+0.0025
  B평균 Δ  : -0.0009   문턱 ±0.0063 (노이즈 0.0031 × 2.0)   부호일치=X
             노이즈 내역 — 짝지은 SE 0.0031 / 시드 0.0000 / 하한 0.0020
  정제 Δ   : -0.0016  (방향 일치)
  ⇒ 구별 불가. 채택하지 않는다(더 단순한 쪽을 유지).

■ 표본가중 eval_only
[W_eval_only] vs [L0_lgb]  →  **동률**
  fold별 Δ : A(2024)=-0.0073  B fold1=-0.0013  B fold2=-0.0087  B fold3=-0.0011
  B평균 Δ  : -0.0037   문턱 ±0.0050 (노이즈 0.0025 × 2.0)   부호일치=O
             노이즈 내역 — 짝지은 SE 0.0025 / 시드 0.0000 / 하한 0.0020
  정제 Δ   : -0.0042  (방향 일치)
  ⇒ 구별 불가. 채택하지 않는다(더 단순한 쪽을 유지).

■ 풍속 손실가중 pcrev
[V_pcrev] vs [L0_lgb]  →  **악화**
  fold별 Δ : A(2024)=-0.0097  B fold1=-0.0097  B fold2=-0.0056  B fold3=-0.0133
  B평균 Δ  : -0.0095   문턱 ±0.0044 (노이즈 0.0022 × 2.0)   부호일치=O
             노이즈 내역 — 짝지은 SE 0.0022 / 시드 0.0000 / 하한 0.0020
  정제 Δ   : -0.0106  (방향 일치)

■ 풍속 손실가중 slope
[V_slope] vs [L0_lgb]  →  **악화**
  fold별 Δ : A(2024)=-0.0108  B fold1=-0.0111  B

---

# 10층 — 내가 틀렸던 것 하나를 다시 연다

## 다시 열어야 하는 이유

우리 가동률 정의는 **"풍속 5 m/s 이상인데 출력이 정격의 1% 이하"** 다.
즉 **완전히 멈춘 터빈만** 잡는다. **절반만 내고 있는 터빈은 '정상'으로 센다.**

그래서 2-3에서 한 검사 — *"가동률 1.0인 행만 골라 채점했더니 σ가 2~6%밖에 안 줄었다"* — 는
**부분 성능저하를 하나도 걸러내지 못했다.** "정상"이라고 고른 행 안에 반쯤 죽은 터빈이
그대로 들어 있었다. **"고장 카드는 닫혔다"는 결론은 무효다.**

## 어떻게 잴 것인가 — 바람을 안 쓴다

기대출력 곡선을 풍속으로 그리는 방법은 **순환에 빠진다.**
2-2에서 실측했듯 **터빈이 덜 돌면 나셀 풍속계가 0.37~0.65 m/s 더 빠르게 읽힌다.**

```
성능저하  →  풍속계가 높게 읽음  →  기대출력 과대평가  →  가동률 과소평가  →  라벨 과대 보정
```

**보정이 필요한 곳일수록 오차가 커지는 방향**이다.

대신 **같은 시각, 같은 그룹의 터빈끼리 비교**한다. 6대가 같은 순간 거의 같은 바람을
맞으므로, 형제들보다 유독 적게 내는 터빈은 성능저하다. **풍속이 아예 필요 없다.**

```
연속가동률 = (터빈별 출력 ÷ 그 시각 형제들의 중앙값, 1로 상한)의 평균
```

⚠️ **후류 함정**: EDA 6-4에서 NW~WNW 바람일 때 wtg01/04만 풍속이 높고 나머지 4대는
낮았다. 이건 고장이 아니라 정상적인 그늘짐이다. 그래서 **풍향별로 각 터빈의 평소 비율을
따로 학습해** 그 기준 대비 얼마나 부족한지를 본다. 후류는 기준선에 녹아든다.

## 이 셀이 하는 일 (둘 다)

### ① 부분 성능저하 진단 — 학습 0회

1. 연속 가동률을 계산하고 지금 정의와 비교한다
2. **성능저하가 없는 행만 골라 채점**했을 때 σ가 얼마나 떨어지는지 잰다 (2-3과 같은 방식)
3. 크게 떨어지면 → 라벨 정제(`B_norm_v2`)를 만들 값어치가 있다
   거의 안 떨어지면 → 이번에야말로 닫는다

### ② 직접 발전량 모델 — 8층 실패에서 배운 것

GBDT 3종 앙상블은 오차 상관이 **ρ=0.977**이라 실패했다.
**필요한 건 다른 구현이 아니라 구조적으로 다른 오차다.**

지금 모델은 사실상 `예보 → 추정풍속 → 발전량` 구조다(피처 중요도 상위를 풍속 계열이 독식).
**풍속에서 파생된 피처를 전부 빼고** 원본 예보만으로 발전량을 직접 예측하면
**완전히 다른 곳에서 틀린다.**

단독 성적은 더 나쁠 것이다. 하지만 오차 상관이 낮으면 **섞어서 σ를 줄일 수 있다.**
GBDT 3종(ρ 0.977)과 비교해서 이 상관이 얼마나 낮은지가 핵심 숫자다.

**비용**: ① 학습 0회 / ② 학습 24회(피처 재선택 포함). 합쳐서 **5~10분**.

In [23]:
# ══════════════════════════════════════════════════════════════════
# ① 부분 성능저하 진단 — 형제 터빈끼리 비교 (바람을 안 쓴다)
# ══════════════════════════════════════════════════════════════════
_scada = {p: pd.read_csv(pl.REPO_ROOT / f"data/train/scada_{p}_train.csv", parse_dates=["kst_dtm"])
          for p in ("vestas", "unison")}

def turbine_hourly_power(g):
    """터빈별 시간 발전량(kWh). 01_preprocessing과 같은 시간 경계 규칙."""
    turbs = pl.TURBINES[g]
    d = _scada[turbs[0][0]]
    out = {}
    for pre, i in turbs:
        s = pd.Series(_scada[pre][f"{pre}_wtg{i:02d}_power_kw10m"].to_numpy(),
                      index=_scada[pre]["kst_dtm"])
        out[f"{pre}{i:02d}"] = s.resample("h", closed="right", label="right").sum(min_count=1)
    return pd.DataFrame(out)

# 풍향(그룹 최근접 LDAPS)을 30도로 나눠 후류를 기준선에 녹인다
wd_bin = (ctx.train.set_index("kst_dtm")[f"{TARGET_COLS[0]}_wd"] // 30).astype("Int64")

avail_cont = {}
for g in TARGET_COLS:
    P = turbine_hourly_power(g)
    cap_t = CAPACITY_KWH[g] / P.shape[1]                 # 터빈 1대의 시간 정격
    ref = P.median(axis=1)
    live = ref > 0.05 * cap_t                            # 바람이 약한 시간은 판정 불가
    # 풍향별로 '이 터빈의 평소 대비율'을 학습 → 후류를 정상으로 본다
    r = P.div(ref, axis=0).where(live)
    wb = wd_bin.reindex(r.index)
    norm = r.groupby(wb).transform("median")             # 그 풍향에서의 평소 비율
    ok_ratio = (r / norm).clip(upper=1.0)                # 평소보다 얼마나 부족한가
    avail_cont[g] = ok_ratio.mean(axis=1).where(live)
AV = pd.DataFrame({g: ctx.train["kst_dtm"].map(avail_cont[g]) for g in TARGET_COLS},
                  index=ctx.train.index)

print("■ 부분 성능저하 비율  (연속 가동률 < 0.7 인 시간)")
for g in TARGET_COLS:
    old_down = (ctx.avail[g] < 0.999).mean()
    new_down = (AV[g] < 0.7).mean()
    print(f"   {g:14s} 지금 정의(완전정지) {old_down:6.1%}   |   연속 가동률 <0.7 {new_down:6.1%}"
          f"   판정가능 {AV[g].notna().mean():5.1%}")
print("   (Codex 보고값: group_1 19.3% / group_2 2.3% / group_3 12.2%)")

# 2-3과 같은 방식: '성능저하 없는 행'만 골라 채점하면 오차 폭이 줄어드나
def sigma_on(mask_fn, variant="L0_lgb"):
    acc = []
    for fold, info in ctx.fold_info.items():
        vi = info["valid_idx"]
        pred = pd.DataFrame({g: PRED[(fold, variant, g)] for g in TARGET_COLS}, index=vi)
        acc.append(ms.per_group_scores(info["actual_df"], pred, row_mask=mask_fn(vi)))
    return pd.concat(acc).groupby(level=0).mean()[["sigma", "ficr", "n"]]

full = sigma_on(lambda vi: None)
old_clean = sigma_on(lambda vi: {g: (ctx.avail.reindex(vi)[g].fillna(1.0) >= 0.999).to_numpy()
                                 for g in TARGET_COLS})
new_clean = sigma_on(lambda vi: {g: (AV.reindex(vi)[g].fillna(1.0) >= 0.95).to_numpy()
                                 for g in TARGET_COLS})
cmp = pd.DataFrame({"전체 σ": full["sigma"], "완전정지 제외 σ": old_clean["sigma"],
                    "부분저하도 제외 σ": new_clean["sigma"]})
cmp["부분저하가 설명하는 몫 %"] = (new_clean["sigma"] / old_clean["sigma"] - 1) * 100
cmp["남은 행 수"] = new_clean["n"]
print("\n■ 성능저하 행을 빼면 오차 폭이 줄어드나")
display(cmp.round(4))
drop = cmp["부분저하가 설명하는 몫 %"].mean()
print(f"\n평균 {drop:+.1f}%  →  " + ("★ 라벨 정제(B_norm_v2)를 만들 값어치가 있다"
      if drop <= -5 else "값어치 작다. 이번에야말로 고장 카드를 닫는다"))

# ══════════════════════════════════════════════════════════════════
# ② 직접 발전량 모델 — 풍속 유래 피처를 전부 빼고 원본 예보만으로
# ══════════════════════════════════════════════════════════════════
def direct_fit_fn(ctx, g, cv_suffix, train_mask, valid_idx):
    X = pl.frame_for_fold(ctx, g, cv_suffix, train_mask)
    bad = [c for c in X.columns
           if (pl.WIND_TAG in c) or ("ws_est" in c) or ("power_curve" in c)]
    Xd = X.drop(columns=bad)
    key = ("keep_direct", g, cv_suffix)
    if key not in ctx.cache:
        m0 = pl.lgbm_train_label(ctx, Xd, g, train_mask, alpha=0.60, seed=pl.SEED, mode="A_asis")
        ctx.cache[key] = pl.select_features(m0, Xd.columns, pl.BEST_TOPN)
    keep = ctx.cache[key]
    m = pl.lgbm_train_label(ctx, Xd[keep], g, train_mask,
                            pl.DEFAULT_TAU, pl.SEED, pl.DEFAULT_LABEL_MODE)
    p = pd.Series(m.predict(Xd.loc[valid_idx, keep]), index=valid_idx)
    return p.clip(lower=0, upper=CAPACITY_KWH[g])

print("\n\n--- 직접 발전량 모델 학습 (풍속 유래 피처 제외, 24회) ---")
_ = pl.run_variant(ctx, PRED, "L10_direct", direct_fit_fn, verbose=False)
bench.add_from_cache(ctx, PRED, "L10_direct")
display(bench.table().loc[["L0_lgb", "L10_direct"]])

# 오차가 얼마나 다른 방향으로 틀리는가 — 8층의 ρ=0.977과 비교
pair = {}
for m in ["L0_lgb", "L8_xgb", "L10_direct", "L6_mlp"]:
    e = []
    for fold, info in ctx.fold_info.items():
        for g in TARGET_COLS:
            a = info["actual_df"][g].to_numpy(dtype=float)
            ok = np.isfinite(a) & (a >= CAPACITY_KWH[g] * 0.10)
            e.append((PRED[(fold, m, g)].to_numpy()[ok] - a[ok]) / CAPACITY_KWH[g])
    pair[m] = np.concatenate(e)
C = pd.DataFrame(pair).corr()
print("\n■ 오차 상관  (8층에서 GBDT끼리는 0.977이었다)")
display(C.round(4))
print(f"\n   LightGBM ↔ XGBoost   : {C.loc['L0_lgb','L8_xgb']:.4f}  (8층, 실패)")
print(f"   LightGBM ↔ 직접모델   : {C.loc['L0_lgb','L10_direct']:.4f}")
print(f"   LightGBM ↔ 산식MLP    : {C.loc['L0_lgb','L6_mlp']:.4f}")

print("\n■ 직접모델을 섞으면")
rows = []
for w in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    bn = f"dmix{int(round(w*100))}"
    for fold in ctx.fold_info:
        for g in TARGET_COLS:
            PRED[(fold, bn, g)] = ((1 - w) * PRED[(fold, "L0_lgb", g)]
                                   + w * PRED[(fold, "L10_direct", g)])
    bench.add_from_cache(ctx, PRED, bn)
    s = bench.fold_scores(bn)
    rows.append(dict(직접모델비중=w, A안=s[ms.A_FOLD], B평균=s[ms.B_FOLDS].mean()))
dm = pd.DataFrame(rows).set_index("직접모델비중")
display(dm.round(4))
bw = dm["B평균"].idxmax()
print(f"\n최고: 비중 {bw} → {dm.loc[bw,'B평균']:.4f}  (기준선 {dm.loc[0.0,'B평균']:.4f})")
if bw > 0:
    print("=" * 64)
    _ = bench.compare("L0_lgb", f"dmix{int(round(bw*100))}")

■ 부분 성능저하 비율  (연속 가동률 < 0.7 인 시간)
   kpx_group_1    지금 정의(완전정지)  16.6%   |   연속 가동률 <0.7   4.1%   판정가능 69.2%
   kpx_group_2    지금 정의(완전정지)  15.7%   |   연속 가동률 <0.7   3.6%   판정가능 68.1%
   kpx_group_3    지금 정의(완전정지)  15.2%   |   연속 가동률 <0.7   2.9%   판정가능 40.0%
   (Codex 보고값: group_1 19.3% / group_2 2.3% / group_3 12.2%)

■ 성능저하 행을 빼면 오차 폭이 줄어드나


,전체 σ,완전정지 제외 σ,부분저하도 제외 σ,부분저하가 설명하는 몫 %,남은 행 수
group,,,,,
kpx_group_1,0.1644,0.1598,0.1529,-4.3195,708.5
kpx_group_2,0.1662,0.1559,0.1619,3.8226,617.0
kpx_group_3,0.1845,0.1805,0.1873,3.7313,824.5



평균 +1.1%  →  값어치 작다. 이번에야말로 고장 카드를 닫는다


--- 직접 발전량 모델 학습 (풍속 유래 피처 제외, 24회) ---


d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\공모전\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

,A안(2024),B안 fold1,B안 fold2,B안 fold3,B평균,B σ,B평균(정제),시드 σ
L0_lgb,0.6418,0.6177,0.6428,0.6462,0.6356,0.0156,0.6429,NaN
L10_direct,0.6309,0.6062,0.6211,0.6401,0.6225,0.0170,0.6299,NaN



■ 오차 상관  (8층에서 GBDT끼리는 0.977이었다)


,L0_lgb,L8_xgb,L10_direct,L6_mlp
L0_lgb,1.0000,0.9819,0.8848,0.9407
L8_xgb,0.9819,1.0000,0.8744,0.9304
L10_direct,0.8848,0.8744,1.0000,0.8485
L6_mlp,0.9407,0.9304,0.8485,1.0000



   LightGBM ↔ XGBoost   : 0.9819  (8층, 실패)
   LightGBM ↔ 직접모델   : 0.8848
   LightGBM ↔ 산식MLP    : 0.9407

■ 직접모델을 섞으면


,A안,B평균
직접모델비중,,
0.0,0.6418,0.6356
0.1,0.6431,0.6369
0.2,0.6438,0.6372
0.3,0.6447,0.6368
0.4,0.6440,0.6366
0.5,0.6426,0.6344



최고: 비중 0.2 → 0.6372  (기준선 0.6356)
[dmix20] vs [L0_lgb]  →  **동률**
  fold별 Δ : A(2024)=+0.0020  B fold1=+0.0034  B fold2=+0.0006  B fold3=+0.0010
  B평균 Δ  : +0.0017   문턱 ±0.0020 (실효크기 하한)   부호일치=O
             노이즈 내역 — 짝지은 SE 0.0009 / 시드 0.0000 / 하한 0.0020
  정제 Δ   : +0.0020  (방향 일치)
  ⇒ 구별 불가. 채택하지 않는다(더 단순한 쪽을 유지).


---

# 11층 — 드디어 걸렸다: 서로 다르게 틀리는 모델

## 9층·10층 정리

| 카드 | 결과 | 판정 |
|---|---|---|
| 표본가중 `metric`(산식 정합) | ΔB -0.0009, 부호 엇갈림 | 동률 |
| 표본가중 `eval_only` | ΔB -0.0037 | 동률 |
| 풍속 손실가중 `slope` | ΔB **-0.0098**, 4 fold 전부 하락 | **악화** |
| 풍속 손실가중 `pcrev` | ΔB **-0.0095**, 4 fold 전부 하락 | **악화** |
| 부분 성능저하 라벨 정제 | σ 평균 **+1.1%** (오히려 나빠짐) | **닫힘** |
| **직접 발전량 모델 블렌드** | ΔB **+0.0017**, **4 fold 전부 +** | **문턱 코앞** |

### 풍속 손실가중이 왜 크게 나빠졌나 (도메인 해석)

"파워커브가 평평한 강풍대는 틀려도 되니 가중을 낮추자"가 논리였다. **그런데 틀렸다.**

평평한 건 **그 지점에서**의 얘기다. 14 m/s를 11 m/s로 잘못 맞히면 **평탄부를 벗어나
급경사 구간으로 떨어진다.** 그리고 강풍 시간대는 발전량이 크므로 **FICR에서 가중이 가장 큰 행**이다.
**가장 비싼 행을 일부러 대충 맞히게 만든 셈**이다.

### 부분 성능저하 — Codex 지적은 옳았지만 실체가 없었다

"완전 정지만 잡고 부분 성능저하는 못 잡는다"는 지적은 정확했다. 그래서 형제 터빈 비교로
다시 쟀다(바람을 안 쓰므로 순환 없음, 풍향별 기준선으로 후류도 정상 처리).

| | 연속 가동률 < 0.7 | Codex 보고값 |
|---|---|---|
| group_1 | **4.1%** | 19.3% |
| group_2 | **3.6%** | 2.3% |
| group_3 | **2.9%** | 12.2% |

우리 측정치가 훨씬 작다. Codex는 **풍속 기반 기대출력**으로 쟀는데, 2-2에서 확인했듯
성능저하 시 나셀 풍속이 높게 읽혀 **기대출력이 부풀고 성능저하가 과대 집계**된다.

그리고 결정적으로, **성능저하 행을 빼도 오차 폭이 안 줄었다**(평균 +1.1%, 3그룹 중 2그룹은 오히려 증가).
**라벨 정제 카드를 닫는다.**

---

## ⭐ 그런데 직접 발전량 모델의 오차 상관이 확 낮았다

| 짝 | 오차 상관 |
|---|---|
| LightGBM ↔ XGBoost | **0.9819** ← 8층이 실패한 이유 |
| LightGBM ↔ 산식MLP | 0.9407 |
| LightGBM ↔ **직접모델** | **0.8848** |
| 산식MLP ↔ **직접모델** | **0.8485** ← 가장 낮다 |

**구현을 바꾸는 건 소용없고 구조를 바꿔야 한다**는 게 숫자로 확인됐다.

그리고 둘만 섞었는데 이미 **4 fold 전부 플러스**(+0.0020 / +0.0034 / +0.0006 / +0.0010),
B평균 **+0.0017**로 문턱(0.0020) 코앞까지 왔다.

## 이 셀 — 셋을 한꺼번에 섞는다 (학습 0회)

상관이 낮은 세 모델을 **동시에** 섞으면 이론상 이만큼 줄어든다.

```
σ_평균 = σ × √((1 + 2ρ̄) / 3),   ρ̄ = (0.8848 + 0.9407 + 0.8485)/3 = 0.891
       = σ × 0.963   →  -3.7%
```

FICR ≈ σ^-0.86 로 환산하면 **FICR +0.013, 총점 +0.007** — 채택 문턱의 3배다.

### 비중은 어떻게 정하나 — 누수를 막는다

비중을 검증 구간에서 맞추면 그 구간 점수는 당연히 좋아진다(5층에서 겪었다: fold1 +0.0116 →
미래 fold -0.0016). 그래서 **fold1(2023 하반기)에서만 비중을 맞추고
fold2 · fold3 · A안(전부 2024년)에서 채점**한다.

같이 보는 것:
- **똑같이 나눈 비중**(1/3씩) — 아무것도 맞추지 않으므로 과적합이 원천적으로 없다.
  맞춘 비중이 이것보다 못하면 맞추는 행위 자체가 해로운 것이다
- **지금 챔피언**(LightGBM 0.5 + MLP 0.5, 리더보드 0.6416)과의 직접 비교

**비용**: 학습 0회. 1~2분.

In [24]:
from itertools import product

MEMBERS = ["L0_lgb", "L10_direct", "L6_mlp", "L8_xgb"]
FIT_FOLD = "B안 fold1"
HOLD = [ms.A_FOLD, "B안 fold2", "B안 fold3"]


def blend_pred(fold, weights):
    return {g: sum(w * PRED[(fold, m, g)] for m, w in zip(MEMBERS, weights))
            for g in TARGET_COLS}


def blend_score(fold, weights):
    info = ctx.fold_info[fold]
    pg = ms.per_group_scores(info["actual_df"],
                             pd.DataFrame(blend_pred(fold, weights), index=info["valid_idx"]))
    return ms.combine(pg)[0]


# ── fold1에서만 비중을 맞춘다 (0.1 단위 격자) ───────────────────────────────
STEP = 10
grid = [tuple(np.array(c) / STEP) for c in product(range(STEP + 1), repeat=len(MEMBERS))
        if sum(c) == STEP]
print(f"후보 비중 {len(grid)}가지를 [{FIT_FOLD}]에서만 평가한다 (학습 0회)")
scores = np.array([blend_score(FIT_FOLD, w) for w in grid])
W_FIT = grid[int(scores.argmax())]
print(f"  최적 비중 : " + "  ".join(f"{m.replace('L0_','').replace('L10_','').replace('L6_','').replace('L8_','')}={w:.1f}"
                                   for m, w in zip(MEMBERS, W_FIT)))
print(f"  fold1 점수 : {scores.max():.4f}  (LightGBM 단독 {blend_score(FIT_FOLD,(1,0,0,0)):.4f})")

# ── 비교 대상들 ─────────────────────────────────────────────────────────────
CANDS = {
    "지금 챔피언 (LGB.5 + MLP.5)": (0.5, 0.0, 0.5, 0.0),
    "LightGBM 단독":              (1.0, 0.0, 0.0, 0.0),
    "똑같이 3등분 (LGB/직접/MLP)": (1/3, 1/3, 1/3, 0.0),
    "fold1에서 맞춘 비중":         W_FIT,
}
rows = []
for label, w in CANDS.items():
    name = "MIX_" + label.split()[0]
    for fold in ctx.fold_info:
        for g in TARGET_COLS:
            PRED[(fold, name, g)] = sum(wi * PRED[(fold, m, g)] for m, wi in zip(MEMBERS, w))
    bench.add_from_cache(ctx, PRED, name)
    s = bench.fold_scores(name)
    rows.append(dict(구성=label, 이름=name, **{k: s[k] for k in [ms.A_FOLD, *ms.B_FOLDS]},
                     미래3개평균=s[HOLD].mean()))
tab = pd.DataFrame(rows).set_index("구성")
print("\n■ 결과   (fold1은 비중을 맞춘 곳이라 낙관적 → '미래3개평균'으로 판정)")
display(tab.drop(columns="이름").round(4))

# ── 판정: 지금 챔피언 대비 ──────────────────────────────────────────────────
champ = tab.loc["지금 챔피언 (LGB.5 + MLP.5)", "이름"]
print("\n" + "=" * 68)
for label in ["똑같이 3등분 (LGB/직접/MLP)", "fold1에서 맞춘 비중"]:
    print(f"■ [{label}]  vs  [지금 챔피언]     ※ fold1 제외하고 읽을 것")
    print("=" * 68)
    _ = bench.compare(champ, tab.loc[label, "이름"])
    d = tab.loc[label, "미래3개평균"] - tab.loc["지금 챔피언 (LGB.5 + MLP.5)", "미래3개평균"]
    print(f"  ★ 미래 3개 fold 평균 차이 : {d:+.4f}"
          f"  ({'채택' if d >= ms.MIN_EFFECT else '문턱 미달'})")
    print()

# ── 오차 폭이 실제로 줄었는지 (이론 예측과 대조) ────────────────────────────
def sig(name):
    acc = []
    for fold, info in ctx.fold_info.items():
        vi = info["valid_idx"]
        pr = pd.DataFrame({g: PRED[(fold, name, g)] for g in TARGET_COLS}, index=vi)
        acc.append(ms.per_group_scores(info["actual_df"], pr))
    return pd.concat(acc).groupby(level=0).mean()[["sigma", "ficr", "band6"]]

print("■ 오차 폭·밴드 통과율이 실제로 좋아졌나")
comp = pd.concat({lab: sig(tab.loc[lab, "이름"]) for lab in CANDS}, axis=1)
display(comp.round(4))

후보 비중 286가지를 [B안 fold1]에서만 평가한다 (학습 0회)
  최적 비중 : lgb=0.6  direct=0.2  mlp=0.2  xgb=0.0
  fold1 점수 : 0.6221  (LightGBM 단독 0.6177)

■ 결과   (fold1은 비중을 맞춘 곳이라 낙관적 → '미래3개평균'으로 판정)


,A안(2024),B안 fold1,B안 fold2,B안 fold3,미래3개평균
구성,,,,,
지금 챔피언 (LGB.5 + MLP.5),0.6429,0.6180,0.6417,0.6533,0.6460
LightGBM 단독,0.6418,0.6177,0.6428,0.6462,0.6436
똑같이 3등분 (LGB/직접/MLP),0.6453,0.6180,0.6413,0.6523,0.6463
fold1에서 맞춘 비중,0.6446,0.6221,0.6426,0.6506,0.6459



■ [똑같이 3등분 (LGB/직접/MLP)]  vs  [지금 챔피언]     ※ fold1 제외하고 읽을 것
[MIX_똑같이] vs [MIX_지금]  →  **동률**
  fold별 Δ : A(2024)=+0.0024  B fold1=-0.0001  B fold2=-0.0004  B fold3=-0.0010
  B평균 Δ  : -0.0005   문턱 ±0.0020 (실효크기 하한)   부호일치=O
             노이즈 내역 — 짝지은 SE 0.0003 / 시드 0.0000 / 하한 0.0020
  정제 Δ   : -0.0003  (방향 일치)
  ⇒ 구별 불가. 채택하지 않는다(더 단순한 쪽을 유지).
  ★ 미래 3개 fold 평균 차이 : +0.0003  (문턱 미달)

■ [fold1에서 맞춘 비중]  vs  [지금 챔피언]     ※ fold1 제외하고 읽을 것
[MIX_fold1에서] vs [MIX_지금]  →  **동률**
  fold별 Δ : A(2024)=+0.0017  B fold1=+0.0041  B fold2=+0.0009  B fold3=-0.0027
  B평균 Δ  : +0.0008   문턱 ±0.0039 (노이즈 0.0019 × 2.0)   부호일치=X
             노이즈 내역 — 짝지은 SE 0.0019 / 시드 0.0000 / 하한 0.0020
  정제 Δ   : +0.0021  (방향 일치)
  ⇒ 구별 불가. 채택하지 않는다(더 단순한 쪽을 유지).
  ★ 미래 3개 fold 평균 차이 : -0.0000  (문턱 미달)

■ 오차 폭·밴드 통과율이 실제로 좋아졌나


지금 챔피언 (LGB.5 + MLP.5)                 LightGBM 단독          \
                             sigma    ficr   band6       sigma    ficr   
group                                                                    
kpx_group_1                 0.1596  0.4130  0.3357      0.1644  0.4225   
kpx_group_2                 0.1634  0.4570  0.3801      0.1662  0.4557   
kpx_group_3                 0.1774  0.3408  0.2729      0.1845  0.3377   

                    똑같이 3등분 (LGB/직접/MLP)                 fold1에서 맞춘 비중  \
              band6                sigma    ficr   band6         sigma   
group                                                                    
kpx_group_1  0.3461               0.1573  0.4121  0.3311        0.1593   
kpx_group_2  0.3813               0.1605  0.4605  0.3801        0.1620   
kpx_group_3  0.2745               0.1742  0.3395  0.2724        0.1772   

                             
               ficr   band6  
group                        
kpx_group_1  0.4182  0.3345  
kpx_group_2  0.4633  0.3835  
kpx_group_3  0.3395  0.2729

---

# 12층 — 전제가 틀렸다: σ를 줄여도 FICR은 안 오른다

## 11층이 밝혀낸 것

블렌드로 **오차 폭(σ)은 예상대로 줄었다.** 그런데 **밴드 통과율(band6)은 오히려 떨어졌다.**

| | σ 변화 | band6 **기대** | band6 **실제** |
|---|---|---|---|
| group_1 | -4.3% | **+0.0123** | **-0.0150** |
| group_2 | -3.4% | +0.0096 | -0.0012 |
| group_3 | -5.6% | +0.0145 | -0.0021 |

정규분포라면 σ가 저만큼 줄 때 통과율이 **+0.010~+0.015** 올라야 한다.
**실제로는 전부 떨어졌다.**

그리고 더 분명한 대조가 있다.

| | σ | band6 | FICR |
|---|---|---|---|
| LightGBM **단독** | 0.1644 | **0.3461** | **0.4225** |
| 지금 챔피언(평균 여러 개) | **0.1596** | 0.3357 | 0.4130 |

**섞을수록 σ는 작아지는데 밴드는 덜 맞힌다.** (group_1 기준)

## 왜 이런 일이 생기나

**평균을 내면 오차 분포가 평평해진다.**

여러 모델을 평균하면 **크게 틀린 경우(꼬리)는 확실히 줄어든다.** σ는 제곱으로 재기 때문에
꼬리가 지배하고, 그래서 σ가 작아진다.

그런데 **정확히 맞힌 경우(봉우리)도 같이 뭉개진다.** 여러 값을 평균하면 결과가
가운데로 몰리면서 분포가 종 모양에 가까워진다(중심극한정리). **뾰족함을 잃는다.**

**FICR은 봉우리만 본다.** ±6% 안에 들어왔느냐만 따지고, 밖으로 얼마나 나갔는지는
5%든 500%든 똑같이 0원이다.

> **σ를 줄이는 것과 밴드를 맞히는 것은 다른 일이었다.**
> 우리는 지금까지 **σ를 줄이면 FICR이 오른다**는 전제로 움직였다. 04의 `FICR ≈ σ^-0.86`,
> "오차 폭이 병목", "1등이 되려면 σ를 13% 줄여야 한다" — **전부 이 전제 위에 있었다.**

## 그래서 방향을 바꾼다 — 노릴 것은 σ가 아니라 band6

여기서 바로 나오는 가설: **우리 예측은 지나치게 뭉개져 있다(과도한 평균화).**

지금 챔피언은 LightGBM 시드 5개 평균 + MLP 시드 5개 평균을 다시 섞는다.
**평균을 세 번 겹쳐 쓰고 있다.** 그럴수록 봉우리가 낮아진다.

## 이 셀 — 세 가지를 학습 0회로 시험한다

### ① σ와 band6가 정말 따로 노는가 (전 변형 대조표)

지금까지 만든 모든 변형에 대해 σ와 band6를 나란히 놓는다.
둘의 순위가 어긋나면 전제가 틀렸다는 게 확정된다.

### ② 평균 대신 중앙값으로 섞기

평균은 봉우리를 뭉갠다. **중앙값은 덜 뭉갠다** — 튀는 값 하나에 끌려가지 않고
가운데 값을 그대로 고르기 때문이다. 같은 모델들을 중앙값으로 섞어 band6를 비교한다.

### ③ 예측을 다시 뾰족하게 펴기 (확대)

평균화가 예측을 가운데로 끌어당겼다면, 다시 밖으로 밀어내면 된다.

```
새 예측 = 평균수준 + k × (예측 − 평균수준)      k > 1이면 벌린다
```

k>1은 σ를 **키운다**. 그런데 예측이 원래 과도하게 눌려 있었다면 **band6는 오른다.**
`k`는 **fold1에서만 맞추고 fold2·fold3·A안에서 채점**한다(5층에서 쓴 그 설계).

⚠️ 이건 04가 닫은 "위치 최적화"와 다르다. 그건 **전부 같은 방향으로 미는 것**(평행이동)이었고,
이건 **가운데는 두고 양쪽으로 벌리는 것**(확대)이다. 편향은 거의 안 변하고 폭만 변한다.

**비용**: 학습 0회. 2~3분.

In [25]:
# ══════════════════════════════════════════════════════════════════
# ① σ 와 band6 가 따로 노는가 — 전 변형 대조
# ══════════════════════════════════════════════════════════════════
def diag(name):
    acc = []
    for fold, info in ctx.fold_info.items():
        vi = info["valid_idx"]
        pr = pd.DataFrame({g: PRED[(fold, name, g)] for g in TARGET_COLS}, index=vi)
        acc.append(ms.per_group_scores(info["actual_df"], pr))
    m = pd.concat(acc).groupby(level=0).mean()
    s = bench.fold_scores(name)
    return dict(sigma=m["sigma"].mean(), band6=m["band6"].mean(), band8=m["band8"].mean(),
                FICR=m["ficr"].mean(), B평균=s[ms.B_FOLDS].mean())

WATCH = {
    "LightGBM 단독": "L0_lgb", "직접모델 단독": "L10_direct", "산식MLP 단독": "L6_mlp",
    "XGBoost 단독": "L8_xgb", "GBDT 3종 평균": "L8_mix3",
    "챔피언 LGB.5+MLP.5": "MIX_지금", "3등분 평균": "MIX_똑같이",
}
D = pd.DataFrame({k: diag(v) for k, v in WATCH.items()}).T
D["σ 순위"] = D["sigma"].rank()                 # 작을수록 1등
D["band6 순위"] = D["band6"].rank(ascending=False)   # 클수록 1등
print("■ σ 와 band6 의 순위가 어긋나는가  (σ는 작을수록·band6는 클수록 1등)")
display(D.round(4))
print(f"두 순위의 상관 = {D['σ 순위'].corr(D['band6 순위']):+.3f}"
      "   (+1이면 '작은 σ = 높은 band6', -1이면 완전히 반대)")

# ══════════════════════════════════════════════════════════════════
# ② 평균 대신 중앙값으로 섞기
# ══════════════════════════════════════════════════════════════════
TRIO = ["L0_lgb", "L10_direct", "L6_mlp"]
for fold in ctx.fold_info:
    for g in TARGET_COLS:
        stack = np.column_stack([PRED[(fold, m, g)].to_numpy() for m in TRIO])
        idx = PRED[(fold, TRIO[0], g)].index
        PRED[(fold, "MED3", g)] = pd.Series(np.median(stack, axis=1), index=idx)
bench.add_from_cache(ctx, PRED, "MED3")
print("\n■ 평균 vs 중앙값 (같은 세 모델)")
display(pd.DataFrame({"평균 3등분": diag("MIX_똑같이"), "중앙값 3등분": diag("MED3"),
                      "LightGBM 단독": diag("L0_lgb")}).T.round(4))

# ══════════════════════════════════════════════════════════════════
# ③ 예측을 다시 벌린다 (확대) — fold1에서만 맞추고 미래에서 채점
# ══════════════════════════════════════════════════════════════════
KS = [0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20, 1.30]
BASES = {"챔피언": "MIX_지금", "LightGBM 단독": "L0_lgb", "3등분 평균": "MIX_똑같이"}
FIT_FOLD, HOLD = "B안 fold1", [ms.A_FOLD, "B안 fold2", "B안 fold3"]


def widen(fold, g, base, k):
    """학습 구간 라벨 평균을 중심으로 예측을 k배 벌린다. (검증 라벨은 안 본다)"""
    info = ctx.fold_info[fold]
    mu = ctx.train.loc[info["train_mask"], g].mean()
    p = PRED[(fold, base, g)]
    return (mu + k * (p - mu)).clip(lower=0, upper=CAPACITY_KWH[g])


rows = []
for label, base in BASES.items():
    for k in KS:
        name = f"WID_{label[:3]}_{int(round(k*100))}"
        for fold in ctx.fold_info:
            for g in TARGET_COLS:
                PRED[(fold, name, g)] = widen(fold, g, base, k)
        bench.add_from_cache(ctx, PRED, name)
        s = bench.fold_scores(name)
        d = diag(name)
        rows.append(dict(기준=label, k=k, fold1=s[FIT_FOLD], 미래3개=s[HOLD].mean(),
                         σ=d["sigma"], band6=d["band6"], FICR=d["FICR"]))
W = pd.DataFrame(rows)
print("\n■ 예측을 k배 벌렸을 때  (k=1.00이 지금 그대로)")
for label in BASES:
    sub = W[W["기준"] == label].set_index("k")
    print(f"\n  [{label}]")
    display(sub.drop(columns="기준").round(4))
    k_fit = sub["fold1"].idxmax()
    print(f"    fold1이 고른 k = {k_fit}  →  미래3개 {sub.loc[k_fit,'미래3개']:.4f} "
          f"(k=1.00일 때 {sub.loc[1.00,'미래3개']:.4f}, 차이 {sub.loc[k_fit,'미래3개']-sub.loc[1.00,'미래3개']:+.4f})")

print("\n" + "=" * 68)
print("판정: fold1이 고른 k가 1보다 크고, 미래 3개 fold에서도 +0.002 이상이면 채택.")
print("      k>1이 이기면 '우리 예측이 과도하게 뭉개져 있었다'가 확정된다.")

■ σ 와 band6 의 순위가 어긋나는가  (σ는 작을수록·band6는 클수록 1등)


,sigma,band6,band8,FICR,B평균,σ 순위,band6 순위
LightGBM 단독,0.1717,0.3339,0.4291,0.4053,0.6356,5.0,1.0
직접모델 단독,0.1719,0.3151,0.4098,0.3861,0.6225,6.0,7.0
산식MLP 단독,0.1669,0.3213,0.4158,0.3922,0.6306,3.0,6.0
XGBoost 단독,0.1734,0.3260,0.4222,0.3981,0.6309,7.0,5.0
GBDT 3종 평균,0.1711,0.3271,0.4226,0.3987,0.6332,4.0,4.0
챔피언 LGB.5+MLP.5,0.1668,0.3296,0.4283,0.4036,0.6377,2.0,2.0
3등분 평균,0.1640,0.3279,0.4294,0.4040,0.6372,1.0,3.0


두 순위의 상관 = +0.393   (+1이면 '작은 σ = 높은 band6', -1이면 완전히 반대)

■ 평균 vs 중앙값 (같은 세 모델)


,sigma,band6,band8,FICR,B평균
평균 3등분,0.1640,0.3279,0.4294,0.4040,0.6372
중앙값 3등분,0.1665,0.3307,0.4296,0.4049,0.6371
LightGBM 단독,0.1717,0.3339,0.4291,0.4053,0.6356



■ 예측을 k배 벌렸을 때  (k=1.00이 지금 그대로)

  [챔피언]


...

    fold1이 고른 k = 1.05  →  미래3개 0.6484 (k=1.00일 때 0.6460, 차이 +0.0024)

  [LightGBM 단독]


,fold1,미래3개,σ,band6,FICR
k,,,,,
0.90,0.6038,0.6236,0.1656,0.2911,0.3644
0.95,0.6153,0.6358,0.1681,0.3115,0.3898
1.00,0.6177,0.6436,0.1717,0.3339,0.4053
1.05,0.6112,0.6413,0.1761,0.3348,0.4030
1.10,0.6024,0.6312,0.1812,0.3241,0.3894
1.15,0.5883,0.6158,0.1868,0.3024,0.3664
1.20,0.5744,0.5994,0.1924,0.2806,0.3421
1.30,0.5507,0.5703,0.2030,0.2428,0.3002


    fold1이 고른 k = 1.0  →  미래3개 0.6436 (k=1.00일 때 0.6436, 차이 +0.0000)

  [3등분 평균]


,fold1,미래3개,σ,band6,FICR
k,,,,,
0.90,0.5952,0.6252,0.1604,0.2910,0.3595
0.95,0.6072,0.6372,0.1617,0.3086,0.3834
1.00,0.6180,0.6463,0.1640,0.3279,0.4040
1.05,0.6209,0.6488,0.1672,0.3407,0.4123
1.10,0.6174,0.6444,0.1712,0.3390,0.4087
1.15,0.6077,0.6313,0.1760,0.3223,0.3902
1.20,0.5953,0.6153,0.1811,0.3017,0.3668
1.30,0.5741,0.5867,0.1916,0.2663,0.3270


    fold1이 고른 k = 1.05  →  미래3개 0.6488 (k=1.00일 때 0.6463, 차이 +0.0025)

판정: fold1이 고른 k가 1보다 크고, 미래 3개 fold에서도 +0.002 이상이면 채택.
      k>1이 이기면 '우리 예측이 과도하게 뭉개져 있었다'가 확정된다.


---

# 13층 — 확대가 통했다. 이제 안전한지 확인하고 다듬는다

## 12층 결과

| 기준 | fold1이 고른 k | 미래 3개 fold | k=1.00 대비 |
|---|---|---|---|
| 챔피언 (LGB+MLP) | **1.05** | 0.6484 | **+0.0024** |
| 3등분 평균 | **1.05** | **0.6488** | **+0.0025** |
| LightGBM **단독** | **1.00** | 0.6436 | +0.0000 |

**단독 모델은 벌릴 필요가 없었고, 평균낸 모델만 이득을 봤다.**
"평균화가 예측을 눌러놨다"는 설명이 정확히 맞았다.

3등분 평균에서 k를 1.00 → 1.05로 올렸을 때:

| | σ | band6 | FICR |
|---|---|---|---|
| k=1.00 | 0.1640 | 0.3279 | 0.4040 |
| k=1.05 | **0.1672** (나빠짐) | **0.3407** | **0.4123** |

**σ를 일부러 2% 악화시키고 밴드 통과율을 +0.0128 얻었다.**

## ⚠️ 그런데 제출 전에 반드시 확인할 것 — v3의 재현 위험

`k>1`은 평균보다 **위에 있는 예측을 더 위로** 민다.
그런데 채점 대상 행(발전량이 큰 시간)은 대부분 평균보다 위에 있으므로,
**채점 행에서는 사실상 상향 조정**이다.

여기서 확정된 교훈이 걸린다.

> **로컬 편향이 +0.03을 넘는 구성은 로컬 점수를 믿지 말 것.**
> v3는 로컬이 +0.0110 올랐는데 리더보드는 **-0.0061 떨어졌다.**
> 2025년은 예보 풍속이 3년 중 가장 센 해라 예측이 애초에 높은 쪽에 있고, 위로 밀 여유가 없다.
> 이전 프로젝트의 아핀 후처리(exp019)도 같은 이유로 실패했다 — **독립 2회 확인**.

**편향을 안 보고 제출하면 v3를 그대로 반복한다.** 이 셀은 편향을 최우선으로 찍는다.

## 이 셀이 하는 세 가지 (전부 학습 0회)

### ① k를 촘촘히 + 편향을 같이 본다

0.02 간격으로 1.00~1.14를 훑고, **채점 행 기준 편향**을 나란히 놓는다.
편향이 +0.03을 넘는 k는 **로컬 점수가 아무리 좋아도 후보에서 뺀다.**

### ② 그룹마다 다른 k가 필요한가

group_3만 오차 폭이 다르다(0.185 vs 0.164). 눌린 정도도 다를 수 있다.
다만 **그룹별 파라미터는 04에서 네 번 연속 기각**됐으므로 기대는 낮게 잡는다.
이번엔 기계론이 다르므로 한 번은 확인한다.

### ③ 비중과 k를 같이 맞추면

지금은 "3등분으로 고정하고 k만" 맞췄다. 둘을 같이 맞추면 더 좋을 수 있지만
**한 fold에서 여러 손잡이를 맞추면 그 fold에만 맞는 답이 나온다**(5층에서 겪었다).
그래서 결과를 내되 **fold1 점수와 미래 점수를 나란히 놓고** 벌어지는지 본다.
벌어지면 과적합이므로 단순한 쪽(3등분 + k 하나)을 택한다.

**비용**: 학습 0회. 2~3분.

In [28]:
for _v in ("k_fit", "safe", "K"):          # 12층에서 쓰던 같은 이름이 남아 있으면 지운다
    globals().pop(_v, None)

KS2 = [1.00, 1.02, 1.04, 1.05, 1.06, 1.08, 1.10, 1.12]
FIT_FOLD, HOLD = "B안 fold1", [ms.A_FOLD, "B안 fold2", "B안 fold3"]
BASE3 = "MIX_똑같이"          # 3등분 평균 (LGB / 직접 / MLP)

# 안전 문턱 — HANDOFF 실측 기준은 **세 그룹 평균 편향**이다
#   v3 +0.051 → 리더보드 깨짐 / v4 +0.024, v5 정상
BIAS_LIMIT = 0.030


def widen_pred(fold, g, base, k):
    info = ctx.fold_info[fold]
    mu = ctx.train.loc[info["train_mask"], g].mean()
    return (mu + k * (PRED[(fold, base, g)] - mu)).clip(lower=0, upper=CAPACITY_KWH[g])


def full_diag(name):
    acc = []
    for fold, info in ctx.fold_info.items():
        vi = info["valid_idx"]
        pr = pd.DataFrame({g: PRED[(fold, name, g)] for g in TARGET_COLS}, index=vi)
        acc.append(ms.per_group_scores(info["actual_df"], pr))
    m = pd.concat(acc).groupby(level=0).mean()
    s = bench.fold_scores(name)
    return dict(fold1=s[FIT_FOLD], 미래3개=s[HOLD].mean(), B평균=s[ms.B_FOLDS].mean(),
                편향=m["bias"].mean(), 편향_최대그룹=m["bias"].max(),
                σ=m["sigma"].mean(), band6=m["band6"].mean(), FICR=m["ficr"].mean())


# ── ① k 정밀 + 편향 안전점검 ────────────────────────────────────────────────
rows = []
for k in KS2:
    name = f"K3_{int(round(k*100))}"
    for fold in ctx.fold_info:
        for g in TARGET_COLS:
            PRED[(fold, name, g)] = widen_pred(fold, g, BASE3, k)
    bench.add_from_cache(ctx, PRED, name)
    rows.append(dict(k=k, 이름=name, **full_diag(name)))
K = pd.DataFrame(rows).set_index("k")
K["편향 안전"] = np.where(K["편향"] <= BIAS_LIMIT, "OK", "⚠️ 위험")
print(f"■ 확대 배율 k — 점수와 편향을 같이 본다  (기준: 3등분 평균, 안전선 편향 ≤ {BIAS_LIMIT})")
display(K.drop(columns="이름").round(4))
print("   참고: v3 편향 +0.051 → 리더보드 깨짐 / v4 +0.024, v5 정상")

safe = K[K["편향"] <= BIAS_LIMIT]
if len(safe) == 0:
    print("\n⚠️ 안전선을 넘는 k가 하나도 없다. k=1.00(확대 없음)으로 되돌린다.")
    k_fit = 1.00
    safe = K.loc[[1.00]]
else:
    k_fit = safe["fold1"].idxmax()
    print(f"\n편향 안전한 k : {[float(x) for x in safe.index]}   →  fold1이 고른 k = {k_fit}")
print(f"   미래 3개 fold : {safe.loc[k_fit,'미래3개']:.4f}  "
      f"(k=1.00 {K.loc[1.00,'미래3개']:.4f}, 차이 {safe.loc[k_fit,'미래3개']-K.loc[1.00,'미래3개']:+.4f})")
print(f"   편향 : 평균 {safe.loc[k_fit,'편향']:+.4f} / 최대그룹 {safe.loc[k_fit,'편향_최대그룹']:+.4f}")

# ── ② 그룹마다 다른 k가 필요한가 ───────────────────────────────────────────
print("\n\n■ 그룹별로 k를 따로 고르면 (fold1 기준, 그 그룹의 기여만 최대화)")
per_g = {}
for g in TARGET_COLS:
    best_k, best_s = 1.00, -9
    for k in KS2:
        info = ctx.fold_info[FIT_FOLD]
        pr = {gg: (widen_pred(FIT_FOLD, gg, BASE3, k) if gg == g
                   else PRED[(FIT_FOLD, BASE3, gg)]) for gg in TARGET_COLS}
        pg = ms.per_group_scores(info["actual_df"], pd.DataFrame(pr, index=info["valid_idx"]))
        s = 0.5 * pg.loc[g, "ficr"] - 0.5 * pg.loc[g, "nmae"]
        if s > best_s:
            best_k, best_s = k, s
    per_g[g] = best_k
print("   " + "  ".join(f"{g.split('_')[-1]}: k={v}" for g, v in per_g.items()))
for fold in ctx.fold_info:
    for g in TARGET_COLS:
        PRED[(fold, "K3_pergroup", g)] = widen_pred(fold, g, BASE3, per_g[g])
bench.add_from_cache(ctx, PRED, "K3_pergroup")
pgr = full_diag("K3_pergroup")
print(f"   미래3개 {pgr['미래3개']:.4f} (편향 {pgr['편향']:+.4f})  vs  공통 k "
      f"{safe.loc[k_fit,'미래3개']:.4f}  →  {pgr['미래3개']-safe.loc[k_fit,'미래3개']:+.4f}")
print("   → 공통 k보다 +0.002 이상 나아야 채택. 04에서 그룹별 파라미터는 네 번 기각됐다.")

# ── ③ 비중과 k를 같이 맞추면 (과적합 점검) ─────────────────────────────────
print("\n\n■ 비중과 k를 같이 맞추면 (fold1에서 맞추고 미래에서 채점, 528가지)")
best = None
for wl in np.arange(0, 1.01, 0.1):
    for wd in np.arange(0, 1.01 - wl + 1e-9, 0.1):
        wm = 1 - wl - wd
        for k in KS2:
            info = ctx.fold_info[FIT_FOLD]
            pr = {}
            for g in TARGET_COLS:
                mu = ctx.train.loc[info["train_mask"], g].mean()
                p = (wl * PRED[(FIT_FOLD, "L0_lgb", g)] + wd * PRED[(FIT_FOLD, "L10_direct", g)]
                     + wm * PRED[(FIT_FOLD, "L6_mlp", g)])
                pr[g] = (mu + k * (p - mu)).clip(lower=0, upper=CAPACITY_KWH[g])
            sc = ms.combine(ms.per_group_scores(info["actual_df"],
                                                pd.DataFrame(pr, index=info["valid_idx"])))[0]
            if best is None or sc > best[0]:
                best = (sc, wl, wd, wm, k)
_, WL, WD, WM, KK = best
print(f"   fold1 최적: LGB={WL:.1f} 직접={WD:.1f} MLP={WM:.1f}, k={KK}  (fold1 {best[0]:.4f})")
for fold in ctx.fold_info:
    for g in TARGET_COLS:
        mu = ctx.train.loc[ctx.fold_info[fold]["train_mask"], g].mean()
        p = (WL * PRED[(fold, "L0_lgb", g)] + WD * PRED[(fold, "L10_direct", g)]
             + WM * PRED[(fold, "L6_mlp", g)])
        PRED[(fold, "JOINT", g)] = (mu + KK * (p - mu)).clip(lower=0, upper=CAPACITY_KWH[g])
bench.add_from_cache(ctx, PRED, "JOINT")
j = full_diag("JOINT")
print(f"   미래3개 {j['미래3개']:.4f}  |  편향 {j['편향']:+.4f}  |  "
      f"fold1 {j['fold1']:.4f}  → fold1과 미래의 벌어짐 {j['fold1']-j['미래3개']:+.4f}")

# ── 최종 비교 ───────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("■ 후보 정리   (판정은 '미래3개' — fold1은 손잡이를 맞춘 곳이라 제외)")
print("=" * 72)
FINAL = {"지금 챔피언 (v6 구성)": "MIX_지금", "3등분 평균 k=1.00": BASE3,
         f"3등분 평균 k={k_fit}": safe.loc[k_fit, "이름"],
         "그룹별 k": "K3_pergroup", "비중+k 동시": "JOINT"}
tabf = pd.DataFrame({lab: full_diag(n) for lab, n in FINAL.items()}).T
tabf["편향 안전"] = np.where(tabf["편향"] <= BIAS_LIMIT, "OK", "⚠️")
display(tabf.round(4))
base_f = tabf.loc["지금 챔피언 (v6 구성)", "미래3개"]
print("\n지금 챔피언 대비 미래3개 차이:")
for lab in FINAL:
    d = tabf.loc[lab, "미래3개"] - base_f
    ok = d >= ms.MIN_EFFECT and tabf.loc[lab, "편향"] <= BIAS_LIMIT
    print(f"   {lab:24s} {d:+.4f}   편향 {tabf.loc[lab,'편향']:+.4f}   {'★ 채택 가능' if ok else ''}")

■ 확대 배율 k — 점수와 편향을 같이 본다  (기준: 3등분 평균, 안전선 편향 ≤ 0.03)


,fold1,미래3개,B평균,편향,편향_최대그룹,σ,band6,FICR,편향 안전
k,,,,,,,,,
1.00,0.6180,0.6463,0.6372,0.0223,0.0335,0.1640,0.3279,0.4040,OK
1.02,0.6201,0.6485,0.6394,0.0264,0.0376,0.1652,0.3333,0.4095,OK
1.04,0.6217,0.6493,0.6409,0.0306,0.0417,0.1665,0.3393,0.4127,⚠️ 위험
1.05,0.6209,0.6488,0.6402,0.0327,0.0437,0.1672,0.3407,0.4123,⚠️ 위험
1.06,0.6210,0.6481,0.6397,0.0348,0.0457,0.1679,0.3411,0.4122,⚠️ 위험
1.08,0.6192,0.6469,0.6383,0.0389,0.0498,0.1695,0.3405,0.4112,⚠️ 위험
1.10,0.6174,0.6444,0.6360,0.0431,0.0539,0.1712,0.3390,0.4087,⚠️ 위험
1.12,0.6149,0.6396,0.6320,0.0472,0.0579,0.1731,0.3326,0.4025,⚠️ 위험


   참고: v3 편향 +0.051 → 리더보드 깨짐 / v4 +0.024, v5 정상

편향 안전한 k : [1.0, 1.02]   →  fold1이 고른 k = 1.02
   미래 3개 fold : 0.6485  (k=1.00 0.6463, 차이 +0.0022)
   편향 : 평균 +0.0264 / 최대그룹 +0.0376


■ 그룹별로 k를 따로 고르면 (fold1 기준, 그 그룹의 기여만 최대화)
   1: k=1.12  2: k=1.0  3: k=1.12
   미래3개 0.6516 (편향 +0.0391)  vs  공통 k 0.6485  →  +0.0031
   → 공통 k보다 +0.002 이상 나아야 채택. 04에서 그룹별 파라미터는 네 번 기각됐다.


■ 비중과 k를 같이 맞추면 (fold1에서 맞추고 미래에서 채점, 528가지)
   fold1 최적: LGB=0.5 직접=0.3 MLP=0.2, k=1.02  (fold1 0.6227)
   미래3개 0.6481  |  편향 +0.0291  |  fold1 0.6227  → fold1과 미래의 벌어짐 -0.0254

■ 후보 정리   (판정은 '미래3개' — fold1은 손잡이를 맞춘 곳이라 제외)


,fold1,미래3개,B평균,편향,편향_최대그룹,σ,band6,FICR,편향 안전
지금 챔피언 (v6 구성),0.6180,0.6460,0.6377,0.0120,0.0235,0.1668,0.3296,0.4036,OK
3등분 평균 k=1.00,0.6180,0.6463,0.6372,0.0223,0.0335,0.1640,0.3279,0.4040,OK
3등분 평균 k=1.02,0.6201,0.6485,0.6394,0.0264,0.0376,0.1652,0.3333,0.4095,OK
그룹별 k,0.6257,0.6516,0.6434,0.0391,0.0529,0.1695,0.3495,0.4213,⚠️
비중+k 동시,0.6227,0.6481,0.6400,0.0291,0.0403,0.1666,0.3346,0.4115,OK



지금 챔피언 대비 미래3개 차이:
   지금 챔피언 (v6 구성)           +0.0000   편향 +0.0120   
   3등분 평균 k=1.00            +0.0003   편향 +0.0223   
   3등분 평균 k=1.02            +0.0026   편향 +0.0264   ★ 채택 가능
   그룹별 k                    +0.0057   편향 +0.0391   
   비중+k 동시                  +0.0021   편향 +0.0291   ★ 채택 가능


---

# 14층 — 리더보드로 보낸다 (학습 0회)

## 13층 결론

| 후보 | 미래3개 | 챔피언 대비 | 편향 | band6 | FICR | 안전 |
|---|---|---|---|---|---|---|
| 지금 챔피언 (v6) | 0.6460 | — | +0.0120 | 0.3296 | 0.4036 | OK |
| **3등분 k=1.02** | **0.6485** | **+0.0026** | +0.0264 | 0.3333 | 0.4095 | **OK** |
| 그룹별 k (1.12/1.00/1.12) | **0.6516** | **+0.0057** | **+0.0391** | 0.3495 | 0.4213 | ⚠️ |
| 비중+k 동시 | 0.6481 | +0.0021 | +0.0291 | 0.3346 | 0.4115 | OK |

**"비중+k 동시"가 단순한 k 하나보다 못했다** — 손잡이를 늘려도 안 좋아졌으니 단순한 쪽이 맞다.

**그룹별 k에서 group_2만 k=1.00을 골랐다.** group_2는 원래 편향이 가장 큰 그룹이다.
최적화가 **"이미 위로 밀린 그룹은 더 밀지 마라"** 를 스스로 찾았다 — 우연이 아니라 기계론이다.
다만 평균 편향 **+0.0391**은 v4(+0.024, 정상)와 v3(+0.051, 깨짐) 사이의 **미지 구간**이다.

## 지금 제출해야 하는 이유

지금까지 전부 **로컬 숫자**다. 리더보드는 08-03 v7 이후 멈춰 있다.
그리고 **확대(k)는 v6 부품에 그대로 얹을 수 있다** — `models/v5_parts.npz`에
v6의 test 예측 조각(LightGBM 5시드 · 산식MLP 5시드)이 저장돼 있어 **학습이 0회**다.

## 왜 챔피언 구성에 얹나 (3등분이 아니라)

3등분은 직접모델을 새로 학습해 test에 적용하는 경로를 짜야 한다(시간이 걸리고 실수 위험도 있다).
반면 **챔피언 + 확대**는 지금 있는 부품만으로 만들 수 있고, **가장 중요한 새 발견(확대) 하나만**
검증한다. 한 번에 하나만 바꾸는 원칙에도 맞다.

12층 측정: 챔피언 기준 **k=1.05에서 미래3개 0.6484(+0.0024)**.
챔피언은 편향이 **+0.0120**으로 3등분(+0.0223)보다 훨씬 낮아 **확대 여유가 더 크다.**

## 이 셀의 순서

1. 챔피언 기준으로 k를 다시 훑고 **편향을 같이** 본다 (로컬, 학습 0회)
2. 편향이 안전한 범위에서 k를 고른다
3. `v5_parts.npz`로 v6를 그대로 복원 → **v6와 비트 단위로 같은지 먼저 확인**
4. 확대를 적용해 `v8` 저장

⚠️ **기존 제출 파일은 절대 건드리지 않는다.** v1~v7이 있으므로 **v8**로 저장한다.
⚠️ `src/submission.py`의 경로 두 개(`SAMPLE_SUBMISSION_PATH`, `SUBMISSIONS_DIR`)를
**둘 다** 절대경로로 덮어쓴다. 하나만 바꾸면 **에러 없이** `notebooks/submissions/`에 조용히 저장된다.

In [31]:
import src.submission as sub
importlib.reload(sub)

# ⚠️ 함정: submission.py 의 함수들은 경로를 **기본 인자**로 받는다.
#      def build_submission(pred_df, sample_path=SAMPLE_SUBMISSION_PATH)
#    기본 인자는 **함수가 정의되는 순간** 값이 박히므로, 나중에 모듈 변수를 바꿔도 안 바뀐다.
#    (SUBMISSIONS_DIR 은 함수 '안에서' 읽으므로 덮어쓰기가 통한다 — 그래서 하나만 통했다.)
#    ⇒ sample_path 는 **호출할 때마다 직접 넘긴다.**
SAMPLE = pl.REPO_ROOT / "data" / "sample_submission.csv"
sub.SUBMISSIONS_DIR = pl.REPO_ROOT / "submissions"
assert SAMPLE.exists(), f"sample_submission 을 못 찾았다: {SAMPLE}"

BIAS_LIMIT = 0.030
CHAMP = "MIX_지금"          # LightGBM 0.5 + 산식MLP 0.5  = v6 구성

# ── 1) 챔피언 기준으로 k를 훑는다 (로컬) ───────────────────────────────────
rows = []
for k in [1.00, 1.02, 1.04, 1.05, 1.06, 1.08, 1.10, 1.12, 1.15]:
    name = f"KC_{int(round(k*100))}"
    for fold in ctx.fold_info:
        for g in TARGET_COLS:
            mu = ctx.train.loc[ctx.fold_info[fold]["train_mask"], g].mean()
            PRED[(fold, name, g)] = (mu + k * (PRED[(fold, CHAMP, g)] - mu)
                                     ).clip(lower=0, upper=CAPACITY_KWH[g])
    bench.add_from_cache(ctx, PRED, name)
    rows.append(dict(k=k, 이름=name, **full_diag(name)))
KC = pd.DataFrame(rows).set_index("k")
KC["편향 안전"] = np.where(KC["편향"] <= BIAS_LIMIT, "OK", "⚠️ 위험")
print("■ 챔피언(v6 구성) 기준 확대 배율")
display(KC.drop(columns="이름").round(4))

safeC = KC[KC["편향"] <= BIAS_LIMIT]
K_SUB = float(safeC["fold1"].idxmax()) if len(safeC) else 1.00
print(f"\n편향 안전한 k : {[float(x) for x in safeC.index]}")
print(f"fold1이 고른 k = {K_SUB}")
print(f"   미래3개 {KC.loc[K_SUB,'미래3개']:.4f} (k=1.00 {KC.loc[1.00,'미래3개']:.4f}, "
      f"{KC.loc[K_SUB,'미래3개']-KC.loc[1.00,'미래3개']:+.4f})   편향 {KC.loc[K_SUB,'편향']:+.4f}")

# ── 2) v6 부품을 불러 v6를 그대로 복원 (비트 단위 확인) ────────────────────
parts = np.load(pl.REPO_ROOT / "models" / "v5_parts.npz")

# ⚠️ 지금 커널은 with_test=False 로 로드했으므로 ctx.test 가 없다.
#    커널을 다시 로드하면 학습 캐시가 통째로 날아가므로, **시각 컬럼만** 따로 읽는다.
#    v5_parts 의 8760행은 이 parquet 의 행 순서와 같다(만들 때 ctx.test 를 썼다).
if getattr(ctx, "test", None) is not None:
    test_dtm = ctx.test["kst_dtm"].reset_index(drop=True)
else:
    test_dtm = pd.read_parquet(pl.PROCESSED_DIR / "test_features_v1.parquet",
                               columns=["kst_dtm"])["kst_dtm"].reset_index(drop=True)
    print("(ctx.test 가 없어 parquet 에서 시각만 읽었다)")
assert len(test_dtm) == 8760, f"test 행 수가 {len(test_dtm)}"
assert str(test_dtm.iloc[0]).startswith("2025-01-01 01:00"), f"첫 시각이 {test_dtm.iloc[0]}"
assert str(test_dtm.iloc[-1]).startswith("2026-01-01 00:00"), f"끝 시각이 {test_dtm.iloc[-1]}"

v6 = {g: 0.5 * parts[f"lgb_{g}"] + 0.5 * parts[f"mlp_{g}"] for g in TARGET_COLS}
df_v6 = pd.DataFrame({"forecast_kst_dtm": test_dtm.dt.strftime("%Y-%m-%d %H:%M:%S").to_numpy(),
                      **v6})
s_v6 = sub.build_submission(df_v6, sample_path=SAMPLE)
old = pd.read_csv(pl.REPO_ROOT / "submissions" / "20260803_v6_metricmlp05.csv", encoding="utf-8-sig")
diff = np.abs(s_v6[TARGET_COLS].to_numpy() - old[TARGET_COLS].to_numpy()).max()
print(f"\n■ v6 복원 검증 : 기존 제출 파일과 최대 차이 {diff:.6f} kWh  "
      + ("→ 동일 OK" if diff < 1e-6 else "→ ⚠️ 다르다! 부품이 v6가 아니다"))
assert diff < 1e-6, "v6 복원 실패 — 여기서 멈출 것"

# ── 3) 확대를 적용해 v8 저장 ───────────────────────────────────────────────
MU = {g: ctx.train[g].mean() for g in TARGET_COLS}       # 전체 학습기간 라벨 평균
print(f"\n■ 확대 중심값(학습 라벨 평균): "
      + ", ".join(f"{g.split('_')[-1]}={MU[g]:,.0f}" for g in TARGET_COLS))

v8 = {g: np.clip(MU[g] + K_SUB * (v6[g] - MU[g]), 0, CAPACITY_KWH[g]) for g in TARGET_COLS}
df_v8 = pd.DataFrame({"forecast_kst_dtm": test_dtm.dt.strftime("%Y-%m-%d %H:%M:%S").to_numpy(),
                      **v8})
s_v8 = sub.build_submission(df_v8, sample_path=SAMPLE)
sub.validate_submission(s_v8, sample_path=SAMPLE)

chg = pd.DataFrame({
    "v6 평균": [v6[g].mean() for g in TARGET_COLS],
    "v8 평균": [v8[g].mean() for g in TARGET_COLS],
    "변화%": [(v8[g].mean() / v6[g].mean() - 1) * 100 for g in TARGET_COLS],
    "v6 표준편차": [v6[g].std() for g in TARGET_COLS],
    "v8 표준편차": [v8[g].std() for g in TARGET_COLS],
    "0으로 잘린 행": [int((MU[g] + K_SUB * (v6[g] - MU[g]) < 0).sum()) for g in TARGET_COLS],
    "용량에 걸린 행": [int((MU[g] + K_SUB * (v6[g] - MU[g]) > CAPACITY_KWH[g]).sum())
                  for g in TARGET_COLS],
}, index=[g.split("_")[-1] for g in TARGET_COLS])
print("\n■ v6 → v8 예측값이 어떻게 바뀌었나")
display(chg.round(2))

fname = f"20260804_v8_widen{int(round(K_SUB*100))}.csv"
path = sub.save_submission(s_v8, fname, sample_path=SAMPLE)
print(f"\n저장 완료: {path}")
print(f"   구성: v6(LightGBM 0.5 + 산식MLP 0.5) 에 확대 k={K_SUB} 적용")
print(f"   로컬 미래3개 {KC.loc[K_SUB,'미래3개']:.4f} (v6 {KC.loc[1.00,'미래3개']:.4f})")
print(f"   로컬 편향 {KC.loc[K_SUB,'편향']:+.4f}  (v4 +0.024 정상 / v3 +0.051 깨짐)")
print(f"   리더보드 기대 : v6가 0.6416이었으므로 대략 0.643~0.645")
print("\n※ 기존 v1~v7 파일은 건드리지 않았다.")

■ 챔피언(v6 구성) 기준 확대 배율


,fold1,미래3개,B평균,편향,편향_최대그룹,σ,band6,FICR,편향 안전
k,,,,,,,,,
1.00,0.6180,0.6460,0.6377,0.0120,0.0235,0.1668,0.3296,0.4036,OK
1.02,0.6184,0.6476,0.6388,0.0160,0.0273,0.1682,0.3348,0.4073,OK
1.04,0.6203,0.6487,0.6403,0.0199,0.0312,0.1697,0.3402,0.4111,OK
1.05,0.6198,0.6484,0.6400,0.0219,0.0332,0.1705,0.3414,0.4111,OK
1.06,0.6193,0.6472,0.6389,0.0239,0.0351,0.1714,0.3419,0.4098,OK
1.08,0.6173,0.6453,0.6370,0.0279,0.0390,0.1731,0.3407,0.4078,OK
1.10,0.6152,0.6416,0.6336,0.0318,0.0429,0.1751,0.3353,0.4033,⚠️ 위험
1.12,0.6099,0.6371,0.6287,0.0357,0.0467,0.1771,0.3300,0.3961,⚠️ 위험
1.15,0.6023,0.6283,0.6206,0.0414,0.0524,0.1803,0.3188,0.3829,⚠️ 위험



편향 안전한 k : [1.0, 1.02, 1.04, 1.05, 1.06, 1.08]
fold1이 고른 k = 1.04
   미래3개 0.6487 (k=1.00 0.6460, +0.0027)   편향 +0.0199
(ctx.test 가 없어 parquet 에서 시각만 읽었다)

■ v6 복원 검증 : 기존 제출 파일과 최대 차이 0.000000 kWh  → 동일 OK

■ 확대 중심값(학습 라벨 평균): 1=6,723, 2=7,183, 3=5,666

■ v6 → v8 예측값이 어떻게 바뀌었나


,v6 평균,v8 평균,변화%,v6 표준편차,v8 표준편차,0으로 잘린 행,용량에 걸린 행
1,8605.02,8680.31,0.87,5910.56,6146.98,0,0
2,9153.82,9232.64,0.86,6384.04,6639.40,0,0
3,8310.84,8416.64,1.27,5057.05,5259.33,0,0



저장 완료: D:\공모전\wind_forecast_new\submissions\20260804_v8_widen104.csv
   구성: v6(LightGBM 0.5 + 산식MLP 0.5) 에 확대 k=1.04 적용
   로컬 미래3개 0.6487 (v6 0.6460)
   로컬 편향 +0.0199  (v4 +0.024 정상 / v3 +0.051 깨짐)
   리더보드 기대 : v6가 0.6416이었으므로 대략 0.643~0.645

※ 기존 v1~v7 파일은 건드리지 않았다.


---

# 15층 — 로컬을 거스른다: 반대 방향으로 간다

## v8 결과 (2026-08-04 제출)

| | 1-NMAE | FICR | Total |
|---|---|---|---|
| v6 | 0.8687 | 0.4145 | **0.6416** |
| **v8** (확대 k=1.04) | 0.86576 | 0.41379 | **0.6398** |
| 변화 | **-0.0029** | -0.0007 | **-0.0018** |

확대는 예측 평균을 **+0.9~1.3%** 올렸다(표준편차 +4%).
**NMAE만 크게 잃고 FICR은 그대로**였다. 노린 것은 안 오고 대가만 치렀다.

## 여기서 확정되는 것 두 가지

### ① 리더보드 숫자 자체가 방향을 알려준다

NMAE는 **조건부 중앙값**에서 최소가 된다.
**위로 밀었더니 NMAE가 나빠졌다** = 우리는 **이미 중앙값보다 위에 있다** = **과대예측 중**이다.
**⇒ 아래로 내리면 NMAE가 좋아져야 한다.**

### ② "위로 미는 조작은 2025년에 손해" — 세 번째 독립 확인

| 시도 | 로컬 | 리더보드 |
|---|---|---|
| v3 (τ=0.60, 편향 +0.051) | **+0.0110** | **-0.0061** |
| exp019 (이전 프로젝트, 아핀 상향) | 개선 | **-0.0087** |
| **v8 (확대 k=1.04)** | **+0.002** | **-0.0018** |

**세 번 다 로컬은 올랐고 리더보드는 떨어졌다.**
**로컬 검증은 '위치' 축에서 방향이 반대로 나온다.**

물리적 이유도 있다. 2025년은 예보 풍속이 3년 중 가장 센 해다
(LDAPS 10m 4.84 → 5.11 m/s, GFS850 8.04 → 8.75 m/s).
정격 근처 시간이 많고, 실제 발전량은 설비용량에서 막히는데 우리 모델은 계속 위로 새고 있다.
그리고 FICR은 발전량으로 가중하므로 **바로 그 고출력 시간대가 점수를 지배한다.**

### ③ 그런데 아래쪽은 한 번도 안 해봤다

τ를 0.50 → 0.60 → 0.65 → 0.70으로 **올리기만** 했다. **0.50 아래는 시험한 적이 없다.**
HANDOFF의 "위치 최적화는 끝났다"는 사실 **"위쪽이 막혔다"** 였을 뿐이다.
확대도 마찬가지로 k>1만 제출했다.

## 이 셀 — k < 1 (압축)

```
새 예측 = 학습라벨평균 + k × (예측 − 학습라벨평균),   k = 0.96
```

k<1이면 **높은 예측은 내려가고 낮은 예측은 올라간다.** 채점 대상은 대부분 높은 쪽이므로
사실상 **고출력 시간대를 끌어내리는** 조작이다.

**대칭 실험이라는 점이 중요하다.** 방금 +4%를 해서 -0.0018을 얻었다.
-4%를 해서 +0.0018 근처가 나오면 **"위치 축은 반대로 가야 한다"가 확정**되고,
그때는 더 내려서(k=0.92 등) 계속 캐면 된다.

## ⚠️ 이건 로컬 검증을 정면으로 거스른다

로컬(2023~24)에서 k=0.95는 **-0.010**으로 크게 나빴다.
그런데 **로컬이 이 축에서 세 번 연속 틀렸으므로 이번엔 리더보드를 믿는다.**
근거가 로컬 1개 vs 리더보드 3개다.

기록해둘 것: **이 실험이 실패하면 "로컬 반전" 가설도 함께 폐기**하고,
위치 축은 v6(k=1.00)로 고정한 뒤 다른 축으로 간다.

**비용**: 학습 0회. 1분.

In [32]:
K_DOWN = 0.96          # v8의 +4%와 대칭이 되도록 -4%

parts = np.load(pl.REPO_ROOT / "models" / "v5_parts.npz")
SAMPLE = pl.REPO_ROOT / "data" / "sample_submission.csv"
sub.SUBMISSIONS_DIR = pl.REPO_ROOT / "submissions"
test_dtm = pd.read_parquet(pl.PROCESSED_DIR / "test_features_v1.parquet",
                           columns=["kst_dtm"])["kst_dtm"].reset_index(drop=True)
MU = {g: pd.read_parquet(pl.PROCESSED_DIR / "train_features_v1.parquet",
                         columns=[g])[g].mean() for g in TARGET_COLS}

v6 = {g: 0.5 * parts[f"lgb_{g}"] + 0.5 * parts[f"mlp_{g}"] for g in TARGET_COLS}
v9 = {g: np.clip(MU[g] + K_DOWN * (v6[g] - MU[g]), 0, CAPACITY_KWH[g]) for g in TARGET_COLS}

chg = pd.DataFrame({
    "v6 평균": [v6[g].mean() for g in TARGET_COLS],
    "v9 평균": [v9[g].mean() for g in TARGET_COLS],
    "평균 변화%": [(v9[g].mean() / v6[g].mean() - 1) * 100 for g in TARGET_COLS],
    "v6 표준편차": [v6[g].std() for g in TARGET_COLS],
    "v9 표준편차": [v9[g].std() for g in TARGET_COLS],
}, index=[g.split("_")[-1] for g in TARGET_COLS])
print(f"■ v6 → v9  (압축 k={K_DOWN})   ※ v8은 k=1.04로 평균 +0.9~1.3%였고 -0.0018이었다")
display(chg.round(2))

df9 = pd.DataFrame({"forecast_kst_dtm": test_dtm.dt.strftime("%Y-%m-%d %H:%M:%S").to_numpy(), **v9})
s9 = sub.build_submission(df9, sample_path=SAMPLE)
sub.validate_submission(s9, sample_path=SAMPLE)
path = sub.save_submission(s9, f"20260804_v9_shrink{int(round(K_DOWN*100))}.csv", sample_path=SAMPLE)
print(f"\n저장 완료: {path}")

# 참고용 — 로컬은 뭐라고 하는가 (거스르고 있다는 걸 명시적으로 남긴다)
if "MIX_지금" in set(bench.df["variant"]):
    name = f"KC_{int(round(K_DOWN*100))}"
    for fold in ctx.fold_info:
        for g in TARGET_COLS:
            mu = ctx.train.loc[ctx.fold_info[fold]["train_mask"], g].mean()
            PRED[(fold, name, g)] = (mu + K_DOWN * (PRED[(fold, "MIX_지금", g)] - mu)
                                     ).clip(lower=0, upper=CAPACITY_KWH[g])
    bench.add_from_cache(ctx, PRED, name)
    d = full_diag(name); b = full_diag("MIX_지금")
    print(f"\n■ 로컬은 뭐라고 하나 (참고용 — 이번엔 거스른다)")
    print(f"   미래3개 : {b['미래3개']:.4f} (k=1.00) → {d['미래3개']:.4f} (k={K_DOWN})  "
          f"{d['미래3개']-b['미래3개']:+.4f}")
    print(f"   편향    : {b['편향']:+.4f} → {d['편향']:+.4f}")
    print(f"   FICR    : {b['FICR']:.4f} → {d['FICR']:.4f}")
    print("   → 로컬은 반대라고 한다. 리더보드 3회(v3·exp019·v8)가 로컬을 이긴다고 보고 진행한다.")

print(f"""
■ 제출 후 판정 (미리 등록)
   0.6416 초과            → 위치 축은 '아래'가 맞다. k=0.92 로 더 내려 v10을 만든다
   0.6398 ~ 0.6416        → 위치 축은 이미 최적. v6로 고정하고 다른 축으로 간다
   0.6398 미만            → '로컬 반전' 가설 폐기. 위치는 v6 고정, 로컬을 다시 믿는다""")

■ v6 → v9  (압축 k=0.96)   ※ v8은 k=1.04로 평균 +0.9~1.3%였고 -0.0018이었다


,v6 평균,v9 평균,평균 변화%,v6 표준편차,v9 표준편차
1,8605.02,8529.74,-0.87,5910.56,5674.14
2,9153.82,9074.99,-0.86,6384.04,6128.68
3,8310.84,8205.05,-1.27,5057.05,4854.77



저장 완료: D:\공모전\wind_forecast_new\submissions\20260804_v9_shrink96.csv

■ 로컬은 뭐라고 하나 (참고용 — 이번엔 거스른다)
   미래3개 : 0.6460 (k=1.00) → 0.6387 (k=0.96)  -0.0073
   편향    : +0.0120 → +0.0041
   FICR    : 0.4036 → 0.3876
   → 로컬은 반대라고 한다. 리더보드 3회(v3·exp019·v8)가 로컬을 이긴다고 보고 진행한다.

■ 제출 후 판정 (미리 등록)
   0.6416 초과            → 위치 축은 '아래'가 맞다. k=0.92 로 더 내려 v10을 만든다
   0.6398 ~ 0.6416        → 위치 축은 이미 최적. v6로 고정하고 다른 축으로 간다
   0.6398 미만            → '로컬 반전' 가설 폐기. 위치는 v6 고정, 로컬을 다시 믿는다


---

# 16층 — 위치는 고정하고 '모양'만 비교한다

## v9 결과 (2026-08-04 제출) — 위치 축이 완전히 닫혔다

| | 1-NMAE | FICR | Total |
|---|---|---|---|
| **v9** (k=0.96, 내림) | **0.87072** ↑ | **0.40326** ↓↓ | 0.63699 |
| **v6** (k=1.00) | 0.8687 | **0.4145** | **0.6416** |
| **v8** (k=1.04, 올림) | 0.86576 ↓ | 0.41379 | 0.63978 |

**NMAE는 예측대로 움직였다** — 내리니 +0.0020 좋아졌다. 우리가 중앙값보다 위에 있었다는 게 맞았다.
**그런데 FICR이 -0.0112로 무너졌다.** 올릴 때(-0.0007)의 16배다.

세 점으로 2차 곡선을 맞추면 총점 꼭짓점은 **k ≈ 1.009**, 거기서 얻을 것은 **+0.0002**뿐이다.
**제출 한 번 쓸 값어치가 없다.**

> **위치·스케일 축은 끝났다. v6가 이미 꼭짓점이다.**
> 이번엔 위·아래를 다 봤으므로 반쪽 결론이 아니다.

## '로컬 반전' 가설은 폐기 — 대신 더 정확한 규칙을 얻었다

로컬은 **내리는 방향은 맞혔고**(-0.0094 예측, 실제 -0.0046) **올리는 방향만 틀렸다**(+0.0024 예측, 실제 -0.0018).
반전이 아니라 **로컬이 '위로 미는 조작'만 과대평가**한다.

> **새 규칙: 평균 예측을 올리는 후보는 로컬 점수를 깎아서 읽는다.**
> **평균을 안 움직이는 후보는 그대로 읽는다.**

## 이 규칙이 다음 카드에 바로 걸린다

3등분 블렌드(LGB+직접모델+MLP)는 로컬에서 챔피언보다 **+0.0003**뿐인데
편향은 **+0.0223 vs +0.0120**으로 더 높다. **위로 미는 성분이 섞여 있어
그 +0.0003조차 과대평가**일 가능성이 크다.

## 그래서 이 셀은 — 위치를 v6에 맞춰놓고 모양만 본다

각 후보의 예측을 **채점 대상 행의 평균이 챔피언과 같아지도록 평행이동**한 뒤 비교한다.
그러면 남는 차이는 **오직 예측의 모양**(어느 시간대를 높게/낮게 보느냐)뿐이다.

이게 왜 옳은 비교인가:
- 위치 축은 이미 최적임이 **리더보드로** 확인됐다 → 위치가 다른 후보는 그 차이만큼 손해를 본다
- 위치를 맞춰야 **다양성·모양의 순수 효과**가 보인다
- 그리고 이렇게 고른 후보는 **평균을 안 움직이므로 로컬 점수를 그대로 믿을 수 있다**

비교 대상:
- 챔피언 (LGB 0.5 + MLP 0.5) = v6
- 3등분 평균 (LGB + 직접 + MLP)
- 중앙값 3종
- LGB + 직접모델 (MLP 없이)
- LGB 단독 / 직접모델 단독

**비용**: 학습 0회. 1~2분.

In [33]:
CHAMP = "MIX_지금"

# 후보들 (전부 이미 캐시에 있다)
for fold in ctx.fold_info:
    for g in TARGET_COLS:
        PRED[(fold, "LGB_DIR", g)] = 0.5 * (PRED[(fold, "L0_lgb", g)]
                                            + PRED[(fold, "L10_direct", g)])
bench.add_from_cache(ctx, PRED, "LGB_DIR")

SHAPES = {"챔피언(v6)": CHAMP, "3등분 평균": "MIX_똑같이", "중앙값 3종": "MED3",
          "LGB+직접(반반)": "LGB_DIR", "LGB 단독": "L0_lgb", "직접모델 단독": "L10_direct"}


def scored_mean(fold, name, g):
    """채점 대상 행에서의 평균 예측 (위치를 맞출 기준)."""
    info = ctx.fold_info[fold]
    a = info["actual_df"][g].to_numpy(dtype=float)
    ok = np.isfinite(a) & (a >= CAPACITY_KWH[g] * 0.10)
    return PRED[(fold, name, g)].to_numpy()[ok].mean()


# ── 위치를 챔피언에 맞춘 판본을 만든다 ─────────────────────────────────────
for label, name in SHAPES.items():
    tag = "RC_" + name
    for fold in ctx.fold_info:
        for g in TARGET_COLS:
            shift = scored_mean(fold, CHAMP, g) - scored_mean(fold, name, g)
            PRED[(fold, tag, g)] = (PRED[(fold, name, g)] + shift
                                    ).clip(lower=0, upper=CAPACITY_KWH[g])
    bench.add_from_cache(ctx, PRED, tag)

raw = pd.DataFrame({lab: full_diag(n) for lab, n in SHAPES.items()}).T
rc = pd.DataFrame({lab: full_diag("RC_" + n) for lab, n in SHAPES.items()}).T

print("■ 위치를 맞추기 **전** (지금까지 봐온 숫자)")
display(raw[["미래3개", "B평균", "편향", "σ", "band6", "FICR"]].round(4))
print("\n■ 위치를 챔피언에 맞춘 **후** — 남은 차이는 '모양'뿐이다")
display(rc[["미래3개", "B평균", "편향", "σ", "band6", "FICR"]].round(4))

base = rc.loc["챔피언(v6)", "미래3개"]
print("\n■ 챔피언 대비 (위치 맞춘 뒤) — 이 숫자는 평균을 안 움직이므로 그대로 믿어도 된다")
for lab in SHAPES:
    d = rc.loc[lab, "미래3개"] - base
    d_raw = raw.loc[lab, "미래3개"] - raw.loc["챔피언(v6)", "미래3개"]
    tag = "★ 채택 가능" if d >= ms.MIN_EFFECT else ""
    print(f"   {lab:16s} 위치맞춤 {d:+.4f}   (맞추기 전 {d_raw:+.4f})   {tag}")

print(f"""
■ 읽는 법
   '맞추기 전'이 좋았는데 '맞춘 뒤' 사라지면 → 그 이득은 **위치를 위로 민 것**에서 왔다.
   v8/v9로 확인했듯 그건 리더보드에서 손해다. 채택하면 안 된다.
   '맞춘 뒤'에도 남으면 → **진짜 모양의 이득**이다. 제출 값어치가 있다.""")

■ 위치를 맞추기 **전** (지금까지 봐온 숫자)


,미래3개,B평균,편향,σ,band6,FICR
챔피언(v6),0.6460,0.6377,0.0120,0.1668,0.3296,0.4036
3등분 평균,0.6463,0.6372,0.0223,0.1640,0.3279,0.4040
중앙값 3종,0.6459,0.6371,0.0190,0.1665,0.3307,0.4049
LGB+직접(반반),0.6427,0.6344,0.0335,0.1669,0.3287,0.4031
LGB 단독,0.6436,0.6356,0.0243,0.1717,0.3339,0.4053
직접모델 단독,0.6307,0.6225,0.0428,0.1719,0.3151,0.3861



■ 위치를 챔피언에 맞춘 **후** — 남은 차이는 '모양'뿐이다


,미래3개,B평균,편향,σ,band6,FICR
챔피언(v6),0.6460,0.6377,0.0120,0.1668,0.3296,0.4036
3등분 평균,0.6433,0.6322,0.0120,0.1640,0.3170,0.3948
중앙값 3종,0.6437,0.6333,0.0120,0.1665,0.3257,0.3980
LGB+직접(반반),0.6355,0.6244,0.0121,0.1668,0.3084,0.3835
LGB 단독,0.6401,0.6313,0.0120,0.1717,0.3223,0.3967
직접모델 단독,0.6241,0.6115,0.0123,0.1716,0.2954,0.3642



■ 챔피언 대비 (위치 맞춘 뒤) — 이 숫자는 평균을 안 움직이므로 그대로 믿어도 된다
   챔피언(v6)          위치맞춤 +0.0000   (맞추기 전 +0.0000)   
   3등분 평균           위치맞춤 -0.0027   (맞추기 전 +0.0003)   
   중앙값 3종           위치맞춤 -0.0022   (맞추기 전 -0.0001)   
   LGB+직접(반반)       위치맞춤 -0.0105   (맞추기 전 -0.0033)   
   LGB 단독           위치맞춤 -0.0058   (맞추기 전 -0.0024)   
   직접모델 단독          위치맞춤 -0.0218   (맞추기 전 -0.0152)   

■ 읽는 법
   '맞추기 전'이 좋았는데 '맞춘 뒤' 사라지면 → 그 이득은 **위치를 위로 민 것**에서 왔다.
   v8/v9로 확인했듯 그건 리더보드에서 손해다. 채택하면 안 된다.
   '맞춘 뒤'에도 남으면 → **진짜 모양의 이득**이다. 제출 값어치가 있다.
